# CS781 Capstone - Phase 1, Submission 02
## Encoder-decoder legal language model experiment

This notebook is a separate Submission 02 workspace. It does not read from or overwrite
Submission 01. It provides the supporting infrastructure for official-data discovery,
schema validation, bounded preprocessing, tokenizer reuse, GPU checks, resumable training,
checkpoint verification, inference-only restoration, resumable prediction generation,
Codabench packaging, and recovery archives.

**Binding academic-integrity boundary:** the capstone permits AI help with ancillary code and
architecture understanding, but prohibits AI from writing the whole assessed architecture,
tuning hyperparameters, or writing the scientific analysis/conclusions. Therefore the
encoder-decoder implementation, its numerical architecture configuration, the training
hyperparameters, hypothesis, ablation, and conclusions remain clearly marked student-owned
inputs. The notebook will refuse full training until those inputs are supplied and attested.

The intended substantive difference from Submission 01 is architectural: a bidirectional
encoder conditions a causal decoder through encoder-decoder cross-attention. Changing only
width, depth, heads, learning rate, decoding, or duration is not enough.


## Safe execution map

- `prepare`: inspect files, schemas, infrastructure, and saved artifacts. This is the default.
- `fresh_train`: prepare the bounded corpus cache and start only when both explicit training
  switches are enabled. It refuses an existing compatible latest checkpoint.
- `resume_train`: reconstruct everything and require a compatible checkpoint; it never falls
  back silently to step zero.
- `inference_only`: restore the saved tokenizer, student model source, configuration, and
  selected checkpoint; load only the official test prompts. The training corpus and local
  evaluator are not required.

`Run All` is safe by default: expensive preprocessing, tokenizer training, GPU smoke testing,
full training, inference, packaging, and recovery export are all opt-in.


## 1. Environment setup and reproducibility

In [ ]:
from __future__ import annotations

import ast
import csv
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import platform
import sys
import tempfile
import time
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, Sequence

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, Sampler

try:
    from tqdm.auto import tqdm
except ImportError as exc:
    raise RuntimeError("tqdm is required; install it in the Kaggle environment") from exc

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def atomic_write_bytes(path: Path, payload: bytes) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())
        temporary = Path(handle.name)
    os.replace(temporary, path)

def atomic_write_json(path: Path, value: Any) -> None:
    atomic_write_bytes(Path(path), (json.dumps(value, indent=2, ensure_ascii=False) + "\n").encode("utf-8"))

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

print({"python": sys.version.split()[0], "torch": torch.__version__, "utc": utc_now()})


In [ ]:
def _discover_attached_tokenizer() -> str | None:
    """Find one frozen Submission 01 tokenizer without assuming a Kaggle dataset slug."""
    search_root = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
    candidates = []
    for tokenizer_json in search_root.rglob("tokenizer.json"):
        directory = tokenizer_json.parent
        if (directory / "tokenizer_config.json").is_file() and (directory / "tokenizer_metadata.json").is_file():
            candidates.append(directory.resolve())
    preferred = [
        item for item in candidates
        if "submission01" in str(item).lower().replace("_", "").replace("-", "")
    ]
    choices = preferred or candidates
    return str(sorted(set(choices))[0]) if len(set(choices)) == 1 else None

def _discover_student_model_source() -> str | None:
    """Find the attached standalone model source locally or under /kaggle/input."""
    local = Path.cwd() / "encoder_decoder_lm.py"
    if local.is_file() and not Path("/kaggle/input").exists():
        return str(local.resolve())
    search_root = Path("/kaggle/input")
    if not search_root.exists():
        return None
    candidates = sorted(path.resolve() for path in search_root.rglob("encoder_decoder_lm.py") if path.is_file())
    return str(candidates[0]) if len(candidates) == 1 else None

@dataclass
class RunConfig:
    experiment_id: str = "phase1_submission02_encoder_decoder"
    seed: int = 781
    execution_mode: str = "prepare"  # prepare | fresh_train | resume_train | inference_only
    workspace: str = field(default_factory=lambda: str(
        Path("/kaggle/working/CS781_Capstone/phase1/submission_02")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "CS781_Capstone/phase1/submission_02"
    ))
    input_root: str = field(default_factory=lambda: str(
        Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
    ))
    train_files: tuple[str, ...] = (
        "train_acts.json",
        "train_high_courts.json",
        "train_supreme_court.json",
        "train_tribunals.json",
    )
    test_file: str = "Phase1- Test.json"
    test_filename_aliases: tuple[str, ...] = ("Phase1- Test.json", "test.json")
    hf_repo_id: str = "Exploration-Lab/CS-781-Capstone"
    hf_allow_patterns: tuple[str, ...] = ("*Phase_1_Pretraining/*", "*Phase-1_Pretraining/*")
    official_file_overrides: dict[str, str] = field(default_factory=dict)
    training_schema_overrides: dict[str, dict[str, Any]] = field(default_factory=dict)
    test_schema_override: dict[str, Any] | None = None

    # Reused from the Submission 01 control for a fair architecture comparison.
    validation_fraction: float = 0.05
    source_mixing_ratios: dict[str, float] = field(default_factory=lambda: {
        "acts": 0.25,
        "high_courts": 0.25,
        "supreme_court": 0.25,
        "tribunals": 0.25,
    })
    # Fixed before preprocessing: at most 10,000 deterministic documents from each
    # of Acts, High Courts, Supreme Court, and Tribunals (40,000 total before deduplication).
    corpus_document_cap_per_source: int | None = 10_000
    corpus_scope_statement: str = (
        "Training uses a deterministic bounded subset of at most 10,000 documents per "
        "source (40,000 before deduplication), not the entire 4.5-billion-token corpus."
    )

    # These discover exactly one attached artifact and deliberately return None on ambiguity.
    # If Kaggle contains multiple matches, replace the value with the explicit dataset path.
    tokenizer_import_path: str | None = field(default_factory=_discover_attached_tokenizer)
    student_model_source_path: str | None = field(default_factory=_discover_student_model_source)
    # Optional path to a downloaded completed notebook for the final recovery archive.
    completed_notebook_path: str | None = None

    architecture_record: dict[str, Any] = field(default_factory=lambda: {
        "family": "encoder-decoder Transformer",
        "substantive_difference_from_submission01": (
            "six-layer bidirectional encoder plus six-layer causal decoder with "
            "encoder-decoder cross-attention"
        ),
        "parameter_range": [50_000_000, 300_000_000],
        "encoder_layers": 6,
        "decoder_layers": 6,
        "hidden_size": 512,
        "attention_heads": 8,
        "ffn_size": 2048,
        "position_representation": "learned_absolute",
        "normalization": "pre_layer_norm",
        "activation": "gelu",
        "embedding_tying": "shared_encoder_decoder_and_output",
        "max_encoder_length": 1024,
        "max_decoder_length": 512,
        "analytical_parameter_count_at_vocab_32000": 61_342_976,
        "student_design_rationale": (
            "A 512-wide 6+6 pre-norm Transformer stays above the 50M parameter floor while "
            "remaining practical on a 16 GB Kaggle GPU. Training uses 512/256-token pairs, "
            "while 1024/512 positional limits provide safer inference headroom. The "
            "bidirectional encoder can form "
            "a full-prompt legal representation, and every decoder block can retrieve it "
            "through cross-attention while preserving causal target prediction."
        ),
    })
    innovation_hypothesis: str = (
        "Compared with the Submission 01 decoder-only baseline, a bidirectional encoder "
        "feeding a causal decoder through cross-attention is expected to improve use of "
        "long-range facts, parties, provisions, and procedural context in legal continuation. "
        "The comparison will use held-out token cross-entropy/perplexity, BERTScore F1, "
        "ROUGE-L F1, BLEU-4, and declared repetition diagnostics. The frozen Submission 01 "
        "BPE mapping, bounded official corpus, deterministic split, source mixture, scratch "
        "training requirement, and evaluation protocol remain controlled; this is a "
        "pre-training hypothesis, not a conclusion."
    )
    ablation_control: str = "Submission 01 decoder-only baseline under a matched data/tokenizer/evaluation protocol"
    student_owned_architecture_attestation: bool = False
    student_configuration_confirmed: bool = False
    full_training_authorized: bool = False
    trained_from_scratch_attestation: bool = False
    decoding: dict[str, Any] = field(default_factory=lambda: {
        "target_words": 128,
        "max_new_tokens": 448,
        "decode_check_interval": 4,
        "policies": [
            {
                "name": "primary_top_k_nucleus",
                "strategy": "top_k_top_p",
                "seed": 781,
                "temperature": 0.85,
                "top_k": 100,
                "top_p": 0.92,
                "repetition_penalty": 1.08,
                "no_repeat_ngram_size": 4,
                "min_unique_word_ratio": 0.55,
                "max_word_fraction": 0.08,
            },
            {
                "name": "retry_top_k_nucleus",
                "strategy": "top_k_top_p",
                "seed": 1781,
                "temperature": 0.95,
                "top_k": 80,
                "top_p": 0.96,
                "repetition_penalty": 1.12,
                "no_repeat_ngram_size": 3,
                "min_unique_word_ratio": 0.50,
                "max_word_fraction": 0.10,
            },
        ],
    })

CFG = RunConfig()
if CFG.execution_mode not in {"prepare", "fresh_train", "resume_train", "inference_only"}:
    raise ValueError(f"Unsupported execution_mode: {CFG.execution_mode}")

ROOT = Path(CFG.workspace)
DIRS = {name: ROOT / name for name in (
    "model", "tokenizer", "data", "checkpoints", "histories", "evaluations",
    "predictions", "plots", "recovery", "manifests", "submission"
)}
for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

RUN_CONFIG_PATH = DIRS["manifests"] / "run_config.json"
def register_run_config(cfg: RunConfig) -> dict[str, Any]:
    current = json.loads(json.dumps(asdict(cfg), ensure_ascii=False))
    mutable_runtime = {
        "execution_mode", "full_training_authorized", "trained_from_scratch_attestation",
        "completed_notebook_path", "tokenizer_import_path", "student_model_source_path",
    }
    mutable_before_checkpoint = {
        "corpus_document_cap_per_source", "corpus_scope_statement", "architecture_record", "innovation_hypothesis",
        "ablation_control", "student_owned_architecture_attestation",
        "student_configuration_confirmed", "decoding",
    }
    if RUN_CONFIG_PATH.exists():
        existing = json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8"))
        checkpoints_exist = any(DIRS["checkpoints"].glob("*.pt"))
        conflicts = {}
        merged = dict(existing)
        for key, new_value in current.items():
            old_value = existing.get(key)
            if old_value == new_value:
                continue
            if key in mutable_runtime or (key in mutable_before_checkpoint and not checkpoints_exist):
                merged[key] = new_value
            else:
                conflicts[key] = (old_value, new_value)
        if conflicts:
            raise RuntimeError(
                f"Existing Submission 02 configuration conflict: {conflicts}. "
                "Use a new experiment workspace instead of mutating a trained run."
            )
        atomic_write_json(RUN_CONFIG_PATH, merged)
        return merged
    atomic_write_json(RUN_CONFIG_PATH, current)
    return current

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)
register_run_config(CFG)
print({
    "workspace": str(ROOT),
    "mode": CFG.execution_mode,
    "corpus_cap_per_source": CFG.corpus_document_cap_per_source,
    "tokenizer_import_path": CFG.tokenizer_import_path,
    "student_model_source_path": CFG.student_model_source_path,
    "corpus_scope": CFG.corpus_scope_statement,
})


In [ ]:
CFG.execution_mode = "fresh_train"
CFG.tokenizer_import_path = None
print(CFG.execution_mode)
print(CFG.student_model_source_path)

## 2. Kaggle GPU detection and optional authentication

In [ ]:
def gpu_inventory() -> list[dict[str, Any]]:
    rows = []
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        free_bytes, total_bytes = torch.cuda.mem_get_info(index)
        rows.append({
            "index": index,
            "device": f"cuda:{index}",
            "name": props.name,
            "total_memory_gib": round(total_bytes / 2**30, 2),
            "free_memory_gib": round(free_bytes / 2**30, 2),
            "capability": f"{props.major}.{props.minor}",
            "bf16_supported": bool(torch.cuda.is_bf16_supported()),
        })
    return rows

RUNTIME = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    "cuda_available": torch.cuda.is_available(),
}
GPUS = gpu_inventory()
print(json.dumps({"runtime": RUNTIME, "gpus": GPUS}, indent=2))
if not GPUS:
    print("WARNING: no CUDA GPU detected. Full training and the representative smoke test are blocked.")
elif len(GPUS) == 1:
    print("Submission 02 will use one explicitly selected GPU.")
else:
    print("Multiple GPUs detected. This notebook remains single-process unless the student explicitly implements and validates DDP.")


In [ ]:
# Optional only when an attached artifact actually requires Hugging Face authentication.
# Do not paste a token into the notebook. Use a Kaggle secret named HF_TOKEN.
def optional_hf_login(secret_name: str = "HF_TOKEN") -> bool:
    try:
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import login
        token = UserSecretsClient().get_secret(secret_name)
        if not token:
            return False
        login(token=token, add_to_git_credential=False)
        return True
    except Exception as exc:
        print(f"HF login skipped: {type(exc).__name__}: {exc}")
        return False

# optional_hf_login()  # Uncomment only if required.

## 3. Official file discovery and schema inspection

In [ ]:
import subprocess

HF_DATA_ROOT=Path('/kaggle/working/capstone_data') if Path('/kaggle/working').exists() else ROOT/'downloaded_official_data'
INSTALL_RUNTIME_DEPENDENCIES=True
DOWNLOAD_PHASE1_FROM_HF=False

def ensure_runtime_dependency(import_name:str,pip_spec:str)->None:
    if importlib.util.find_spec(import_name) is not None:
        return
    if not INSTALL_RUNTIME_DEPENDENCIES:
        raise RuntimeError(
            f'Missing {import_name!r}. Set INSTALL_RUNTIME_DEPENDENCIES=True and rerun this cell '
            f'to install {pip_spec!r} into the Kaggle session.'
        )
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pip_spec])

def kaggle_hf_token(secret_name:str='HF_TOKEN')->str|None:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(secret_name) or None
    except Exception:
        return None

def inspect_huggingface_phase1_files(repo_id:str)->list[str]:
    ensure_runtime_dependency('huggingface_hub','huggingface_hub>=0.25,<2')
    from huggingface_hub import HfApi
    api=HfApi(token=kaggle_hf_token())
    entries=list(api.list_repo_tree(repo_id=repo_id,repo_type='dataset',recursive=True))
    phase1=[];total_bytes=0
    for entry in entries:
        name=str(getattr(entry,'path',''))
        if 'phase_1' in name.lower() or 'phase-1' in name.lower():
            size=int(getattr(entry,'size',0) or 0);phase1.append(name);total_bytes+=size
            print(f'{name} ({size/2**30:.2f} GiB)' if size else name)
    if not phase1:
        raise RuntimeError(f'No Phase 1 files were visible in {repo_id}; check access/authentication')
    free=shutil.disk_usage(HF_DATA_ROOT.parent if HF_DATA_ROOT.parent.exists() else Path.cwd()).free
    print({'phase1_download_gib':round(total_bytes/2**30,2),'destination_free_gib':round(free/2**30,2)})
    if total_bytes and free<total_bytes+8*2**30:
        print('WARNING: direct download leaves less than 8 GiB for compressed preprocessing and checkpoints; prefer a private attached Kaggle dataset.')
    return phase1

def download_huggingface_phase1(cfg:RunConfig,destination:Path)->Path:
    ensure_runtime_dependency('huggingface_hub','huggingface_hub>=0.25,<2')
    ensure_runtime_dependency('ijson','ijson>=3.3,<4')
    from huggingface_hub import snapshot_download
    published=inspect_huggingface_phase1_files(cfg.hf_repo_id)
    required=set(cfg.train_files)
    available_names={Path(name).name for name in published}
    missing=sorted(required-available_names)
    if missing: raise RuntimeError(f'Official repository listing is missing required training files: {missing}')
    if not (set(cfg.test_filename_aliases)&available_names): raise RuntimeError(f'No official test filename matched {cfg.test_filename_aliases}')
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    resolved=snapshot_download(
        repo_id=cfg.hf_repo_id,repo_type='dataset',token=kaggle_hf_token(),
        allow_patterns=list(cfg.hf_allow_patterns),local_dir=str(destination),
    )
    print('Downloaded Phase 1 snapshot to',resolved)
    return Path(resolved)

print({'data_search_roots':[str(Path(CFG.input_root)),str(HF_DATA_ROOT)],'hf_destination_free_gib':round(shutil.disk_usage(HF_DATA_ROOT.parent if HF_DATA_ROOT.parent.exists() else Path.cwd()).free/2**30,2)})
if DOWNLOAD_PHASE1_FROM_HF:
    download_huggingface_phase1(CFG,HF_DATA_ROOT)

def discover_named_files(roots:Sequence[Path],cfg:RunConfig)->dict[str,list[Path]]:
    wanted=set(cfg.train_files)
    aliases=set(cfg.test_filename_aliases)
    found={name:[] for name in [*cfg.train_files,cfg.test_file]}
    for root in roots:
        root=Path(root)
        if not root.exists(): continue
        for path in root.rglob('*'):
            if not path.is_file(): continue
            if path.name in wanted:
                found[path.name].append(path.resolve())
            elif path.name in aliases:
                found[cfg.test_file].append(path.resolve())
    for name in found:
        found[name]=sorted(set(found[name]))
    return found

DATA_SEARCH_ROOTS=[]
for candidate in (Path(CFG.input_root),HF_DATA_ROOT):
    resolved=candidate.resolve() if candidate.exists() else candidate
    if resolved not in DATA_SEARCH_ROOTS:
        DATA_SEARCH_ROOTS.append(resolved)
DISCOVERED=discover_named_files(DATA_SEARCH_ROOTS,CFG)
for name,paths in sorted(DISCOVERED.items()):
    print(name,'->',[str(path) for path in paths])

def require_unique_file(discovered:Mapping[str,Sequence[Path]],name:str)->Path:
    override=CFG.official_file_overrides.get(name)
    if override:
        selected=Path(override).expanduser().resolve()
        if not selected.is_file():
            raise FileNotFoundError(f'Configured override for {name} does not exist: {selected}')
        return selected
    matches=list(discovered.get(name,[]))
    if len(matches)!=1:
        raise RuntimeError(
            f'Expected exactly one {name!r}; found {len(matches)}: {matches}. '
            'Set CFG.official_file_overrides[name] to an inspected exact path; duplicates are never selected silently.'
        )
    return Path(matches[0])


In [ ]:
MAX_IN_MEMORY_JSON_BYTES = 256 * 2**20

def _ijson():
    ensure_runtime_dependency("ijson", "ijson>=3.3,<4")
    import ijson
    return ijson

def detect_json_layout(path: Path) -> tuple[str, str | None]:
    ijson = _ijson()
    with Path(path).open("rb") as handle:
        parser = ijson.parse(handle)
        try:
            _, event, _ = next(parser)
        except StopIteration:
            raise ValueError(f"Empty JSON file: {path}")
        if event == "start_array":
            return "array", None
        if event != "start_map":
            raise TypeError(f"Unsupported JSON root event {event}: {path}")
        pending_key = None
        for prefix, event, value in parser:
            if prefix == "" and event == "map_key":
                pending_key = str(value)
                continue
            if pending_key is not None:
                if event == "start_array":
                    return "wrapper_array", pending_key
                if event in ("start_map", "string", "number", "boolean", "null"):
                    return "id_keyed_object", None
        return "empty_object", None

def iter_json_records(path: Path, *, id_field: str | None = None, text_field: str | None = None):
    ijson = _ijson()
    layout, wrapper = detect_json_layout(path)
    with Path(path).open("rb") as handle:
        if layout == "array":
            for index, row in enumerate(ijson.items(handle, "item")):
                if not isinstance(row, dict):
                    raise TypeError("Array records must be JSON objects")
                result = dict(row)
                result.setdefault("_record_index", index)
                yield result
        elif layout == "wrapper_array":
            for index, row in enumerate(ijson.items(handle, f"{wrapper}.item")):
                if not isinstance(row, dict):
                    raise TypeError("Wrapped records must be JSON objects")
                result = dict(row)
                result.setdefault("_record_index", index)
                yield result
        elif layout == "id_keyed_object":
            for index, (key, value) in enumerate(ijson.kvitems(handle, "")):
                if isinstance(value, dict):
                    row = dict(value)
                    if id_field and id_field not in row:
                        row[id_field] = str(key)
                    row.setdefault("_container_key", str(key))
                    row.setdefault("_record_index", index)
                    yield row
                elif isinstance(value, str):
                    yield {
                        id_field or "_container_key": str(key),
                        text_field or "_container_value": value,
                        "_container_key": str(key),
                        "_record_index": index,
                    }
                else:
                    raise TypeError(f"Unsupported top-level value under {key!r}: {type(value).__name__}")
        elif layout != "empty_object":
            raise TypeError(f"Unsupported JSON layout: {layout}")

def sample_json_records(path: Path, limit: int = 256) -> list[dict[str, Any]]:
    import itertools
    return list(itertools.islice(iter_json_records(path), int(limit)))

def inspect_schema(path: Path, sample_size: int = 3) -> dict[str, Any]:
    rows = sample_json_records(path, max(256, sample_size))
    keys = Counter(key for row in rows for key in row)
    layout, wrapper = detect_json_layout(path)
    return {
        "path": str(path),
        "bytes": Path(path).stat().st_size,
        "layout": layout,
        "wrapper_key": wrapper,
        "sampled_records": len(rows),
        "sample_field_presence": dict(keys),
        "samples": [
            {key: {"type": type(value).__name__, "preview": str(value)[:140]} for key, value in row.items()}
            for row in rows[:sample_size]
        ],
    }

def infer_record_schema(path: Path, *, role: str) -> dict[str, Any]:
    rows = sample_json_records(path, 256)
    if not rows:
        raise ValueError(f"No records in {path}")
    text_candidates = (
        "_container_value", "text", "document", "content", "case_text",
        "judgment", "judgement", "prompt",
    )
    common_text = [
        name for name in text_candidates
        if all(isinstance(row.get(name), str) and row[name].strip() for row in rows)
    ]
    if len(common_text) != 1:
        raise RuntimeError(f"{role} text field is ambiguous in {path}: {common_text}")

    id_candidates = ("_container_key", "document_id", "doc_id", "case_id", "uid", "id")
    valid_ids = []
    for name in id_candidates:
        values = [row.get(name) for row in rows]
        if all(value is not None and str(value).strip() for value in values):
            if len({str(value) for value in values}) == len(values):
                valid_ids.append(name)
    if role == "test":
        if len(valid_ids) != 1:
            raise RuntimeError(f"Official test IDs are ambiguous in {path}: {valid_ids}")
        return {"id_field": valid_ids[0], "id_strategy": "field", "text_field": common_text[0]}
    if len(valid_ids) == 1:
        return {"id_field": valid_ids[0], "id_strategy": "field", "text_field": common_text[0]}
    if not valid_ids:
        return {"id_field": None, "id_strategy": "record_index", "text_field": common_text[0]}
    raise RuntimeError(f"Training IDs are ambiguous in {path}: {valid_ids}")

def resolve_official_schemas(cfg: RunConfig, discovered: Mapping[str, Sequence[Path]]) -> dict[str, Any]:
    training = {}
    for filename in cfg.train_files:
        inferred = infer_record_schema(require_unique_file(discovered, filename), role="training")
        override = cfg.training_schema_overrides.get(filename)
        if override is not None and dict(override) != inferred:
            raise RuntimeError(f"Schema override for {filename} conflicts with inspected data: {override} != {inferred}")
        training[filename] = inferred
    test = infer_record_schema(require_unique_file(discovered, cfg.test_file), role="test")
    if cfg.test_schema_override is not None and dict(cfg.test_schema_override) != test:
        raise RuntimeError(f"Test schema override conflicts with inspected data: {cfg.test_schema_override} != {test}")
    return {"training": training, "test": test}

_schema_names = [*CFG.train_files, CFG.test_file]
if all(len(DISCOVERED.get(name, [])) == 1 or name in CFG.official_file_overrides for name in _schema_names):
    SCHEMAS = {name: inspect_schema(require_unique_file(DISCOVERED, name)) for name in _schema_names}
    print(json.dumps(SCHEMAS, indent=2, ensure_ascii=False))
    RESOLVED_SCHEMA = resolve_official_schemas(CFG, DISCOVERED)
    atomic_write_json(DIRS["manifests"] / "resolved_json_schemas.json", RESOLVED_SCHEMA)
    print("Verified per-file schemas:", json.dumps(RESOLVED_SCHEMA, indent=2))
else:
    SCHEMAS = {}
    RESOLVED_SCHEMA = None
    unresolved = {
        name: len(DISCOVERED.get(name, [])) for name in _schema_names
        if len(DISCOVERED.get(name, [])) != 1 and name not in CFG.official_file_overrides
    }
    print("Schema inspection deferred until files resolve uniquely:", unresolved)


## 4. Bounded official-corpus cache, provenance, deduplication, and split

**Declared scope:** this experiment uses at most 10,000 deterministically selected documents from each of the four sources (at most 40,000 before deduplication). It is a bounded subset and must never be described as training on the entire 4.5-billion-token corpus. The cap follows the assignment's bounded-corpus requirement and the limited Kaggle compute budget.


In [ ]:
import heapq
import zlib

@dataclass(frozen=True)
class Document:
    uid: str
    text: str | None
    source: str
    raw_id: str
    content_hash: str
    word_count: int
    store_path: str | None = None
    store_offset: int | None = None
    store_length: int | None = None

def normalize_text(value: Any) -> str:
    if not isinstance(value, str):
        raise TypeError(f"Text must be a string, got {type(value).__name__}")
    result = value.replace("\r\n", "\n").strip()
    if not result:
        raise ValueError("Empty text")
    return result

def document_text(document: Document) -> str:
    if document.text is not None:
        return document.text
    if document.store_path is None or document.store_offset is None or document.store_length is None:
        raise RuntimeError(f"Document has no text storage: {document.uid}")
    with Path(document.store_path).open("rb") as handle:
        handle.seek(document.store_offset)
        payload = handle.read(document.store_length)
    return zlib.decompress(payload).decode("utf-8")

def fast_file_identity(path: Path, sample_bytes: int = 4 * 2**20) -> dict[str, Any]:
    path = Path(path)
    size = path.stat().st_size
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        digest.update(handle.read(sample_bytes))
        if size > sample_bytes:
            handle.seek(max(0, size - sample_bytes))
            digest.update(handle.read(sample_bytes))
    return {"name": path.name, "bytes": size, "head_tail_sha256": digest.hexdigest()}

def stable_raw_id(row: Mapping[str, Any], index: int, schema: Mapping[str, Any]) -> str:
    if schema["id_strategy"] == "record_index":
        return f"row-{index:012d}"
    value = str(row[schema["id_field"]]).strip()
    if not value:
        raise ValueError("Empty identifier")
    return value

def deterministic_selected_indices(
    path: Path, source: str, schema: Mapping[str, Any], cap: int, seed: int
) -> tuple[set[int], int]:
    if cap <= 0:
        raise ValueError("corpus_document_cap_per_source must be positive")
    heap: list[tuple[int, int]] = []
    count = 0
    for index, row in enumerate(iter_json_records(path, id_field=schema.get("id_field"), text_field=schema["text_field"])):
        raw_id = stable_raw_id(row, index, schema)
        score = int(hashlib.sha256(f"{seed}:{source}:{raw_id}".encode()).hexdigest(), 16)
        item = (-score, index)
        if len(heap) < cap:
            heapq.heappush(heap, item)
        elif item > heap[0]:
            heapq.heapreplace(heap, item)
        count += 1
    return {index for _, index in heap}, count

def _load_document_index(index_path: Path, store_path: Path) -> list[Document]:
    documents = []
    with Path(index_path).open("r", encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            documents.append(Document(
                uid=row["uid"], text=None, source=row["source"], raw_id=row["raw_id"],
                content_hash=row["content_hash"], word_count=int(row["word_count"]),
                store_path=str(store_path), store_offset=int(row["store_offset"]),
                store_length=int(row["store_length"]),
            ))
    return documents

def load_bounded_documents(
    path: Path,
    source: str,
    schema: Mapping[str, Any],
    store_dir: Path,
    cap: int,
    seed: int,
) -> tuple[list[Document], list[dict[str, Any]], dict[str, Any]]:
    path = Path(path)
    identity = fast_file_identity(path)
    selection_config = {
        "source_identity": identity,
        "schema": dict(schema),
        "cap": int(cap),
        "seed": int(seed),
        "selection": "smallest SHA-256(seed:source:stable_raw_id)",
        "format_version": 2,
    }
    key = hashlib.sha256(json.dumps(selection_config, sort_keys=True).encode()).hexdigest()[:20]
    store_dir = Path(store_dir)
    store_dir.mkdir(parents=True, exist_ok=True)
    store_path = store_dir / f"{source}.{key}.zbin"
    index_path = store_dir / f"{source}.{key}.index.jsonl"
    meta_path = store_dir / f"{source}.{key}.meta.json"
    if store_path.exists() and index_path.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("selection_config") != selection_config:
            raise RuntimeError(f"Existing cache metadata mismatch: {meta_path}")
        return _load_document_index(index_path, store_path), list(meta.get("rejections", [])), meta

    selected, total_records = deterministic_selected_indices(path, source, schema, cap, seed)
    with tempfile.NamedTemporaryFile(dir=store_dir, delete=False) as handle:
        temporary_store = Path(handle.name)
    with tempfile.NamedTemporaryFile(dir=store_dir, delete=False) as handle:
        temporary_index = Path(handle.name)
    rejections = []
    accepted = 0
    try:
        with temporary_store.open("wb") as store_handle, temporary_index.open("w", encoding="utf-8") as index_handle:
            for index, row in enumerate(iter_json_records(path, id_field=schema.get("id_field"), text_field=schema["text_field"])):
                if index not in selected:
                    continue
                try:
                    raw_id = stable_raw_id(row, index, schema)
                    text = normalize_text(row[schema["text_field"]])
                    encoded = text.encode("utf-8")
                    compressed = zlib.compress(encoded, level=3)
                    offset = store_handle.tell()
                    store_handle.write(compressed)
                    record = {
                        "uid": f"{source}:{raw_id}", "source": source, "raw_id": raw_id,
                        "content_hash": hashlib.sha256(encoded).hexdigest(),
                        "word_count": len(text.split()), "store_offset": offset,
                        "store_length": len(compressed),
                    }
                    index_handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                    accepted += 1
                except Exception as exc:
                    rejections.append({"index": index, "error": f"{type(exc).__name__}: {exc}"})
            store_handle.flush(); os.fsync(store_handle.fileno())
            index_handle.flush(); os.fsync(index_handle.fileno())
        os.replace(temporary_store, store_path)
        os.replace(temporary_index, index_path)
    finally:
        temporary_store.unlink(missing_ok=True)
        temporary_index.unlink(missing_ok=True)
    meta = {
        "selection_config": selection_config,
        "total_source_records": total_records,
        "accepted": accepted,
        "rejected": len(rejections),
        "rejections": rejections,
        "created_utc": utc_now(),
    }
    atomic_write_json(meta_path, meta)
    return _load_document_index(index_path, store_path), rejections, meta

def corpus_audit(docs: Sequence[Document]) -> dict[str, Any]:
    ids = Counter(document.uid for document in docs)
    hashes = Counter(document.content_hash for document in docs)
    return {
        "documents": len(docs),
        "sources": dict(Counter(document.source for document in docs)),
        "whitespace_word_count": sum(document.word_count for document in docs),
        "words_by_source": {
            source: sum(document.word_count for document in docs if document.source == source)
            for source in sorted({document.source for document in docs})
        },
        "duplicate_uids": sorted(key for key, value in ids.items() if value > 1),
        "duplicate_content_hashes": sorted(key for key, value in hashes.items() if value > 1),
    }

def deduplicate_exact_documents(docs: Sequence[Document]) -> tuple[list[Document], list[dict[str, str]]]:
    kept, dropped, seen_ids, seen_hashes = [], [], {}, {}
    for document in sorted(docs, key=lambda value: (value.uid, value.content_hash)):
        reason = "duplicate_uid" if document.uid in seen_ids else "duplicate_exact_text" if document.content_hash in seen_hashes else None
        if reason:
            retained = seen_ids.get(document.uid) if reason == "duplicate_uid" else seen_hashes[document.content_hash]
            dropped.append({"uid": document.uid, "kept_uid": retained, "reason": reason})
            continue
        kept.append(document)
        seen_ids[document.uid] = document.uid
        seen_hashes[document.content_hash] = document.uid
    return kept, dropped

def dataset_fingerprint(docs: Sequence[Document]) -> str:
    digest = hashlib.sha256()
    for document in sorted(docs, key=lambda value: value.uid):
        digest.update(f"{document.uid}\t{document.content_hash}\n".encode())
    return digest.hexdigest()


In [ ]:
def deterministic_document_split(documents:Sequence[Document],validation_fraction:float,seed:int)->tuple[list[Document],list[Document],dict[str,Any]]:
    if not 0<validation_fraction<1: raise ValueError('Choose validation_fraction strictly between zero and one')
    audit=corpus_audit(documents)
    if audit['duplicate_uids'] or audit['duplicate_content_hashes']: raise ValueError('Deduplicate exact documents before splitting')
    train=[];validation=[];per_source={}
    for source in sorted({document.source for document in documents}):
        group=sorted((document for document in documents if document.source==source),key=lambda value:value.uid)
        random.Random(seed+int(hashlib.sha256(source.encode()).hexdigest()[:8],16)).shuffle(group)
        n_val=max(1,round(len(group)*validation_fraction))
        if n_val>=len(group): raise ValueError(f'Need at least one training document for {source}')
        validation.extend(group[:n_val]);train.extend(group[n_val:])
        per_source[source]={'total':len(group),'training':len(group)-n_val,'validation':n_val}
    if {document.uid for document in train}&{document.uid for document in validation}: raise AssertionError('Split leakage')
    metadata={
        'created_utc':utc_now(),'seed':seed,'validation_fraction':validation_fraction,
        'strategy':'source-stratified deterministic document split','per_source':per_source,
        'dataset_fingerprint':dataset_fingerprint(documents),
        'training_ids':sorted(document.uid for document in train),
        'validation_ids':sorted(document.uid for document in validation),
    }
    return train,validation,metadata

def save_or_verify_split(path:Path,meta:Mapping[str,Any])->None:
    path=Path(path)
    if path.exists():
        old=json.loads(path.read_text(encoding='utf-8'))
        keys=('seed','validation_fraction','strategy','per_source','dataset_fingerprint','training_ids','validation_ids')
        if any(old.get(key)!=meta.get(key) for key in keys): raise RuntimeError('Existing data split differs; refusing overwrite')
        print('Verified existing split',path)
    else: atomic_write_json(path,dict(meta))

def validate_source_mixture(ratios:Mapping[str,float],sources:Iterable[str])->dict[str,float]:
    allowed=set(sources)
    if set(ratios)!=allowed: raise ValueError(f'Mixture keys {set(ratios)} must equal {allowed}')
    values={key:float(value) for key,value in ratios.items()}
    if any(not math.isfinite(value) or value<=0 for value in values.values()): raise ValueError('All source proportions must be finite and > 0')
    if not math.isclose(sum(values.values()),1.0,rel_tol=1e-6,abs_tol=1e-6): raise ValueError('Source proportions must sum to 1.0')
    return values

def source_sampling_indices(documents:Sequence[Document],ratios:Mapping[str,float],samples:int,seed:int)->list[int]:
    if samples<=0: raise ValueError('samples must be >0')
    ratios=validate_source_mixture(ratios,(document.source for document in documents))
    pools={source:[index for index,document in enumerate(documents) if document.source==source] for source in ratios}
    if any(not pool for pool in pools.values()): raise ValueError('Missing source documents')
    rng=random.Random(seed);sources=list(sorted(ratios));weights=[ratios[source] for source in sources]
    return [rng.choice(pools[source]) for source in rng.choices(sources,weights=weights,k=samples)]

def _stable_document_order(documents:Sequence[Document],seed:int)->list[Document]:
    return sorted(documents,key=lambda document:hashlib.sha256(f'{seed}:{document.uid}'.encode()).hexdigest())

def make_continuation_examples(documents:Sequence[Document],prompt_words:int=512,continuation_words:int=128,max_per_source:int=16,seed:int=781)->list[dict[str,str]]:
    result=[];needed=prompt_words+continuation_words
    for source in sorted({document.source for document in documents}):
        collected=0
        for document in _stable_document_order([value for value in documents if value.source==source],seed):
            if document.word_count<needed: continue
            units=document_text(document).split()
            result.append({'id':document.uid,'source':source,'prompt':' '.join(units[:prompt_words]),'reference':' '.join(units[prompt_words:needed])})
            collected+=1
            if collected>=max_per_source: break
    return result

def sampled_cross_split_overlap_audit(train:Sequence[Document],validation:Sequence[Document],seed:int,max_per_split:int=10_000)->dict[str,Any]:
    def signature(document:Document)->str:
        units=document_text(document).lower().split()
        sample=' '.join([*units[:128],*units[-128:]])
        return hashlib.sha256(sample.encode()).hexdigest()
    sampled_train=_stable_document_order(train,seed)[:max_per_split]
    sampled_val=_stable_document_order(validation,seed+1)[:max_per_split]
    train_signatures={signature(document):document.uid for document in sampled_train}
    overlaps=[]
    for document in sampled_val:
        value=signature(document)
        if value in train_signatures: overlaps.append({'training_uid':train_signatures[value],'validation_uid':document.uid})
    return {'method':'first+last 128 normalized whitespace words SHA-256','training_sample':len(sampled_train),'validation_sample':len(sampled_val),'overlaps':overlaps}


In [ ]:
def load_official_training_corpus(cfg: RunConfig, discovered: Mapping[str, Sequence[Path]], schemas: Mapping[str, Any]):
    if cfg.corpus_document_cap_per_source is None:
        raise RuntimeError("Set CFG.corpus_document_cap_per_source explicitly before bounded preprocessing")
    ratios = validate_source_mixture(
        cfg.source_mixing_ratios,
        (Path(filename).stem.removeprefix("train_") for filename in cfg.train_files),
    )
    all_docs, rejections, provenance = [], {}, {}
    store_dir = DIRS["data"] / "bounded_document_store"
    for filename in cfg.train_files:
        path = require_unique_file(discovered, filename)
        source = Path(filename).stem.removeprefix("train_")
        docs, bad, meta = load_bounded_documents(
            path, source, schemas["training"][filename], store_dir,
            cfg.corpus_document_cap_per_source, cfg.seed,
        )
        all_docs.extend(docs)
        rejections[source] = bad
        provenance[source] = {
            "file": filename,
            "source_identity": meta["selection_config"]["source_identity"],
            "total_source_records": meta["total_source_records"],
            "selected": len(docs),
            "rejected_selected_records": len(bad),
            "selection": meta["selection_config"],
        }
    clean, dropped = deduplicate_exact_documents(all_docs)
    audit = corpus_audit(clean)
    audit.update({
        "selected_before_deduplication": len(all_docs),
        "removed_exact_duplicates": len(dropped),
        "requested_sampling_ratios": ratios,
        "sampling_unit": "document draws, not token percentages",
        "official_test_excluded": True,
    })
    train, validation, split = deterministic_document_split(clean, cfg.validation_fraction, cfg.seed)
    audit["training"] = corpus_audit(train)
    audit["validation"] = corpus_audit(validation)
    save_or_verify_split(DIRS["manifests"] / "document_split.json", split)
    overlap = sampled_cross_split_overlap_audit(train, validation, cfg.seed)
    artifacts = {
        "dataset_provenance.json": provenance,
        "dataset_audit.json": audit,
        "rejected_records.json": rejections,
        "deduplication_log.json": dropped,
        "source_sampling_config.json": {"ratios": ratios, "unit": "document draws"},
        "cross_split_overlap_audit.json": overlap,
    }
    for name, value in artifacts.items():
        path = DIRS["manifests"] / name
        if path.exists() and json.loads(path.read_text(encoding="utf-8")) != value:
            raise RuntimeError(f"Persisted manifest changed: {path}")
        if not path.exists():
            atomic_write_json(path, value)
    examples = make_continuation_examples(validation, max_per_source=16, seed=cfg.seed)
    examples_path = DIRS["data"] / "local_continuation_examples.json"
    if examples_path.exists() and json.loads(examples_path.read_text(encoding="utf-8")) != examples:
        raise RuntimeError("Fixed development examples changed")
    if not examples_path.exists():
        atomic_write_json(examples_path, examples)
    atomic_write_json(DIRS["manifests"] / "local_generation_protocol.json", {
        "selection": "up to 16 eligible documents per source in deterministic hash order",
        "seed": cfg.seed, "prompt_words": 512, "reference_words": 128,
        "examples": len(examples), "ids": [row["id"] for row in examples],
    })
    return train, validation, examples, audit

RUN_CORPUS_PREPROCESSING = True
if CFG.execution_mode == "inference_only":
    TRAIN_DOCS = VAL_DOCS = LOCAL_DEV = DATA_AUDIT = None
    print("Inference-only mode: skipped every training-corpus load and preprocessing operation.")
elif RUN_CORPUS_PREPROCESSING:
    if RESOLVED_SCHEMA is None:
        raise RuntimeError("Official file schemas must be resolved before preprocessing")
    TRAIN_DOCS, VAL_DOCS, LOCAL_DEV, DATA_AUDIT = load_official_training_corpus(CFG, DISCOVERED, RESOLVED_SCHEMA)
    print("Loaded bounded official corpus cache:", DATA_AUDIT)
else:
    TRAIN_DOCS = VAL_DOCS = LOCAL_DEV = DATA_AUDIT = None
    print("Corpus preprocessing is opt-in. Review the cap and set RUN_CORPUS_PREPROCESSING=True when needed.")


In [ ]:
print("Train:", len(TRAIN_DOCS))
print("Validation:", len(VAL_DOCS))
print("Local dev:", len(LOCAL_DEV))

## 5. Tokenizer persistence and compatibility

In [ ]:
def directory_fingerprint(path:Path)->str:
    path=Path(path);digest=hashlib.sha256()
    files=sorted(p for p in path.rglob('*') if p.is_file() and p.name!='tokenizer_metadata.json')
    if not files: raise ValueError(f'No tokenizer files found: {path}')
    for file in files: digest.update(file.relative_to(path).as_posix().encode()+bytes.fromhex(sha256_file(file)))
    return digest.hexdigest()
def tokenizer_encode(tok:Any,text:str)->list[int]:
    try: result=tok.encode(text,add_special_tokens=False)
    except TypeError: result=tok.encode(text)
    return list(result)
def tokenizer_decode(tok:Any,ids:Sequence[int])->str:
    try: return tok.decode(list(ids),skip_special_tokens=True)
    except TypeError: return tok.decode(list(ids))
def tokenizer_metadata(tok:Any,location:Path)->dict[str,Any]:
    probes=['The Court considered section 14 and dismissed the appeal.','§ 21—न्यायालय: AIR 2024 SC 17;  multiple   spaces\nand punctuation.']
    results=[]
    for probe in probes:
        ids=tokenizer_encode(tok,probe);decoded=tokenizer_decode(tok,ids)
        if not ids or decoded!=probe: raise ValueError(f'Tokenizer failed exact roundtrip: {probe!r} -> {decoded!r}')
        results.append({'input':probe,'ids':ids,'decoded':decoded})
    special={k:getattr(tok,k,None) for k in ('bos_token_id','eos_token_id','pad_token_id','unk_token_id')}
    if len({value for value in special.values() if value is not None})!=len([value for value in special.values() if value is not None]): raise ValueError('Special token IDs must be distinct')
    return {'created_utc':utc_now(),'vocab_size':len(tok),'special_token_ids':special,'fingerprint':directory_fingerprint(location),'round_trip_probes':results}
def register_tokenizer(tok:Any,destination:Path,save_function:Callable[[Any,Path],Any]|None=None)->dict[str,Any]:
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True);meta_path=destination/'tokenizer_metadata.json'
    if meta_path.exists():
        previous=json.loads(meta_path.read_text(encoding='utf-8'))
        if previous.get('fingerprint')!=directory_fingerprint(destination): raise RuntimeError('Tokenizer registered files have changed')
        if previous.get('vocab_size')!=len(tok): raise RuntimeError('Loaded tokenizer size differs')
        return previous
    if any(destination.iterdir()): raise RuntimeError('Nonempty tokenizer directory; choose explicit restore/adaptation')
    if save_function: save_function(tok,destination)
    elif hasattr(tok,'save_pretrained'): tok.save_pretrained(destination)
    else: raise TypeError('Course tokenizer requires an explicit save_function(tok,destination) adapter')
    meta=tokenizer_metadata(tok,destination);atomic_write_json(meta_path,meta);return meta
def tokenizer_token_audit(docs:Iterable[Document],tok:Any)->dict[str,Any]:
    by_source=Counter(); count=0
    for d in docs: by_source[d.source]+=len(tokenizer_encode(tok,document_text(d)));count+=1
    return {'documents':count,'tokens':sum(by_source.values()),'tokens_by_source':dict(by_source)}

COURSE_BPE_SOURCE ='"""Deterministic byte-level BPE tokenizer used by the Phase 1 notebook.\n\nThe implementation is deliberately small: it wraps the `tokenizers` BPE model,\nexposes the interface expected by the notebook, and persists every artifact\nneeded to recreate the token-to-ID mapping in a later Kaggle session.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\n\nSPECIAL_TOKENS = ("<pad>", "<unk>", "<bos>", "<eos>")\n\n\nclass CourseBPETokenizer:\n    """Minimal tokenizer adapter with a stable encode/decode/save contract."""\n\n    def __init__(self, backend):\n        self.backend = backend\n        self.pad_token_id = backend.token_to_id(SPECIAL_TOKENS[0])\n        self.unk_token_id = backend.token_to_id(SPECIAL_TOKENS[1])\n        self.bos_token_id = backend.token_to_id(SPECIAL_TOKENS[2])\n        self.eos_token_id = backend.token_to_id(SPECIAL_TOKENS[3])\n\n    def __len__(self) -> int:\n        return int(self.backend.get_vocab_size())\n\n    def encode(self, text: str, add_special_tokens: bool = False) -> list[int]:\n        if not isinstance(text, str):\n            raise TypeError("text must be a string")\n        ids = list(self.backend.encode(text, add_special_tokens=False).ids)\n        if add_special_tokens:\n            ids = [self.bos_token_id, *ids, self.eos_token_id]\n        return ids\n\n    def decode(self, ids: Sequence[int], skip_special_tokens: bool = True) -> str:\n        return self.backend.decode([int(value) for value in ids], skip_special_tokens=skip_special_tokens)\n\n    @classmethod\n    def train(\n        cls,\n        texts: Iterable[str],\n        *,\n        vocab_size: int = 32_000,\n        min_frequency: int = 2,\n    ) -> "CourseBPETokenizer":\n        try:\n            from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers\n        except ImportError as exc:\n            raise RuntimeError(\n                "The `tokenizers` package is required. Kaggle normally includes it; "\n                "otherwise install a pinned version before this cell."\n            ) from exc\n\n        if vocab_size < 1_024:\n            raise ValueError("vocab_size must be at least 1024")\n        if min_frequency < 1:\n            raise ValueError("min_frequency must be positive")\n\n        backend = Tokenizer(models.BPE(unk_token=SPECIAL_TOKENS[1]))\n        backend.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True)\n        backend.decoder = decoders.ByteLevel()\n        trainer = trainers.BpeTrainer(\n            vocab_size=int(vocab_size),\n            min_frequency=int(min_frequency),\n            special_tokens=list(SPECIAL_TOKENS),\n            initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),\n            show_progress=True,\n        )\n        backend.train_from_iterator((str(text) for text in texts), trainer=trainer)\n        tokenizer = cls(backend)\n        if len(tokenizer) != vocab_size:\n            raise RuntimeError(\n                f"BPE training produced {len(tokenizer)} tokens, expected exactly {vocab_size}. "\n                "Inspect the corpus and do not begin model training with a changing vocabulary."\n            )\n        return tokenizer\n\n    def save_pretrained(self, destination: str | Path) -> tuple[str, ...]:\n        destination = Path(destination)\n        destination.mkdir(parents=True, exist_ok=True)\n        tokenizer_json = destination / "tokenizer.json"\n        self.backend.save(str(tokenizer_json))\n        # Save the explicit BPE vocabulary and merge table as well as the unified\n        # tokenizer.json, so the learned mapping is auditable without code.\n        model_files = tuple(self.backend.model.save(str(destination), "course_bpe"))\n        config = {\n            "tokenizer_class": type(self).__name__,\n            "tokenizer_file": tokenizer_json.name,\n            "model_type": "byte_level_bpe",\n            "special_tokens": list(SPECIAL_TOKENS),\n            "special_token_ids": {\n                "pad_token_id": self.pad_token_id,\n                "unk_token_id": self.unk_token_id,\n                "bos_token_id": self.bos_token_id,\n                "eos_token_id": self.eos_token_id,\n            },\n            "vocab_size": len(self),\n        }\n        (destination / "tokenizer_config.json").write_text(\n            json.dumps(config, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8"\n        )\n        return (str(tokenizer_json), *model_files, str(destination / "tokenizer_config.json"))\n\n    @classmethod\n    def from_pretrained(cls, location: str | Path) -> "CourseBPETokenizer":\n        try:\n            from tokenizers import Tokenizer\n        except ImportError as exc:\n            raise RuntimeError("The `tokenizers` package is required to restore the BPE tokenizer") from exc\n\n        location = Path(location)\n        tokenizer_path = location / "tokenizer.json"\n        config_path = location / "tokenizer_config.json"\n        if not tokenizer_path.is_file() or not config_path.is_file():\n            raise FileNotFoundError(f"Incomplete tokenizer directory: {location}")\n        config = json.loads(config_path.read_text(encoding="utf-8"))\n        tokenizer = cls(Tokenizer.from_file(str(tokenizer_path)))\n        if len(tokenizer) != int(config["vocab_size"]):\n            raise RuntimeError("Saved tokenizer vocabulary size does not match tokenizer_config.json")\n        actual_ids = {\n            "pad_token_id": tokenizer.pad_token_id,\n            "unk_token_id": tokenizer.unk_token_id,\n            "bos_token_id": tokenizer.bos_token_id,\n            "eos_token_id": tokenizer.eos_token_id,\n        }\n        if actual_ids != config["special_token_ids"]:\n            raise RuntimeError("Saved tokenizer special-token mapping is inconsistent")\n        return tokenizer\n'

TOKENIZER_TRAINING_CONFIG={
    'type':'byte_level_bpe','vocab_size':32_000,'min_frequency':2,
    'training_documents':'TRAIN_DOCS only','special_tokens':['<pad>','<unk>','<bos>','<eos>'],
}
RUN_TOKENIZER_TRAINING=False

def install_python_source(path:Path,source:str)->None:
    path=Path(path)
    if path.exists() and path.read_text(encoding='utf-8')!=source and any(DIRS['checkpoints'].glob('*.pt')):
        raise RuntimeError(f'Refusing to change source after checkpoints exist: {path}')
    if not path.exists() or path.read_text(encoding='utf-8')!=source: atomic_write_bytes(path,source.encode('utf-8'))

def import_source_module(module_name:str,path:Path):
    spec=importlib.util.spec_from_file_location(module_name,path)
    if spec is None or spec.loader is None: raise ImportError(f'Cannot import {path}')
    module=importlib.util.module_from_spec(spec);sys.modules[module_name]=module;spec.loader.exec_module(module);return module

BPE_SOURCE_PATH=DIRS['model']/'course_bpe.py';install_python_source(BPE_SOURCE_PATH,COURSE_BPE_SOURCE)
course_tokenizer_import = Path(CFG.tokenizer_import_path).expanduser() if CFG.tokenizer_import_path else None
if course_tokenizer_import is not None and not (DIRS['tokenizer']/'tokenizer.json').exists():
    if not course_tokenizer_import.is_dir(): raise FileNotFoundError(course_tokenizer_import)
    if any(DIRS['tokenizer'].iterdir()): raise RuntimeError('Tokenizer destination is nonempty')
    for source_file in course_tokenizer_import.iterdir():
        if source_file.is_file(): shutil.copy2(source_file, DIRS['tokenizer']/source_file.name)
    print('Imported the frozen Submission 01/course tokenizer from',course_tokenizer_import)
course_bpe=import_source_module('cs781_course_bpe',BPE_SOURCE_PATH)
if (DIRS['tokenizer']/'tokenizer.json').exists():
    tokenizer=course_bpe.CourseBPETokenizer.from_pretrained(DIRS['tokenizer'])
elif TRAIN_DOCS is not None and RUN_TOKENIZER_TRAINING:
    tokenizer=course_bpe.CourseBPETokenizer.train(
        (document_text(document) for document in TRAIN_DOCS),
        vocab_size=TOKENIZER_TRAINING_CONFIG['vocab_size'],min_frequency=TOKENIZER_TRAINING_CONFIG['min_frequency'],
    )
else: tokenizer=None

if tokenizer is not None:
    TOKENIZER_META=register_tokenizer(tokenizer,DIRS['tokenizer'])
    audit_docs=TRAIN_DOCS[:min(256,len(TRAIN_DOCS))] if TRAIN_DOCS else []
    TOKENIZER_AUDIT=tokenizer_token_audit(audit_docs,tokenizer) if audit_docs else None
    if TOKENIZER_AUDIT is not None:
        TOKENIZER_AUDIT['scope']='deterministic first 256 training documents (or all when fewer)'
        atomic_write_json(DIRS['manifests']/'tokenizer_audit.json',TOKENIZER_AUDIT)
    print('Restored/registered tokenizer:',TOKENIZER_META)
else:
    TOKENIZER_META=None;TOKENIZER_AUDIT=None
    print('No saved/imported tokenizer found. Set CFG.tokenizer_import_path, or explicitly authorize one tokenizer-training run.')


## 6. Student-owned encoder-decoder model and enforced contract

For contiguous training tokens split into source `x` and target `y`, the intended model family is:

- `H = Encoder(x)`, where encoder self-attention may use both left and right source context.
- The decoder uses causal self-attention over `y_<t` and cross-attention to `H`.
- `p(y_t | y_<t, x) = softmax(W h_t)`.
- Teacher-forced loss is the mean negative log-likelihood over non-padding target tokens.

This description establishes the experiment contract, not its assessed implementation. Your model module must
independently define `build_model(config)` and either expose a standard forward method accepting
`encoder_input_ids`, `decoder_input_ids`, and `encoder_attention_mask`, or define
`forward_adapter(model, encoder_input_ids, decoder_input_ids, encoder_attention_mask)`.
The exact layer design, mathematical derivation, rationale, and numerical configuration must be yours.


In [ ]:
# The standalone module exposes make_model_config(vocab_size), so no notebook-only
# override is needed; checkpoint reconstruction uses the source module plus the saved dict.
STUDENT_MODEL_CONFIG_OVERRIDE = None
MODEL = MODEL_CONFIG = RECONSTRUCT_MODEL = MODEL_MODULE = None
MODEL_SOURCE_PATH = MODEL_SOURCE_SHA256 = None

def install_student_model_source(source_path: Path) -> Path:
    source_path = Path(source_path).expanduser().resolve()
    if not source_path.is_file():
        raise FileNotFoundError(source_path)
    tree = ast.parse(source_path.read_text(encoding="utf-8"), filename=str(source_path))
    banned = {"transformers", "fairseq"}
    imported = {
        alias.name.split(".")[0]
        for node in ast.walk(tree) if isinstance(node, ast.Import)
        for alias in node.names
    }
    imported |= {
        (node.module or "").split(".")[0]
        for node in ast.walk(tree) if isinstance(node, ast.ImportFrom)
    }
    if imported & banned:
        raise RuntimeError(f"Pretrained-model libraries are forbidden in the submitted model source: {imported & banned}")
    destination = DIRS["model"] / "encoder_decoder_lm.py"
    payload = source_path.read_bytes()
    if destination.exists() and destination.read_bytes() != payload and any(DIRS["checkpoints"].glob("*.pt")):
        raise RuntimeError("Refusing to change model source after checkpoints exist")
    atomic_write_bytes(destination, payload)
    return destination

def seq2seq_logits(model, encoder_input_ids, decoder_input_ids, encoder_attention_mask=None):
    if MODEL_MODULE is not None and callable(getattr(MODEL_MODULE, "forward_adapter", None)):
        output = MODEL_MODULE.forward_adapter(model, encoder_input_ids, decoder_input_ids, encoder_attention_mask)
    else:
        output = model(
            encoder_input_ids=encoder_input_ids,
            decoder_input_ids=decoder_input_ids,
            encoder_attention_mask=encoder_attention_mask,
        )
    return output.logits if hasattr(output, "logits") else output[0] if isinstance(output, (tuple, list)) else output

def count_parameters(model: torch.nn.Module) -> dict[str, int]:
    return {
        "total": sum(parameter.numel() for parameter in model.parameters()),
        "trainable": sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad),
    }

def validate_seq2seq_contract(model, vocab_size: int, device: str, source_length: int = 12, target_length: int = 7):
    model = model.to(device)
    model.train(); model.zero_grad(set_to_none=True)
    encoder_ids = torch.randint(0, vocab_size, (2, source_length), dtype=torch.long, device=device)
    decoder_ids = torch.randint(0, vocab_size, (2, target_length), dtype=torch.long, device=device)
    encoder_mask = torch.ones_like(encoder_ids, dtype=torch.bool)
    logits = seq2seq_logits(model, encoder_ids, decoder_ids, encoder_mask)
    expected = (2, target_length, vocab_size)
    if tuple(logits.shape) != expected:
        raise ValueError(f"Expected logits shape {expected}, got {tuple(logits.shape)}")
    if not torch.isfinite(logits).all():
        raise FloatingPointError("Non-finite logits")
    labels = torch.randint(0, vocab_size, (2, target_length), dtype=torch.long, device=device)
    loss = torch.nn.functional.cross_entropy(logits.reshape(-1, vocab_size), labels.reshape(-1))
    loss.backward()
    gradients = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
    if not gradients or not all(torch.isfinite(g).all().item() for g in gradients):
        raise FloatingPointError("Missing or non-finite gradients")
    counts = count_parameters(model)
    if not 50_000_000 <= counts["total"] <= 300_000_000:
        raise ValueError(f"Official parameter bound violated: {counts['total']:,}")
    model.zero_grad(set_to_none=True)
    return {"logits_shape": list(logits.shape), "loss": float(loss.detach().cpu()), **counts, "device": device}

_restored_model_source = DIRS["model"] / "encoder_decoder_lm.py"
_configured_model_source = (
    Path(CFG.student_model_source_path).expanduser()
    if CFG.student_model_source_path
    else _restored_model_source if _restored_model_source.exists() else None
)
if _configured_model_source is not None:
    if TOKENIZER_META is None:
        raise RuntimeError("Register/restore the tokenizer before importing the student model")
    MODEL_SOURCE_PATH = install_student_model_source(_configured_model_source)
    MODEL_SOURCE_SHA256 = sha256_file(MODEL_SOURCE_PATH)
    MODEL_MODULE = import_source_module(f"cs781_submission02_model_{MODEL_SOURCE_SHA256[:12]}", MODEL_SOURCE_PATH)
    if not callable(getattr(MODEL_MODULE, "build_model", None)):
        raise AttributeError("Student module must define build_model(config)")
    if STUDENT_MODEL_CONFIG_OVERRIDE is not None:
        MODEL_CONFIG = dict(STUDENT_MODEL_CONFIG_OVERRIDE)
    elif callable(getattr(MODEL_MODULE, "make_model_config", None)):
        MODEL_CONFIG = dict(MODEL_MODULE.make_model_config(int(TOKENIZER_META["vocab_size"])))
    else:
        candidate = getattr(MODEL_MODULE, "MODEL_CONFIG", None)
        if not isinstance(candidate, Mapping):
            raise RuntimeError("Provide STUDENT_MODEL_CONFIG_OVERRIDE, make_model_config(vocab_size), or module MODEL_CONFIG")
        MODEL_CONFIG = dict(candidate)
    if int(MODEL_CONFIG.get("vocab_size", -1)) != int(TOKENIZER_META["vocab_size"]):
        raise ValueError("Model vocabulary does not match the frozen tokenizer")
    RECONSTRUCT_MODEL = MODEL_MODULE.build_model
    MODEL = RECONSTRUCT_MODEL(MODEL_CONFIG)
    MODEL_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    MODEL_REPORT = validate_seq2seq_contract(MODEL, TOKENIZER_META["vocab_size"], MODEL_DEVICE)
    MODEL_REPORT.update({
        "architecture_family": "encoder-decoder Transformer",
        "model_source_sha256": MODEL_SOURCE_SHA256,
        "model_config": MODEL_CONFIG,
    })
    atomic_write_json(DIRS["manifests"] / "model_validation.json", MODEL_REPORT)
    MODEL.to("cpu")
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("Student model validation:", json.dumps(MODEL_REPORT, indent=2))
else:
    MODEL_REPORT = None
    print("STUDENT ACTION REQUIRED: set CFG.student_model_source_path to the independently authored encoder-decoder module.")


## 7. Stateful sampling, histories, checkpoints, and progress

In [ ]:
class StatefulRandomSampler(Sampler[int]):
    # Cursor is updated when DataLoader requests indices, possibly ahead of consumed batches if num_workers>0.
    # For exact checkpoint continuation use num_workers=0 or save a committed (consumed) batch cursor.
    def __init__(self,data_source:Sequence[Any],seed:int,epoch:int=0,cursor:int=0):
        self.length=len(data_source);self.seed=int(seed);self.epoch=int(epoch);self.cursor=int(cursor)
    def _order(self)->list[int]:
        return torch.randperm(self.length,generator=torch.Generator().manual_seed(self.seed+self.epoch)).tolist()
    def __iter__(self):
        for position in range(self.cursor,self.length):
            self.cursor=position+1
            yield self._order_cache[position] if hasattr(self,'_order_cache') else self._get_order_item(position)
    def _get_order_item(self,position):
        if not hasattr(self,'_order_cache'): self._order_cache=self._order()
        return self._order_cache[position]
    def __len__(self): return max(0,self.length-self.cursor)
    def next_epoch(self): self.epoch+=1;self.cursor=0;self.__dict__.pop('_order_cache',None)
    def state_dict(self): return {'length':self.length,'seed':self.seed,'epoch':self.epoch,'cursor':self.cursor}
    def load_state_dict(self,state:Mapping[str,Any]):
        if int(state['length'])!=self.length or int(state['seed'])!=self.seed: raise ValueError('Incompatible sampler')
        cursor=int(state['cursor'])
        if not 0<=cursor<=self.length: raise ValueError('Sampler cursor outside dataset')
        self.epoch=int(state['epoch']);self.cursor=cursor;self.__dict__.pop('_order_cache',None)

class JsonlHistory:
    def __init__(self,path:Path): self.path=Path(path);self.path.parent.mkdir(parents=True,exist_ok=True)
    def records(self)->list[dict[str,Any]]:
        if not self.path.exists(): return []
        lines=self.path.read_text(encoding='utf-8').splitlines();rows=[]
        for i,line in enumerate(lines):
            if not line.strip():continue
            try: rows.append(json.loads(line))
            except json.JSONDecodeError:
                if i==len(lines)-1: raise RuntimeError('Incomplete last history line. Restore last checkpoint/history or explicitly repair; do not silently discard')
                raise
        steps=[r['step'] for r in rows]
        if len(steps)!=len(set(steps)) or steps!=sorted(steps): raise ValueError('Duplicate or nonmonotonic history steps')
        return rows
    def append(self,record:Mapping[str,Any]):
        existing=self.records()
        if existing and int(record['step'])<=int(existing[-1]['step']): raise ValueError('Refusing duplicated/nonmonotonic history step')
        with self.path.open('ab') as h:
            h.write((json.dumps(dict(record),ensure_ascii=False)+'\n').encode());h.flush();os.fsync(h.fileno())

In [ ]:
CHECKPOINT_VERSION=2
def capture_rng_state(device:str|None=None)->dict[str,Any]:
    gpu=None
    if device and str(device).startswith('cuda') and torch.cuda.is_available():
        gpu={'index':torch.device(device).index or 0,'state':torch.cuda.get_rng_state(torch.device(device))}
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch_cpu':torch.get_rng_state(),'torch_cuda_device':gpu}
def restore_rng_state(state:Mapping[str,Any])->None:
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch_cpu'])
    gpu=state.get('torch_cuda_device')
    if gpu is not None:
        if not torch.cuda.is_available() or gpu['index']>=torch.cuda.device_count(): raise RuntimeError('Saved CUDA RNG device unavailable')
        torch.cuda.set_rng_state(gpu['state'],device=gpu['index'])
def save_checkpoint_atomic(path:Path,*,model:torch.nn.Module,optimizer:torch.optim.Optimizer,scheduler:Any,scaler:Any,sampler:StatefulRandomSampler,progress:Mapping[str,Any],model_config:Mapping[str,Any],tokenizer_fingerprint:str,dataset_fingerprint_value:str,training_config:Mapping[str,Any],history_path:Path,validation_history:Sequence[Mapping[str,Any]],experiment_id:str,device:str|None=None,model_source_sha256:str|None=None)->dict[str,Any]:
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    hist=sha256_file(history_path) if Path(history_path).exists() else None
    state={'checkpoint_version':CHECKPOINT_VERSION,'experiment_id':experiment_id,'created_utc':utc_now(),'model_state':model.state_dict(),'optimizer_state':optimizer.state_dict(),'scheduler_state':scheduler.state_dict() if scheduler is not None else None,'scaler_state':scaler.state_dict() if scaler is not None else None,'sampler_state':sampler.state_dict(),'rng_state':capture_rng_state(device),'progress':dict(progress),'model_config':dict(model_config),'tokenizer_fingerprint':tokenizer_fingerprint,'dataset_fingerprint':dataset_fingerprint_value,'training_config':dict(training_config),'history_sha256':hist,'validation_history':list(validation_history),'model_source_sha256':model_source_sha256}
    with tempfile.NamedTemporaryFile(dir=path.parent,suffix='.pt',delete=False) as h: tmp=Path(h.name)
    try:
        torch.save(state,tmp)
        check=torch.load(tmp,map_location='cpu',weights_only=False)
        if check.get('checkpoint_version')!=CHECKPOINT_VERSION or check.get('progress')!=dict(progress): raise RuntimeError('Checkpoint readback failed')
        digest=sha256_file(tmp)
        if path.exists():
            previous=path.with_name('previous_known_good.pt')
            with tempfile.NamedTemporaryFile(dir=path.parent,suffix='.pt',delete=False) as h: prev_tmp=Path(h.name)
            try:
                shutil.copyfile(path,prev_tmp)
                torch.load(prev_tmp,map_location='cpu',weights_only=False)
                os.replace(prev_tmp,previous)
            finally: prev_tmp.unlink(missing_ok=True)
        os.replace(tmp,path)
        return {'path':str(path),'sha256':digest,'step':progress.get('step')}
    finally: tmp.unlink(missing_ok=True)

In [ ]:
def restore_checkpoint(path:Path,*,model:torch.nn.Module,optimizer:torch.optim.Optimizer,sampler:StatefulRandomSampler,expected_tokenizer_fingerprint:str,expected_dataset_fingerprint:str,scheduler:Any=None,scaler:Any=None,expected_experiment_id:str|None=None,expected_model_config:Mapping[str,Any]|None=None,history_path:Path|None=None,device:str|None=None,expected_model_source_sha256:str|None=None)->dict[str,Any]:
    state=torch.load(Path(path),map_location='cpu',weights_only=False) # Only load your own trusted checkpoint files.
    if state.get('checkpoint_version')!=CHECKPOINT_VERSION: raise ValueError('Checkpoint schema mismatch')
    if expected_experiment_id is not None and state['experiment_id']!=expected_experiment_id: raise ValueError('Experiment ID mismatch')
    if state['tokenizer_fingerprint']!=expected_tokenizer_fingerprint: raise ValueError('Tokenizer fingerprint mismatch')
    if state['dataset_fingerprint']!=expected_dataset_fingerprint: raise ValueError('Dataset fingerprint mismatch')
    if expected_model_config is not None and dict(expected_model_config)!=state['model_config']: raise ValueError('Architecture config mismatch')
    if expected_model_source_sha256 is not None and state.get('model_source_sha256')!=expected_model_source_sha256: raise ValueError('Model source fingerprint mismatch')
    if history_path is not None:
        actual=sha256_file(history_path) if Path(history_path).exists() else None
        if actual!=state['history_sha256']: raise ValueError('History file differs from checkpoint; restore matching history or explicitly reconcile after crash')
    if (scheduler is None)!=(state['scheduler_state'] is None): raise ValueError('Scheduler state incompatibility')
    if (scaler is None)!=(state['scaler_state'] is None): raise ValueError('AMP scaler state incompatibility')
    model.load_state_dict(state['model_state'],strict=True);optimizer.load_state_dict(state['optimizer_state'])
    if device is not None:
        for opt_state in optimizer.state.values():
            for k,v in opt_state.items():
                if isinstance(v,torch.Tensor): opt_state[k]=v.to(device)
    if scheduler is not None: scheduler.load_state_dict(state['scheduler_state'])
    if scaler is not None: scaler.load_state_dict(state['scaler_state'])
    sampler.load_state_dict(state['sampler_state']);restore_rng_state(state['rng_state'])
    return state

@dataclass
class ProgressState:
    experiment_id:str
    device:str
    total_steps:int
    step:int=0
    epoch:int=0
    tokens_processed:int=0
    cumulative_seconds:float=0.0
    last_checkpoint_step:int=0

class ProgressReporter:
    def __init__(self,state:ProgressState):
        self.state=state;self.start=time.perf_counter();self.base_cumulative=state.cumulative_seconds;self.last=self.start;self.last_step=state.step;self.last_tokens=state.tokens_processed
        self.bar=tqdm(total=state.total_steps,initial=state.step,desc=f'{state.experiment_id} [{state.device}]')
    def update(self,*,step:int,epoch:int,loss:float,learning_rate:float,tokens_processed:int,gradient_norm:float|None):
        now=time.perf_counter();dt=max(now-self.last,1e-9);ds=step-self.last_step;dtoken=tokens_processed-self.last_tokens
        if ds<0 or dtoken<0: raise ValueError('Progress cannot go backwards')
        cumulative=self.base_cumulative+(now-self.start);rate=ds/dt
        self.bar.update(ds);self.bar.set_postfix({'epoch':epoch,'loss':f'{loss:.4f}','lr':f'{learning_rate:.3g}','tok':tokens_processed,'tok/s':f'{dtoken/dt:.0f}','step/s':f'{rate:.2f}','grad':'NA' if gradient_norm is None else f'{gradient_norm:.3f}','elapsed_s':f'{cumulative:.0f}','eta_s':f'{max(0,self.state.total_steps-step)/rate:.0f}' if rate>0 else 'inf','ckpt':self.state.last_checkpoint_step})
        self.state.step=step;self.state.epoch=epoch;self.state.tokens_processed=tokens_processed;self.state.cumulative_seconds=cumulative;self.last=now;self.last_step=step;self.last_tokens=tokens_processed
    def close(self): self.bar.close()

def reconcile_history_to_checkpoint(checkpoint_path:Path,history_path:Path,*,apply:bool=False)->dict[str,Any]:
    checkpoint=torch.load(Path(checkpoint_path),map_location='cpu',weights_only=False)
    expected=checkpoint.get('history_sha256');history_path=Path(history_path)
    payload=history_path.read_bytes() if history_path.exists() else b''
    if (hashlib.sha256(payload).hexdigest() if payload else None)==expected:
        return {'status':'already_matching','kept_bytes':len(payload),'discarded_bytes':0}
    boundaries=[0];boundaries.extend(index+1 for index,value in enumerate(payload) if value==10)
    match=None
    for boundary in boundaries:
        prefix=payload[:boundary];digest=hashlib.sha256(prefix).hexdigest() if prefix else None
        if digest==expected: match=boundary
    if match is None: raise RuntimeError('History is not a strict extension of the checkpoint history; restore a matching recovery archive')
    report={'status':'rollback_available','kept_bytes':match,'discarded_bytes':len(payload)-match,'checkpoint_step':checkpoint['progress']['step']}
    if apply:
        backup=history_path.with_name(f"{history_path.name}.uncommitted.after_step_{checkpoint['progress']['step']}.{int(time.time())}")
        atomic_write_bytes(backup,payload);atomic_write_bytes(history_path,payload[:match]);report.update(status='rolled_back',backup=str(backup))
    return report


## 8. Recovery/bootstrap discovery

Both `training_recovery.latest.zip` and `phase1_submission02_recovery.zip` use the same member-level SHA-256 integrity manifest. Verification and restoration stream large checkpoint files rather than loading them fully into RAM.


In [ ]:
RECOVERY_MANIFEST_MEMBER = "manifests/recovery_integrity.json"

def validate_zip(path: Path) -> dict[str, Any]:
    with zipfile.ZipFile(path, "r") as archive:
        bad = archive.testzip()
        if bad is not None:
            raise zipfile.BadZipFile(f"CRC failure: {bad}")
        return {"path": str(path), "members": archive.namelist(), "sha256": sha256_file(path)}

def discover_recovery_archives(input_root: Path) -> list[Path]:
    if not Path(input_root).exists():
        return []
    names = {"phase1_submission02_recovery.zip", "training_recovery.latest.zip"}
    return sorted(path for path in Path(input_root).rglob("*.zip") if path.name in names)

RECOVERY_CANDIDATES = discover_recovery_archives(Path(CFG.input_root))
print("Recovery candidates:", [str(path) for path in RECOVERY_CANDIDATES])
if len(RECOVERY_CANDIDATES) > 1:
    print("Multiple recovery archives found; choose the intended full or periodic archive explicitly.")

def verify_recovery_archive(path: Path) -> dict[str, Any]:
    import stat
    path = Path(path)
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("Duplicate ZIP member names")
        if RECOVERY_MANIFEST_MEMBER not in names:
            raise ValueError("No recovery member SHA manifest")
        manifest = json.loads(archive.read(RECOVERY_MANIFEST_MEMBER))
        expected = {row["path"]: row for row in manifest["files"]}
        actual = {
            name for name in names
            if not name.endswith("/") and name != RECOVERY_MANIFEST_MEMBER
        }
        if set(expected) != actual:
            raise ValueError(
                f"Manifest/archive mismatch: missing={set(expected) - actual}, "
                f"extras={actual - set(expected)}"
            )
        for member in archive.infolist():
            member_path = Path(member.filename)
            if (
                member_path.is_absolute()
                or ".." in member_path.parts
                or not member.filename
                or member.filename.startswith("\\")
            ):
                raise ValueError("Unsafe member path")
            if stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("Symlink in archive is forbidden")
        for name, item in expected.items():
            digest = hashlib.sha256()
            with archive.open(name, "r") as source:
                for chunk in iter(lambda: source.read(1 << 20), b""):
                    digest.update(chunk)
            if digest.hexdigest() != item["sha256"]:
                raise ValueError(f"SHA mismatch: {name}")
            if archive.getinfo(name).file_size != item["bytes"]:
                raise ValueError(f"Size mismatch: {name}")
    return {
        "path": str(path),
        "members": names,
        "sha256": sha256_file(path),
        "integrity_manifest": "PASS",
        "verified_files": len(expected),
    }

def recovery_integrity_manifest(root: Path, files: Sequence[Path]) -> dict[str, Any]:
    root = Path(root).resolve()
    rows = []
    seen = set()
    for source in sorted({Path(item).resolve() for item in files}):
        if not source.is_file() or source.is_symlink():
            raise ValueError(f"Recovery member must be a regular file: {source}")
        if root not in source.parents:
            raise ValueError(f"Recovery member is outside root: {source}")
        relative = source.relative_to(root).as_posix()
        if relative == RECOVERY_MANIFEST_MEMBER or relative in seen:
            raise ValueError(f"Reserved or duplicate recovery member: {relative}")
        seen.add(relative)
        rows.append({"path": relative, "bytes": source.stat().st_size, "sha256": sha256_file(source)})
    return {"format": "cs781_recovery_integrity_v1", "created_utc": utc_now(), "files": rows}

def write_verified_recovery_archive(
    root: Path,
    files: Sequence[Path],
    output: Path,
    *,
    manifest: Mapping[str, Any] | None = None,
) -> dict[str, Any]:
    root = Path(root).resolve()
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    manifest = dict(manifest or recovery_integrity_manifest(root, files))
    by_relative = {
        Path(item).resolve().relative_to(root).as_posix(): Path(item).resolve()
        for item in files
    }
    if set(by_relative) != {row["path"] for row in manifest["files"]}:
        raise ValueError("Recovery file list does not match its integrity manifest")
    with tempfile.NamedTemporaryFile(dir=output.parent, suffix=".zip", delete=False) as handle:
        temporary = Path(handle.name)
    try:
        with zipfile.ZipFile(temporary, "w", allowZip64=True) as archive:
            for row in manifest["files"]:
                source = by_relative[row["path"]]
                # Tensor checkpoints barely compress, so store them directly.
                compression = zipfile.ZIP_STORED if source.suffix == ".pt" else zipfile.ZIP_DEFLATED
                archive.write(source, row["path"], compress_type=compression)
            archive.writestr(
                RECOVERY_MANIFEST_MEMBER,
                json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
                compress_type=zipfile.ZIP_DEFLATED,
            )
        report = verify_recovery_archive(temporary)
        os.replace(temporary, output)
        report["path"] = str(output)
        return report
    finally:
        temporary.unlink(missing_ok=True)

def restore_recovery_archive(
    archive_path: Path, destination: Path, *, allow_existing_identical: bool = True
) -> dict[str, Any]:
    verification = verify_recovery_archive(archive_path)
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    restored = 0
    with zipfile.ZipFile(archive_path) as archive:
        manifest = json.loads(archive.read(RECOVERY_MANIFEST_MEMBER))
        expected = {row["path"]: row for row in manifest["files"]}
        # Preflight every path and conflict before extracting anything.
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f"Unsafe path: {member.filename}")
            if member.is_dir():
                continue
            if target.exists():
                if not target.is_file():
                    raise RuntimeError(f"Restore path is not a file: {target}")
                archive_sha = (
                    expected[member.filename]["sha256"]
                    if member.filename in expected
                    else hashlib.sha256(archive.read(member.filename)).hexdigest()
                )
                if not allow_existing_identical or archive_sha != sha256_file(target):
                    raise RuntimeError(f"Restore conflict; refusing to overwrite existing file: {target}")
        for member in archive.infolist():
            if member.is_dir():
                continue
            target = destination / member.filename
            if target.exists():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with tempfile.NamedTemporaryFile(dir=target.parent, delete=False) as handle:
                temporary = Path(handle.name)
                with archive.open(member, "r") as source:
                    shutil.copyfileobj(source, handle, length=1 << 20)
                handle.flush()
                os.fsync(handle.fileno())
            try:
                os.replace(temporary, target)
            finally:
                temporary.unlink(missing_ok=True)
            restored += 1
    return {**verification, "destination": str(destination), "restored_files": restored}


## 9. Disposable smoke tests before full training

In [ ]:
def run_infrastructure_smoke_tests() -> dict[str, str]:
    results: dict[str, str] = {}
    with tempfile.TemporaryDirectory() as temporary_name:
        temporary = Path(temporary_name)
        fixture_texts = [
            " ".join(["alpha"] * 700),
            " ".join(["beta"] * 700),
            " ".join(["gamma"] * 700),
        ]
        docs = [
            Document(
                uid=f"fixture:row-{index:012d}", text=text, source="fixture",
                raw_id=f"row-{index:012d}", content_hash=hashlib.sha256(text.encode()).hexdigest(),
                word_count=len(text.split()),
            )
            for index, text in enumerate(fixture_texts)
        ]
        assert stable_raw_id({"_record_index": 0}, 0, {"id_strategy": "record_index", "id_field": None}) == "row-000000000000"
        results["idless_training_schema_strategy"] = "PASS"
        train, validation, split = deterministic_document_split(docs, 1 / 3, 781)
        assert not ({doc.uid for doc in train} & {doc.uid for doc in validation})
        save_or_verify_split(temporary / "split.json", split)
        results["document_split_and_leakage"] = "PASS"
        examples = make_continuation_examples(validation)
        assert len(examples) == 1 and len(examples[0]["prompt"].split()) == 512
        assert len(examples[0]["reference"].split()) == 128
        results["local_continuation_examples"] = "PASS"
        sampler = StatefulRandomSampler(range(10), seed=9)
        iterator = iter(sampler); prefix = [next(iterator) for _ in range(4)]
        state = sampler.state_dict(); resumed = StatefulRandomSampler(range(10), seed=9)
        resumed.load_state_dict(state); suffix = list(resumed)
        assert len(prefix + suffix) == 10 and len(set(prefix + suffix)) == 10
        results["sampler_continuation"] = "PASS"
        history = JsonlHistory(temporary / "history.jsonl")
        history.append({"step": 1, "loss": 1.0})
        assert history.records() == [{"step": 1, "loss": 1.0}]
        results["history_continuity"] = "PASS"
        dummy_prediction = {"1": {"text": " ".join(["word"] * 128)}}
        prediction_path = temporary / "prediction.json"; atomic_write_json(prediction_path, dummy_prediction)
        zip_path = temporary / "submission.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
            archive.write(prediction_path, "prediction.json")
        assert validate_zip(zip_path)["members"] == ["prediction.json"]
        results["zip_integrity"] = "PASS"
        recovery_root = temporary / "recovery_root"
        recovery_root.mkdir()
        payload_path = recovery_root / "payload.txt"
        payload_path.write_text("verified recovery payload", encoding="utf-8")
        recovery_zip = temporary / "training_recovery.latest.zip"
        write_verified_recovery_archive(recovery_root, [payload_path], recovery_zip)
        restored_root = temporary / "restored"
        restore_report = restore_recovery_archive(recovery_zip, restored_root)
        assert (restored_root / "payload.txt").read_text(encoding="utf-8") == "verified recovery payload"
        assert restore_report["integrity_manifest"] == "PASS"
        results["periodic_recovery_roundtrip"] = "PASS"
    return results

SMOKE_RESULTS = run_infrastructure_smoke_tests()
print(json.dumps(SMOKE_RESULTS, indent=2))


In [ ]:
def validate_decoder_causality(model, vocab_size: int, device: str) -> None:
    model.eval()
    encoder = torch.randint(0, vocab_size, (1, 11), dtype=torch.long, device=device)
    first = torch.randint(0, vocab_size, (1, 9), dtype=torch.long, device=device)
    second = first.clone(); second[:, 6:] = torch.randint(0, vocab_size, (1, 3), device=device)
    mask = torch.ones_like(encoder, dtype=torch.bool)
    with torch.no_grad():
        first_logits = seq2seq_logits(model, encoder, first, mask)[:, :6]
        second_logits = seq2seq_logits(model, encoder, second, mask)[:, :6]
    if not torch.allclose(first_logits, second_logits, atol=2e-5, rtol=2e-5):
        raise AssertionError("Decoder future-token leakage detected")

def validate_encoder_conditioning(model, vocab_size: int, device: str) -> None:
    model.eval()
    first_encoder = torch.randint(0, vocab_size, (1, 11), dtype=torch.long, device=device)
    second_encoder = torch.randint(0, vocab_size, (1, 11), dtype=torch.long, device=device)
    decoder = torch.randint(0, vocab_size, (1, 7), dtype=torch.long, device=device)
    mask = torch.ones_like(first_encoder, dtype=torch.bool)
    with torch.no_grad():
        a = seq2seq_logits(model, first_encoder, decoder, mask)
        b = seq2seq_logits(model, second_encoder, decoder, mask)
    if torch.allclose(a, b, atol=1e-7, rtol=1e-7):
        raise AssertionError("Model output is insensitive to encoder input; cross-attention may be disconnected")

def validate_incremental_generation_cache(model, vocab_size: int, device: str) -> None:
    prepare = getattr(MODEL_MODULE, "prepare_generation", None)
    decode = getattr(MODEL_MODULE, "decode_step", None)
    if not callable(prepare) or not callable(decode):
        raise AssertionError("Model source must expose prepare_generation and decode_step adapters")
    model.eval()
    encoder = torch.randint(0, vocab_size, (1, 13), dtype=torch.long, device=device)
    decoder = torch.randint(0, vocab_size, (1, 8), dtype=torch.long, device=device)
    mask = torch.ones_like(encoder, dtype=torch.bool)
    with torch.no_grad():
        full = seq2seq_logits(model, encoder, decoder, mask)
        state = prepare(model, encoder, mask)
        steps = []
        for position in range(decoder.shape[1]):
            logits, state = decode(model, decoder[:, position:position + 1], state)
            steps.append(logits[:, None, :])
        cached = torch.cat(steps, dim=1)
    if not torch.allclose(full, cached, atol=2e-4, rtol=2e-4):
        difference = float((full - cached).abs().max().detach().cpu())
        raise AssertionError(f"Incremental cache changed logits; max_abs_diff={difference}")

def run_student_integration_smoke_test(model, tokenizer, model_config, reconstruct_model, device=None):
    if MODEL_SOURCE_SHA256 is None:
        raise RuntimeError("Student model source is not registered")
    device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
    meta = tokenizer_metadata(tokenizer, DIRS["tokenizer"])
    report = validate_seq2seq_contract(model, meta["vocab_size"], device)
    validate_decoder_causality(model, meta["vocab_size"], device)
    validate_encoder_conditioning(model, meta["vocab_size"], device)
    validate_incremental_generation_cache(model, meta["vocab_size"], device)
    report["decoder_causality"] = "PASS"
    report["encoder_conditioning"] = "PASS"
    report["incremental_generation_cache"] = "PASS"
    probe_ids = tokenizer_encode(tokenizer, "A constitutional court considered the appeal.")
    if not probe_ids or min(probe_ids) < 0 or max(probe_ids) >= meta["vocab_size"]:
        raise AssertionError("Tokenizer emitted an out-of-vocabulary ID")
    clone = reconstruct_model(model_config)
    clone.load_state_dict(model.state_dict(), strict=True)
    report["reconstruction"] = "PASS"
    report["model_source_sha256"] = MODEL_SOURCE_SHA256
    del clone
    model.to("cpu")
    if str(device).startswith("cuda"): torch.cuda.empty_cache()
    atomic_write_json(DIRS["manifests"] / "student_integration_smoke_test.json", report)
    return report

if MODEL is not None and tokenizer is not None:
    INTEGRATION_SMOKE_RESULTS = run_student_integration_smoke_test(
        MODEL, tokenizer, MODEL_CONFIG, RECONSTRUCT_MODEL
    )
    print(json.dumps(INTEGRATION_SMOKE_RESULTS, indent=2))
else:
    INTEGRATION_SMOKE_RESULTS = None
    print("Student encoder-decoder integration smoke test is waiting for model and tokenizer registration.")


## 10. Resumable encoder-decoder teacher-forcing training

The default 2 × 16 accumulation schedule keeps the effective batch at 32. Gradient checkpointing is disabled for speed at this model/sequence size; the mandatory GPU smoke test remains the memory gate and must pass before full training. Validation uses 16 fixed sequences per source and records stable document/window provenance.


In [ ]:
# Frozen before the first checkpoint. Effective batch: 2 x 16 = 32 sequences;
# planned training budget: 12,000 x 32 x (512 + 256) = 294,912,000 non-padding
# sequence positions before accounting for short-document padding.
TRAINING_CONFIG = {
    "experiment_id": CFG.experiment_id,
    "objective": "teacher-forced target-token cross-entropy conditioned on a bidirectional source encoding",
    "optimizer": "AdamW",
    "learning_rate": 3e-4,
    "betas": [0.9, 0.95],
    "epsilon": 1e-8,
    "weight_decay": 0.1,
    "schedule": "linear_warmup_cosine_decay",
    "warmup_steps": 600,
    "min_lr_ratio": 0.1,
    "batch_size": 2,
    "gradient_accumulation_steps": 16,
    "source_length": 512,
    "target_length": 256,
    "max_steps": 12_000,
    "samples_per_epoch": 32_000,
    "gradient_clip_norm": 1.0,
    "gradient_checkpointing": False,
    "validation_interval": 500,
    "validation_sequences_per_source": 16,
    "checkpoint_interval": 250,
    "recovery_interval": 1000,
    "num_workers": 0,
    "source_mixture": dict(CFG.source_mixing_ratios),
    "source_mixture_unit": "document draws, resampled deterministically per epoch",
    "seed": CFG.seed,
    "precision_policy": "bf16 if supported; otherwise fp16+GradScaler on CUDA; fp32 on CPU",
}

def validate_student_training_config(config: Mapping[str, Any]) -> dict[str, Any]:
    required = (
        "optimizer", "learning_rate", "betas", "epsilon", "weight_decay", "schedule",
        "warmup_steps", "min_lr_ratio", "batch_size", "gradient_accumulation_steps",
        "source_length", "target_length", "max_steps", "samples_per_epoch",
        "gradient_clip_norm", "gradient_checkpointing", "validation_interval",
        "validation_sequences_per_source",
    )
    missing = [key for key in required if config.get(key) is None]
    if missing:
        raise RuntimeError(f"Student training configuration is incomplete: {missing}")
    if config["optimizer"] != "AdamW":
        raise ValueError("This infrastructure currently validates AdamW only; document and implement another optimizer explicitly")
    if config["schedule"] != "linear_warmup_cosine_decay":
        raise ValueError("This infrastructure currently validates linear_warmup_cosine_decay only")
    positive = (
        "learning_rate", "epsilon", "batch_size", "gradient_accumulation_steps", "source_length",
        "target_length", "max_steps", "samples_per_epoch", "gradient_clip_norm",
        "validation_interval", "validation_sequences_per_source", "checkpoint_interval", "recovery_interval",
    )
    if any(float(config[key]) <= 0 for key in positive):
        raise ValueError("Configured positive quantities must be > 0")
    if int(config["num_workers"]) != 0:
        raise ValueError("num_workers must remain 0 for exact committed sampler restoration")
    if len(config["betas"]) != 2 or not all(0.0 <= float(value) < 1.0 for value in config["betas"]):
        raise ValueError("AdamW betas must contain two values in [0, 1)")
    if float(config["weight_decay"]) < 0.0:
        raise ValueError("weight_decay must be non-negative")
    if not 0.0 <= float(config["min_lr_ratio"]) <= 1.0:
        raise ValueError("min_lr_ratio must be in [0, 1]")
    if int(config["warmup_steps"]) >= int(config["max_steps"]):
        raise ValueError("warmup_steps must be smaller than max_steps")
    if MODEL_CONFIG is not None:
        if int(config["source_length"]) > int(MODEL_CONFIG["max_encoder_length"]):
            raise ValueError("source_length exceeds the model encoder context")
        if int(config["target_length"]) > int(MODEL_CONFIG["max_decoder_length"]):
            raise ValueError("target_length exceeds the model decoder context")
        if int(CFG.decoding["max_new_tokens"]) + 1 > int(MODEL_CONFIG["max_decoder_length"]):
            raise ValueError("Declared generation length exceeds the model decoder context")
    return dict(config)

def save_or_verify_json(path: Path, value: Any) -> None:
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8")) != value:
            raise RuntimeError(f"Existing immutable manifest differs: {path}")
    else:
        atomic_write_json(path, value)

def append_event(path: Path, event: Mapping[str, Any]) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("ab") as handle:
        handle.write((json.dumps(dict(event), ensure_ascii=False) + "\n").encode("utf-8"))
        handle.flush(); os.fsync(handle.fileno())

class MixedDocumentSeq2SeqDataset(Dataset):
    def __init__(
        self, documents, tokenizer, ratios, samples, source_length, target_length, seed,
        cache_documents=128, include_provenance=False,
    ):
        self.documents = list(documents); self.tokenizer = tokenizer; self.ratios = dict(ratios)
        self.samples = int(samples); self.source_length = int(source_length); self.target_length = int(target_length)
        self.seed = int(seed); self.epoch = 0; self.cache_documents = int(cache_documents); self.cache = {}
        self.include_provenance = bool(include_provenance)
        self.pad_id = getattr(tokenizer, "pad_token_id", None)
        self.bos_id = getattr(tokenizer, "bos_token_id", None)
        self.eos_id = getattr(tokenizer, "eos_token_id", None)
        if None in (self.pad_id, self.bos_id, self.eos_id):
            raise ValueError("Seq2seq training requires distinct pad, bos, and eos token IDs")
        if len({int(self.pad_id), int(self.bos_id), int(self.eos_id)}) != 3:
            raise ValueError("pad, bos, and eos token IDs must be distinct")
        self._resample()
    def _resample(self):
        self.document_indices = source_sampling_indices(self.documents, self.ratios, self.samples, self.seed + self.epoch)
    def __len__(self): return self.samples
    def set_epoch(self, epoch: int): self.epoch = int(epoch); self._resample()
    def _tokens(self, document_index: int) -> list[int]:
        if document_index not in self.cache:
            if len(self.cache) >= self.cache_documents: self.cache.pop(next(iter(self.cache)))
            self.cache[document_index] = tokenizer_encode(self.tokenizer, document_text(self.documents[document_index]))
        return self.cache[document_index]
    def __getitem__(self, index: int):
        document_index = self.document_indices[int(index)]
        document = self.documents[document_index]
        ids = self._tokens(document_index)
        needed = self.source_length + self.target_length
        start = 0
        if len(ids) >= needed:
            rng = random.Random(self.seed + self.epoch * len(self) + int(index))
            start = rng.randrange(0, len(ids) - needed + 1)
            source = list(ids[start:start + self.source_length])
            target = list(ids[start + self.source_length:start + needed])
        elif len(ids) >= 2:
            split = min(self.source_length, max(1, len(ids) - 1))
            source = list(ids[:split])
            target = list(ids[split:split + self.target_length])
        else:
            source = list(ids) or [int(self.eos_id)]
            target = []
        target = [*target, int(self.eos_id)][:self.target_length]
        source_valid = len(source); target_valid = len(target)
        source = (source + [int(self.pad_id)] * self.source_length)[:self.source_length]
        labels = (target + [-100] * self.target_length)[:self.target_length]
        decoder_input = [int(self.bos_id)] + [token for token in target[:-1]]
        decoder_input = (decoder_input + [int(self.pad_id)] * self.target_length)[:self.target_length]
        result = {
            "encoder_input_ids": torch.tensor(source, dtype=torch.long),
            "encoder_attention_mask": torch.tensor([True] * source_valid + [False] * (self.source_length - source_valid)),
            "decoder_input_ids": torch.tensor(decoder_input, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "document_index": document_index,
            "processed_tokens": source_valid + target_valid,
        }
        if self.include_provenance:
            result.update({
                "source_name": document.source,
                "document_uid": document.uid,
                "document_content_sha256": document.content_hash,
                "selected_token_offset": start,
                "source_token_count": source_valid,
                "target_token_count": target_valid,
            })
        return result

def build_fixed_seq2seq_validation_rows(documents, tokenizer, config):
    rows = []
    per_source = int(config["validation_sequences_per_source"])
    for source in sorted({document.source for document in documents}):
        source_documents = [document for document in documents if document.source == source]
        dataset = MixedDocumentSeq2SeqDataset(
            source_documents, tokenizer, {source: 1.0}, per_source,
            int(config["source_length"]), int(config["target_length"]),
            int(config["seed"]) + 10_000, include_provenance=True,
        )
        rows.extend(dataset[index] for index in range(len(dataset)))
    manifest = [
        {
            "source": row["source_name"],
            "document_uid": row["document_uid"],
            "document_content_sha256": row["document_content_sha256"],
            "selected_token_offset": int(row["selected_token_offset"]),
            "source_token_count": int(row["source_token_count"]),
            "target_token_count": int(row["target_token_count"]),
            "processed_tokens": int(row["processed_tokens"]),
        }
        for row in rows
    ]
    return rows, manifest

def choose_precision(device: str):
    if not str(device).startswith("cuda"):
        return None, "fp32", False
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16, "bf16", False
    return torch.float16, "fp16", True

def make_scaler(enabled: bool):
    try: return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError): return torch.cuda.amp.GradScaler(enabled=enabled)

def move_to_device(tensor: torch.Tensor, device: str) -> torch.Tensor:
    return tensor.to(device, non_blocking=str(device).startswith("cuda"))

def scaler_update_succeeded(scale_before: float, scale_after: float) -> bool:
    return float(scale_after) >= float(scale_before)

def learning_rate_multiplier(step: int, max_steps: int, warmup_steps: int, min_ratio: float) -> float:
    if step < warmup_steps:
        return max(step, 1) / max(warmup_steps, 1)
    progress = min(1.0, (step - warmup_steps) / max(1, max_steps - warmup_steps))
    return float(min_ratio) + (1.0 - float(min_ratio)) * 0.5 * (1.0 + math.cos(math.pi * progress))

def atomic_copy(source: Path, destination: Path) -> None:
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=destination.parent, suffix=destination.suffix, delete=False) as handle:
        temporary = Path(handle.name)
    try:
        shutil.copyfile(source, temporary)
        if sha256_file(source) != sha256_file(temporary): raise IOError("Atomic copy verification failed")
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)

def save_inference_checkpoint(path, model, model_config, tokenizer_meta, dataset_fp, step, validation_loss):
    state = {
        "format": "phase1_submission02_seq2seq_inference_v1", "created_utc": utc_now(),
        "model_state": model.state_dict(), "model_config": dict(model_config),
        "model_source_sha256": MODEL_SOURCE_SHA256,
        "tokenizer_fingerprint": tokenizer_meta["fingerprint"], "dataset_fingerprint": dataset_fp,
        "step": int(step), "validation_loss": None if validation_loss is None else float(validation_loss),
    }
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix=".pt", delete=False) as handle:
        temporary = Path(handle.name)
    try:
        torch.save(state, temporary)
        check = torch.load(temporary, map_location="cpu", weights_only=False)
        if check.get("format") != state["format"] or check.get("model_source_sha256") != MODEL_SOURCE_SHA256:
            raise RuntimeError("Inference checkpoint readback failed")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)
    return {"path": str(path), "sha256": sha256_file(path), "step": step}

def export_training_recovery_snapshot(root: Path) -> Path:
    root = Path(root).resolve()
    output = DIRS["recovery"] / "training_recovery.latest.zip"
    include = []
    for relative in ("model", "tokenizer", "manifests", "histories"):
        include.extend(path for path in (root / relative).rglob("*") if path.is_file())
    local_examples = root / "data" / "local_continuation_examples.json"
    if local_examples.exists():
        include.append(local_examples)
    for name in ("checkpoint.latest.pt", "previous_known_good.pt", "selected_model.pt", "model_final.pt"):
        checkpoint_path = DIRS["checkpoints"] / name
        if checkpoint_path.exists():
            include.append(checkpoint_path)
    include = [
        path for path in sorted(set(include))
        if path.resolve() != (root / RECOVERY_MANIFEST_MEMBER).resolve()
    ]
    report = write_verified_recovery_archive(root, include, output)
    atomic_write_json(DIRS["recovery"] / "training_recovery.latest.json", {
        "created_utc": utc_now(),
        "path": str(output),
        "sha256": report["sha256"],
        "bytes": output.stat().st_size,
        "integrity_manifest": report["integrity_manifest"],
        "verified_files": report["verified_files"],
        "restore_helper_compatible": True,
    })
    return output

@torch.no_grad()
def evaluate_seq2seq_loss(model, loader, device, amp_dtype):
    was_training = model.training; model.eval(); loss_sum = 0.0; token_count = 0
    for batch in loader:
        encoder = move_to_device(batch["encoder_input_ids"], device)
        encoder_mask = move_to_device(batch["encoder_attention_mask"], device)
        decoder = move_to_device(batch["decoder_input_ids"], device)
        labels = move_to_device(batch["labels"], device)
        with torch.autocast(device_type=torch.device(device).type, dtype=amp_dtype, enabled=amp_dtype is not None):
            logits = seq2seq_logits(model, encoder, decoder, encoder_mask)
            loss = torch.nn.functional.cross_entropy(
                logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), ignore_index=-100, reduction="sum"
            )
        loss_sum += float(loss.cpu()); token_count += int((labels != -100).sum())
    model.train(was_training)
    if not token_count: raise ValueError("Validation loader contained no targets")
    return loss_sum / token_count

def train_submission02(model, tokenizer, train_docs, val_docs, model_config, tokenizer_meta, *, device="cuda:0", resume=False):
    cfg = validate_student_training_config(TRAINING_CONFIG)
    if not CFG.student_configuration_confirmed or not CFG.student_owned_architecture_attestation:
        raise RuntimeError("Confirm and attest the student-owned architecture/training configuration first")
    latest = DIRS["checkpoints"] / "checkpoint.latest.pt"
    if resume and not latest.exists():
        raise FileNotFoundError("resume_train requires checkpoint.latest.pt; refusing to start from step zero")
    if not resume and latest.exists():
        raise RuntimeError("fresh_train found an existing checkpoint; use resume_train or a new workspace")
    save_or_verify_json(DIRS["manifests"] / "training_config.json", cfg)
    split_meta = json.loads((DIRS["manifests"] / "document_split.json").read_text(encoding="utf-8"))
    dataset_fp = split_meta["dataset_fingerprint"]
    ratios = validate_source_mixture(cfg["source_mixture"], (doc.source for doc in train_docs))
    dataset = MixedDocumentSeq2SeqDataset(
        train_docs, tokenizer, ratios, cfg["samples_per_epoch"], cfg["source_length"],
        cfg["target_length"], cfg["seed"],
    )
    sampler = StatefulRandomSampler(dataset, seed=cfg["seed"])
    loader = DataLoader(dataset, batch_size=cfg["batch_size"], sampler=sampler, num_workers=0,
                        pin_memory=str(device).startswith("cuda"), drop_last=True)
    validation_rows, validation_manifest = build_fixed_seq2seq_validation_rows(val_docs, tokenizer, cfg)
    save_or_verify_json(DIRS["manifests"] / "validation_sequence_manifest.json", {
        "protocol": "fixed source-conditioned target sequences with stable document and token-window provenance", "rows": validation_manifest,
    })
    validation_loader = DataLoader(validation_rows, batch_size=cfg["batch_size"], shuffle=False, num_workers=0, pin_memory=str(device).startswith("cuda"))
    model = model.to(device)
    if hasattr(model, "set_gradient_checkpointing"):
        model.set_gradient_checkpointing(bool(cfg["gradient_checkpointing"]))
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["learning_rate"], betas=tuple(cfg["betas"]),
        eps=cfg["epsilon"], weight_decay=cfg["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: learning_rate_multiplier(step, cfg["max_steps"], cfg["warmup_steps"], cfg["min_lr_ratio"])
    )
    amp_dtype, actual_precision, use_scaler = choose_precision(device)
    scaler = make_scaler(use_scaler); checkpoint_scaler = scaler if use_scaler else None
    history_path = DIRS["histories"] / "training.jsonl"; history = JsonlHistory(history_path)
    progress = ProgressState(cfg["experiment_id"], device, cfg["max_steps"])
    validation_history = []
    if resume:
        preview = torch.load(latest, map_location="cpu", weights_only=False)
        if preview.get("training_config") != cfg: raise RuntimeError("Checkpoint training config differs")
        if preview.get("model_source_sha256") != MODEL_SOURCE_SHA256: raise RuntimeError("Checkpoint model source differs")
        reconcile_history_to_checkpoint(latest, history_path, apply=True)
        state = restore_checkpoint(
            latest, model=model, optimizer=optimizer, scheduler=scheduler, scaler=checkpoint_scaler,
            sampler=sampler, expected_tokenizer_fingerprint=tokenizer_meta["fingerprint"],
            expected_dataset_fingerprint=dataset_fp, expected_experiment_id=cfg["experiment_id"],
            expected_model_config=model_config, history_path=history_path, device=device,
            expected_model_source_sha256=MODEL_SOURCE_SHA256,
        )
        progress = ProgressState(**state["progress"])
        validation_history = list(state.get("validation_history", []))
        dataset.set_epoch(sampler.epoch)
        print("Restored optimizer step:", progress.step)
    elif history.records():
        raise RuntimeError("Training history exists without a matching fresh/resume decision")
    reporter = ProgressReporter(progress); iterator = iter(loader)
    best_loss = min((float(row["loss"]) for row in validation_history), default=math.inf)
    try:
        while reporter.state.step < cfg["max_steps"]:
            model.train(); optimizer.zero_grad(set_to_none=True); microbatches = []; valid_targets = 0; processed_tokens = 0
            for _ in range(cfg["gradient_accumulation_steps"]):
                try: batch = next(iterator)
                except StopIteration:
                    sampler.next_epoch(); dataset.set_epoch(sampler.epoch); iterator = iter(loader); batch = next(iterator)
                valid_targets += int((batch["labels"] != -100).sum())
                processed_tokens += int(batch["processed_tokens"].sum())
                microbatches.append(batch)
            if valid_targets <= 0: raise ValueError("Accumulation window has no target tokens")
            numerator = 0.0
            for batch in microbatches:
                encoder = move_to_device(batch["encoder_input_ids"], device)
                encoder_mask = move_to_device(batch["encoder_attention_mask"], device)
                decoder = move_to_device(batch["decoder_input_ids"], device)
                labels = move_to_device(batch["labels"], device)
                with torch.autocast(device_type=torch.device(device).type, dtype=amp_dtype, enabled=amp_dtype is not None):
                    logits = seq2seq_logits(model, encoder, decoder, encoder_mask)
                    loss_sum = torch.nn.functional.cross_entropy(
                        logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), ignore_index=-100, reduction="sum"
                    )
                    backward_loss = loss_sum / valid_targets
                if not torch.isfinite(loss_sum): raise FloatingPointError("Non-finite training loss")
                (scaler.scale(backward_loss) if use_scaler else backward_loss).backward()
                numerator += float(loss_sum.detach().cpu())
            if use_scaler: scaler.unscale_(optimizer)
            gradient_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip_norm"]).cpu())
            learning_rate = float(optimizer.param_groups[0]["lr"])
            if use_scaler:
                before = float(scaler.get_scale()); scaler.step(optimizer); scaler.update()
                if not scaler_update_succeeded(before, scaler.get_scale()):
                    append_event(DIRS["histories"] / "overflow_events.jsonl", {
                        "attempted_step": reporter.state.step + 1,
                        "epoch": sampler.epoch,
                        "scale_before": before,
                        "scale_after": float(scaler.get_scale()),
                        "created_utc": utc_now(),
                    })
                    continue
            else:
                optimizer.step()
            scheduler.step()
            step = reporter.state.step + 1
            total_tokens = reporter.state.tokens_processed + processed_tokens
            mean_loss = numerator / valid_targets
            validation_loss = None; improved = False
            if step % cfg["validation_interval"] == 0 or step == cfg["max_steps"]:
                validation_loss = evaluate_seq2seq_loss(model, validation_loader, device, amp_dtype)
                validation_history.append({"step": step, "loss": validation_loss})
                improved = validation_loss < best_loss; best_loss = min(best_loss, validation_loss)
            reporter.update(step=step, epoch=sampler.epoch, loss=mean_loss, learning_rate=learning_rate,
                            tokens_processed=total_tokens, gradient_norm=gradient_norm)
            history.append({
                "step": step, "epoch": sampler.epoch, "loss": mean_loss,
                "validation_loss": validation_loss, "learning_rate": learning_rate,
                "gradient_norm": gradient_norm, "tokens_processed": total_tokens,
                "tokens_per_second": total_tokens / max(reporter.state.cumulative_seconds, 1e-9),
                "gpu_memory_gib": torch.cuda.max_memory_allocated(torch.device(device)) / 2**30 if str(device).startswith("cuda") else None,
                "precision": actual_precision, "checkpoint_step": reporter.state.last_checkpoint_step,
                "created_utc": utc_now(),
            })
            if step % cfg["checkpoint_interval"] == 0 or improved or step == cfg["max_steps"]:
                reporter.state.last_checkpoint_step = step
                save_checkpoint_atomic(
                    latest, model=model, optimizer=optimizer, scheduler=scheduler, scaler=checkpoint_scaler,
                    sampler=sampler, progress=asdict(reporter.state), model_config=model_config,
                    tokenizer_fingerprint=tokenizer_meta["fingerprint"], dataset_fingerprint_value=dataset_fp,
                    training_config=cfg, history_path=history_path, validation_history=validation_history,
                    experiment_id=cfg["experiment_id"], device=device, model_source_sha256=MODEL_SOURCE_SHA256,
                )
                if improved:
                    save_inference_checkpoint(DIRS["checkpoints"] / "selected_model.pt", model, model_config,
                                              tokenizer_meta, dataset_fp, step, validation_loss)
                if step % cfg["recovery_interval"] == 0:
                    export_training_recovery_snapshot(ROOT)
        save_inference_checkpoint(DIRS["checkpoints"] / "model_final.pt", model, model_config,
                                  tokenizer_meta, dataset_fp, reporter.state.step,
                                  validation_history[-1]["loss"] if validation_history else None)
        if not (DIRS["checkpoints"] / "selected_model.pt").exists():
            atomic_copy(DIRS["checkpoints"] / "model_final.pt", DIRS["checkpoints"] / "selected_model.pt")
        export_training_recovery_snapshot(ROOT)
    finally:
        reporter.close()
    result = {
        "completed_step": reporter.state.step, "tokens_processed": reporter.state.tokens_processed,
        "best_validation_loss": best_loss, "actual_precision": actual_precision,
        "latest_checkpoint": str(latest), "selected_checkpoint": str(DIRS["checkpoints"] / "selected_model.pt"),
    }
    atomic_write_json(DIRS["manifests"] / "training_result.json", result)
    return result

def run_gpu_training_smoke_test(device="cuda:0"):
    if not torch.cuda.is_available(): raise RuntimeError("Representative smoke test requires CUDA")
    cfg = validate_student_training_config(TRAINING_CONFIG)
    if MODEL is None or tokenizer is None or not TRAIN_DOCS or not VAL_DOCS:
        raise RuntimeError("Model, tokenizer, and bounded corpus cache are required")
    validation_rows, validation_manifest = build_fixed_seq2seq_validation_rows(VAL_DOCS, tokenizer, cfg)
    save_or_verify_json(DIRS["manifests"] / "validation_sequence_manifest.json", {
        "protocol": "fixed source-conditioned target sequences with stable document and token-window provenance", "rows": validation_manifest,
    })
    tiny = MixedDocumentSeq2SeqDataset(
        TRAIN_DOCS, tokenizer, cfg["source_mixture"], max(2, cfg["batch_size"] * 2),
        cfg["source_length"], cfg["target_length"], cfg["seed"],
    )
    smoke_sampler = StatefulRandomSampler(tiny, seed=cfg["seed"])
    batch = next(iter(DataLoader(tiny, batch_size=cfg["batch_size"], sampler=smoke_sampler, num_workers=0)))
    smoke_model = RECONSTRUCT_MODEL(MODEL_CONFIG).to(device)
    optimizer = torch.optim.AdamW(smoke_model.parameters(), lr=cfg["learning_rate"], betas=tuple(cfg["betas"]),
                                  eps=cfg["epsilon"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: learning_rate_multiplier(step, 10, 1, cfg["min_lr_ratio"])
    )
    amp_dtype, precision, use_scaler = choose_precision(device); scaler = make_scaler(use_scaler)
    torch.cuda.reset_peak_memory_stats(torch.device(device)); optimizer.zero_grad(set_to_none=True)
    encoder = move_to_device(batch["encoder_input_ids"], device); mask = move_to_device(batch["encoder_attention_mask"], device)
    decoder = move_to_device(batch["decoder_input_ids"], device); labels = move_to_device(batch["labels"], device)
    with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=amp_dtype is not None):
        logits = seq2seq_logits(smoke_model, encoder, decoder, mask)
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), ignore_index=-100)
    (scaler.scale(loss) if use_scaler else loss).backward()
    if use_scaler: scaler.unscale_(optimizer)
    gradient_norm = float(torch.nn.utils.clip_grad_norm_(smoke_model.parameters(), cfg["gradient_clip_norm"]).cpu())
    if use_scaler:
        before = float(scaler.get_scale()); scaler.step(optimizer); scaler.update()
        if not scaler_update_succeeded(before, scaler.get_scale()): raise RuntimeError("FP16 smoke update overflowed")
    else: optimizer.step()
    scheduler.step()
    loss_value = float(loss.detach().cpu())
    split_meta = json.loads((DIRS["manifests"] / "document_split.json").read_text(encoding="utf-8"))
    with tempfile.TemporaryDirectory() as temporary_name:
        temporary = Path(temporary_name)
        history = JsonlHistory(temporary / "history.jsonl")
        history.append({"step": 1, "loss": loss_value})
        checkpoint = temporary / "smoke.pt"
        progress = ProgressState(
            "submission02_gpu_smoke", device, 1, step=1,
            tokens_processed=int(batch["processed_tokens"].sum()), last_checkpoint_step=1,
        )
        save_checkpoint_atomic(
            checkpoint, model=smoke_model, optimizer=optimizer, scheduler=scheduler,
            scaler=scaler if use_scaler else None, sampler=smoke_sampler,
            progress=asdict(progress), model_config=MODEL_CONFIG,
            tokenizer_fingerprint=TOKENIZER_META["fingerprint"],
            dataset_fingerprint_value=split_meta["dataset_fingerprint"],
            training_config={"smoke": True}, history_path=history.path,
            validation_history=[], experiment_id="submission02_gpu_smoke", device=device,
            model_source_sha256=MODEL_SOURCE_SHA256,
        )
        clone = RECONSTRUCT_MODEL(MODEL_CONFIG)
        clone_optimizer = torch.optim.AdamW(
            clone.parameters(), lr=cfg["learning_rate"], betas=tuple(cfg["betas"]),
            eps=cfg["epsilon"], weight_decay=cfg["weight_decay"],
        )
        clone_scheduler = torch.optim.lr_scheduler.LambdaLR(
            clone_optimizer, lambda step: learning_rate_multiplier(step, 10, 1, cfg["min_lr_ratio"])
        )
        clone_scaler = make_scaler(use_scaler)
        clone_sampler = StatefulRandomSampler(tiny, seed=cfg["seed"])
        restored = restore_checkpoint(
            checkpoint, model=clone, optimizer=clone_optimizer, scheduler=clone_scheduler,
            scaler=clone_scaler if use_scaler else None, sampler=clone_sampler,
            expected_tokenizer_fingerprint=TOKENIZER_META["fingerprint"],
            expected_dataset_fingerprint=split_meta["dataset_fingerprint"],
            expected_experiment_id="submission02_gpu_smoke", expected_model_config=MODEL_CONFIG,
            history_path=history.path, device=None,
            expected_model_source_sha256=MODEL_SOURCE_SHA256,
        )
        if restored["progress"]["step"] != 1 or not clone_optimizer.state:
            raise AssertionError("Checkpoint did not restore model, optimizer, or progress")
    report = {
        "status": "PASS", "loss": loss_value, "gradient_norm": gradient_norm,
        "precision": precision, "peak_gpu_memory_gib": torch.cuda.max_memory_allocated(torch.device(device)) / 2**30,
        "source_length": cfg["source_length"], "target_length": cfg["target_length"],
        "batch_size": cfg["batch_size"], "checkpoint_restore": "PASS", "created_utc": utc_now(),
    }
    atomic_write_json(DIRS["manifests"] / "gpu_training_smoke_test.json", report)
    del smoke_model, optimizer, logits, loss, clone, clone_optimizer; torch.cuda.empty_cache()
    return report

RUN_GPU_TRAINING_SMOKE_TEST = False
GPU_TRAINING_SMOKE_RESULT = run_gpu_training_smoke_test("cuda:0") if RUN_GPU_TRAINING_SMOKE_TEST else None

def preflight_full_training():
    problems = []
    if CFG.execution_mode not in {"fresh_train", "resume_train"}: problems.append("execution_mode is not a training mode")
    if not CFG.full_training_authorized: problems.append("CFG.full_training_authorized is False")
    if not CFG.student_owned_architecture_attestation: problems.append("student architecture attestation is False")
    if not CFG.student_configuration_confirmed: problems.append("student configuration confirmation is False")
    if not CFG.trained_from_scratch_attestation: problems.append("scratch-training attestation is False")
    try: validate_student_training_config(TRAINING_CONFIG)
    except Exception as exc: problems.append(str(exc))
    if MODEL is None or tokenizer is None: problems.append("model/tokenizer unavailable")
    if not TRAIN_DOCS or not VAL_DOCS: problems.append("bounded training/validation cache unavailable")
    for name in ("document_split.json", "model_validation.json", "gpu_training_smoke_test.json", "validation_sequence_manifest.json"):
        if not (DIRS["manifests"] / name).exists(): problems.append(f"missing manifest {name}")
    if problems: raise RuntimeError("Full-training preflight failed:\n- " + "\n- ".join(problems))

RUN_FULL_TRAINING = True
if RUN_FULL_TRAINING:
    preflight_full_training()
    TRAINING_RESULT = train_submission02(
        MODEL, tokenizer, TRAIN_DOCS, VAL_DOCS, MODEL_CONFIG, TOKENIZER_META,
        device="cuda:0", resume=CFG.execution_mode == "resume_train",
    )
else:
    TRAINING_RESULT = None
    print("Training is opt-in and blocked until the student-owned core and representative GPU smoke test pass.")


In [ ]:
CFG.execution_mode = "resume_train"
CFG.student_owned_architecture_attestation = True
CFG.student_configuration_confirmed = True
CFG.trained_from_scratch_attestation = True
CFG.full_training_authorized = True
register_run_config(CFG)

In [ ]:
import json

print(json.dumps(
    GPU_TRAINING_SMOKE_RESULT,
    indent=2
))

## 11. Local evaluation plumbing

Optional local evaluation now generates the fixed held-out continuations with the declared policies and computes dependency-free ROUGE-L F1, smoothed corpus BLEU-4, exact match, timing, repetition, prompt-truncation, and token-budget diagnostics. If the official course evaluator is attached, its BERTScore/ROUGE/BLEU results are recorded alongside these local metrics.


In [ ]:
EXPECTED_METRIC_FIELDS=('bertscore_f1','rouge_l_f1','bleu_4')
METRIC_ALIASES={
    'bertscore_f1':'bertscore_f1','bert_score_f1':'bertscore_f1','bertscore':'bertscore_f1',
    'rouge_l_f1':'rouge_l_f1','rougel_f1':'rouge_l_f1','rouge-l_f1':'rouge_l_f1',
    'bleu_4':'bleu_4','bleu4':'bleu_4','bleu-4':'bleu_4',
}
def perplexity_from_loss(loss): return float(math.exp(loss)) if loss<709 else math.inf
def normalize_official_metrics(raw:Mapping[str,Any])->dict[str,float]:
    normalized={}
    for key,value in raw.items():
        canonical=METRIC_ALIASES.get(str(key).strip().lower().replace(' ','_'))
        if canonical: normalized[canonical]=float(value)
    missing=[key for key in EXPECTED_METRIC_FIELDS if key not in normalized]
    if missing: raise ValueError(f'Official evaluator result is missing required metrics {missing}; raw keys={list(raw)}')
    return {key:normalized[key] for key in EXPECTED_METRIC_FIELDS}
def run_official_evaluator(evaluator,predictions,references,checkpoint_id):
    if len(predictions)!=len(references) or not predictions: raise ValueError('Predictions/references must be nonempty and aligned')
    metrics=normalize_official_metrics(evaluator(predictions,references));record={'checkpoint_id':checkpoint_id,'created_utc':utc_now(),'metrics':metrics,'examples':len(predictions)}
    atomic_write_json(DIRS['evaluations']/f'{checkpoint_id}.json',record);csv_path=DIRS['evaluations']/'metrics.csv';write_header=not csv_path.exists()
    expected_header=['checkpoint_id','created_utc','examples',*EXPECTED_METRIC_FIELDS]
    if csv_path.exists():
        with csv_path.open('r',newline='',encoding='utf-8') as existing_handle:
            existing_header=next(csv.reader(existing_handle),None)
        if existing_header!=expected_header: raise RuntimeError(f'Existing metrics.csv schema differs: {existing_header}')
    with csv_path.open('a',newline='',encoding='utf-8') as handle:
        writer=csv.DictWriter(handle,fieldnames=expected_header)
        if write_header: writer.writeheader()
        writer.writerow({'checkpoint_id':checkpoint_id,'created_utc':record['created_utc'],'examples':len(predictions),**metrics})
    return record

def load_official_evaluator(path:Path,function_name:str,argument_order:str):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(path)
    if not function_name: raise ValueError('Set CS781_EVALUATOR_FUNCTION to the inspected official callable name')
    if argument_order not in ('predictions_references','references_predictions'): raise ValueError('Set CS781_EVALUATOR_ARGUMENT_ORDER explicitly')
    module=import_source_module(f'cs781_official_evaluator_{sha256_file(path)[:12]}',path);function=getattr(module,function_name,None)
    if not callable(function): raise AttributeError(f'{path} has no callable {function_name!r}')
    def adapted(predictions,references):
        result=function(predictions,references) if argument_order=='predictions_references' else function(references,predictions)
        if not isinstance(result,Mapping): raise TypeError(f'Official evaluator must return a mapping, got {type(result).__name__}')
        return result
    return adapted

def discover_evaluator_candidates(root:Path)->list[Path]:
    if not Path(root).exists(): return []
    return sorted(path for path in Path(root).rglob('*.py') if any(term in path.name.lower() for term in ('eval','metric','score')) and ROOT.resolve() not in path.resolve().parents)

OFFICIAL_EVALUATOR_PATH=os.environ.get('CS781_EVALUATOR_PATH');OFFICIAL_EVALUATOR_FUNCTION=os.environ.get('CS781_EVALUATOR_FUNCTION')
OFFICIAL_EVALUATOR_ARGUMENT_ORDER=os.environ.get('CS781_EVALUATOR_ARGUMENT_ORDER')
OFFICIAL_EVALUATOR=(load_official_evaluator(Path(OFFICIAL_EVALUATOR_PATH),OFFICIAL_EVALUATOR_FUNCTION,OFFICIAL_EVALUATOR_ARGUMENT_ORDER) if OFFICIAL_EVALUATOR_PATH else None)
if OFFICIAL_EVALUATOR is None:
    EVALUATOR_CANDIDATES=discover_evaluator_candidates(Path(CFG.input_root));print('Inspect official evaluator candidates:',[str(path) for path in EVALUATOR_CANDIDATES])
    print('Set CS781_EVALUATOR_PATH, CS781_EVALUATOR_FUNCTION and CS781_EVALUATOR_ARGUMENT_ORDER explicitly.')


def _metric_tokens(text: str) -> list[str]:
    return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)

def _lcs_length(first: Sequence[str], second: Sequence[str]) -> int:
    previous = [0] * (len(second) + 1)
    for left in first:
        current = [0]
        for index, right in enumerate(second, 1):
            current.append(previous[index - 1] + 1 if left == right else max(previous[index], current[-1]))
        previous = current
    return previous[-1]

def rouge_l_f1(prediction: str, reference: str) -> float:
    predicted = _metric_tokens(prediction); expected = _metric_tokens(reference)
    if not predicted or not expected:
        return 0.0
    overlap = _lcs_length(predicted, expected)
    precision = overlap / len(predicted); recall = overlap / len(expected)
    return 0.0 if precision + recall == 0.0 else 2.0 * precision * recall / (precision + recall)

def corpus_bleu_4(predictions: Sequence[str], references: Sequence[str]) -> float:
    if len(predictions) != len(references) or not predictions:
        raise ValueError("BLEU requires nonempty aligned predictions and references")
    clipped = [0] * 4; totals = [0] * 4; predicted_length = 0; reference_length = 0
    for prediction, reference in zip(predictions, references):
        predicted = _metric_tokens(prediction); expected = _metric_tokens(reference)
        predicted_length += len(predicted); reference_length += len(expected)
        for order in range(1, 5):
            predicted_ngrams = Counter(tuple(predicted[i:i + order]) for i in range(max(0, len(predicted) - order + 1)))
            expected_ngrams = Counter(tuple(expected[i:i + order]) for i in range(max(0, len(expected) - order + 1)))
            clipped[order - 1] += sum(min(count, expected_ngrams[gram]) for gram, count in predicted_ngrams.items())
            totals[order - 1] += sum(predicted_ngrams.values())
    # Add-one smoothing prevents a small local sample from collapsing to zero.
    precisions = [(clipped[i] + 1.0) / (totals[i] + 1.0) for i in range(4)]
    brevity_penalty = (
        1.0 if predicted_length > reference_length
        else math.exp(1.0 - reference_length / max(predicted_length, 1))
    )
    return float(brevity_penalty * math.exp(sum(math.log(value) for value in precisions) / 4.0))

def compute_local_text_metrics(predictions: Sequence[str], references: Sequence[str]) -> dict[str, float]:
    if len(predictions) != len(references) or not predictions:
        raise ValueError("Local metrics require nonempty aligned predictions and references")
    rouge_values = [rouge_l_f1(prediction, reference) for prediction, reference in zip(predictions, references)]
    exact = [float(prediction.strip() == reference.strip()) for prediction, reference in zip(predictions, references)]
    return {
        "rouge_l_f1": float(sum(rouge_values) / len(rouge_values)),
        "bleu_4": corpus_bleu_4(predictions, references),
        "exact_match": float(sum(exact) / len(exact)),
    }


In [ ]:
def plot_saved_history(history_path: Path, output_path: Path) -> None:
    import matplotlib.pyplot as plt
    records = JsonlHistory(history_path).records()
    if not records:
        raise ValueError("No history records to plot")
    figure, axes = plt.subplots(2, 2, figsize=(12, 8))
    fields = [("loss", "Training loss"), ("learning_rate", "Learning rate"), ("tokens_per_second", "Tokens/s"), ("gpu_memory_gib", "GPU memory GiB")]
    for axis, (field_name, title) in zip(axes.flat, fields):
        x, y = zip(*[(row["step"], row[field_name]) for row in records if row.get(field_name) is not None]) if any(row.get(field_name) is not None for row in records) else ([], [])
        axis.plot(x, y)
        axis.set_title(title)
        axis.set_xlabel("Optimizer step")
        axis.grid(alpha=0.25)
    figure.tight_layout()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_path, dpi=160)
    plt.close(figure)

# plot_saved_history(DIRS["histories"] / "training.jsonl", DIRS["plots"] / "training_history.png")

## 12. Resumable encoder-decoder generation and prediction management

Inference caches the encoder output, decoder cross-attention keys/values, and autoregressive self-attention keys/values. The encoder positional limit is 1,024 BPE tokens; overlong prompts retain the first 128 tokens plus the most recent 896 tokens, and every truncation is recorded. The decoder allows up to 448 generated tokens for a verified 128-word continuation. Run the separate 16-example benchmark before official generation to measure speed and budget success.


In [ ]:
WORD_PATTERN = re.compile(r"\S+")
def words(text: str) -> list[str]: return WORD_PATTERN.findall(text.strip())
def exact_word_prefix(text: str, count: int = 128) -> str:
    units = words(text)
    if len(units) < count: raise ValueError(f"Only {len(units)} words, need {count}")
    return " ".join(units[:count])

def _policy_value(policy: Mapping[str, Any], name: str, default=None):
    value = policy.get(name, default)
    return default if value is None else value

def select_next_token(logits: torch.Tensor, generated: Sequence[int], policy: Mapping[str, Any], generator):
    scores = logits.float().clone()
    repetition_penalty = float(_policy_value(policy, "repetition_penalty", 1.0))
    if repetition_penalty <= 0:
        raise ValueError("repetition_penalty must be positive")
    repeated = sorted(set(int(value) for value in generated))
    if repeated and repetition_penalty != 1.0:
        repeated_ids = torch.tensor(repeated, dtype=torch.long, device=scores.device)
        selected = scores.index_select(0, repeated_ids)
        scores[repeated_ids] = torch.where(
            selected > 0, selected / repetition_penalty, selected * repetition_penalty
        )
    ngram = int(_policy_value(policy, "no_repeat_ngram_size", 0))
    if ngram > 0 and len(generated) >= ngram - 1:
        prefix = tuple(generated[-(ngram - 1):]) if ngram > 1 else tuple()
        banned = set()
        for index in range(len(generated) - ngram + 1):
            gram = tuple(generated[index:index + ngram])
            if gram[:-1] == prefix:
                banned.add(gram[-1])
        if banned:
            banned_ids = torch.tensor(sorted(banned), dtype=torch.long, device=scores.device)
            scores[banned_ids] = -torch.inf
    strategy = str(policy.get("strategy"))
    if strategy == "greedy":
        return int(scores.argmax().item())
    if strategy not in {"top_k", "top_p", "top_k_top_p"}:
        raise ValueError(f"Unsupported declared decoding strategy: {strategy}")
    temperature = float(_policy_value(policy, "temperature", 1.0))
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    scores = scores / temperature
    top_k = int(_policy_value(policy, "top_k", 0))
    if top_k > 0:
        candidate_scores, candidate_ids = torch.topk(scores, min(top_k, scores.numel()))
    else:
        candidate_scores = scores
        candidate_ids = torch.arange(scores.numel(), device=scores.device)
    top_p = float(_policy_value(policy, "top_p", 1.0))
    if not 0 < top_p <= 1:
        raise ValueError("top_p must be in (0, 1]")
    if top_p < 1:
        candidate_scores, order = torch.sort(candidate_scores, descending=True)
        candidate_ids = candidate_ids.index_select(0, order)
        probabilities = torch.softmax(candidate_scores, dim=-1)
        cumulative = probabilities.cumsum(dim=-1)
        remove = cumulative > top_p
        remove[1:] = remove[:-1].clone(); remove[0] = False
        candidate_scores = candidate_scores.masked_fill(remove, -torch.inf)
    probabilities = torch.softmax(candidate_scores, dim=-1)
    if not torch.isfinite(probabilities).all() or float(probabilities.sum()) <= 0:
        raise FloatingPointError("Invalid sampling distribution")
    sampled_position = torch.multinomial(probabilities, 1, generator=generator)
    return int(candidate_ids.index_select(0, sampled_position).item())

def generation_quality_report(text: str) -> dict[str, Any]:
    units = words(text); lowered = [unit.lower() for unit in units]
    counts = Counter(lowered)
    return {
        "word_count": len(units),
        "unique_word_ratio": len(counts) / max(1, len(units)),
        "max_word_fraction": max(counts.values(), default=0) / max(1, len(units)),
    }

def bounded_encoder_prompt(token_ids: Sequence[int], maximum: int) -> tuple[list[int], dict[str, Any]]:
    token_ids = list(token_ids)
    if len(token_ids) <= maximum:
        return token_ids, {"original_prompt_tokens": len(token_ids), "retained_prompt_tokens": len(token_ids), "prompt_truncated": False, "truncation": "none"}
    head = min(128, maximum // 4)
    retained = [*token_ids[:head], *token_ids[-(maximum - head):]]
    return retained, {
        "original_prompt_tokens": len(token_ids),
        "retained_prompt_tokens": len(retained),
        "prompt_truncated": True,
        "truncation": f"head_{head}_plus_tail_{maximum - head}",
    }

@torch.inference_mode()
def seq2seq_continuation(model, tokenizer, prompt, device, *, policy, target_words=128, max_new_tokens=None):
    if max_new_tokens is None:
        raise ValueError("A declared max_new_tokens is required")
    max_encoder_length = int(MODEL_CONFIG.get("max_encoder_length", MODEL_CONFIG.get("context_length", 0)))
    if max_encoder_length <= 0:
        raise ValueError("MODEL_CONFIG must declare max_encoder_length")
    max_decoder_length = int(MODEL_CONFIG.get("max_decoder_length", 0))
    if max_decoder_length <= 1:
        raise ValueError("MODEL_CONFIG must declare max_decoder_length")
    if int(max_new_tokens) + 1 > max_decoder_length:
        raise ValueError("max_new_tokens plus BOS exceeds the configured decoder context")
    prompt_ids = tokenizer_encode(tokenizer, prompt)
    if not prompt_ids:
        raise ValueError("Prompt encoded to no tokens")
    encoder_ids, prompt_report = bounded_encoder_prompt(prompt_ids, max_encoder_length)
    bos_id = getattr(tokenizer, "bos_token_id", None); eos_id = getattr(tokenizer, "eos_token_id", None)
    if bos_id is None:
        raise ValueError("Encoder-decoder inference requires bos_token_id")
    encoder = torch.tensor([encoder_ids], dtype=torch.long, device=device)
    encoder_mask = torch.ones_like(encoder, dtype=torch.bool)
    generated = []
    seed = int(policy.get("seed"))
    generator = torch.Generator(device=device).manual_seed(seed)
    model = model.to(device); model.eval()
    prepare = getattr(MODEL_MODULE, "prepare_generation", None)
    decode = getattr(MODEL_MODULE, "decode_step", None)
    cache_used = callable(prepare) and callable(decode)
    if cache_used:
        generation_state = prepare(model, encoder, encoder_mask)
        next_input = torch.tensor([[int(bos_id)]], dtype=torch.long, device=device)
    check_interval = max(1, int(CFG.decoding.get("decode_check_interval", 1)))
    for step in range(int(max_new_tokens)):
        if cache_used:
            step_logits, generation_state = decode(model, next_input, generation_state)
            next_id = select_next_token(step_logits[0], generated, policy, generator)
        else:
            decoder = torch.tensor([[int(bos_id), *generated]], dtype=torch.long, device=device)
            logits = seq2seq_logits(model, encoder, decoder, encoder_mask)
            next_id = select_next_token(logits[0, -1], generated, policy, generator)
        generated.append(next_id)
        if cache_used:
            next_input = torch.tensor([[next_id]], dtype=torch.long, device=device)
        should_decode = (
            len(generated) % check_interval == 0
            or step + 1 == int(max_new_tokens)
            or (eos_id is not None and next_id == int(eos_id))
        )
        if should_decode:
            continuation = tokenizer_decode(tokenizer, generated)
            if len(words(continuation)) >= target_words + 1:
                result = exact_word_prefix(continuation, target_words)
                report = generation_quality_report(result)
                report.update(prompt_report)
                report.update({
                    "generated_tokens": len(generated),
                    "max_new_tokens": int(max_new_tokens),
                    "token_budget_utilization": len(generated) / int(max_new_tokens),
                    "incremental_kv_cache": cache_used,
                    "decode_check_interval": check_interval,
                })
                if report["unique_word_ratio"] < float(_policy_value(policy, "min_unique_word_ratio", 0.0)):
                    raise RuntimeError(f"Declared uniqueness threshold failed: {report}")
                if report["max_word_fraction"] > float(_policy_value(policy, "max_word_fraction", 1.0)):
                    raise RuntimeError(f"Declared repetition threshold failed: {report}")
                return result, report
            if eos_id is not None and next_id == int(eos_id):
                raise RuntimeError(f"Premature EOS after {len(words(continuation))} words")
    raise RuntimeError(f"Generation ended before {target_words} verified words")

def load_test_records(path: Path, schema: Mapping[str, Any]) -> list[dict[str, str]]:
    id_field, prompt_field = schema["id_field"], schema["text_field"]
    result, seen = [], set()
    for index, row in enumerate(iter_json_records(path, id_field=id_field, text_field=prompt_field)):
        identifier = str(row[id_field]); prompt = normalize_text(row[prompt_field])
        if identifier in seen: raise ValueError(f"Duplicate official ID: {identifier}")
        seen.add(identifier); result.append({"id": identifier, "prompt": prompt})
    return result

def test_records_fingerprint(records):
    digest = hashlib.sha256()
    for row in records:
        digest.update(f"{row['id']}\t{hashlib.sha256(row['prompt'].encode()).hexdigest()}\n".encode())
    return digest.hexdigest()


In [ ]:
def load_selected_model_for_inference(checkpoint_path, tokenizer_meta, reconstruct_model, device):
    state = torch.load(Path(checkpoint_path), map_location="cpu", weights_only=False)
    if state.get("tokenizer_fingerprint") != tokenizer_meta["fingerprint"]:
        raise RuntimeError("Selected checkpoint tokenizer mismatch")
    if state.get("model_source_sha256") != MODEL_SOURCE_SHA256:
        raise RuntimeError("Selected checkpoint model-source mismatch")
    restored = reconstruct_model(state["model_config"])
    restored.load_state_dict(state["model_state"], strict=True)
    return restored.to(device).eval(), state

def generate_official_predictions(model, tokenizer, test_records, device, *, checkpoint_id, tokenizer_meta):
    policies = list(CFG.decoding.get("policies") or [])
    max_new_tokens = CFG.decoding.get("max_new_tokens")
    if not policies or max_new_tokens is None:
        raise RuntimeError("Student must declare ordered decoding policies and max_new_tokens before inference")
    for index, policy in enumerate(policies):
        if not isinstance(policy, Mapping) or not policy.get("name") or policy.get("seed") is None:
            raise ValueError(f"Decoding policy {index} requires a name and deterministic seed")
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", checkpoint_id)
    partial_path = DIRS["predictions"] / f"official_predictions.{safe}.partial.json"
    meta_path = partial_path.with_suffix(".meta.json")
    attempts_path = DIRS["predictions"] / f"generation_attempts.{safe}.jsonl"
    expected_meta = {
        "checkpoint_id": checkpoint_id,
        "tokenizer_fingerprint": tokenizer_meta["fingerprint"],
        "model_source_sha256": MODEL_SOURCE_SHA256,
        "test_fingerprint": test_records_fingerprint(test_records),
        "decoding": dict(CFG.decoding),
    }
    predictions = {}
    if partial_path.exists():
        if not meta_path.exists() or json.loads(meta_path.read_text(encoding="utf-8")) != expected_meta:
            raise RuntimeError("Partial predictions do not authenticate current checkpoint/tokenizer/model/test/decoding")
        predictions = json.loads(partial_path.read_text(encoding="utf-8"))
        expected_ids = {row["id"] for row in test_records}
        if set(predictions) - expected_ids: raise RuntimeError("Partial prediction file contains foreign IDs")
        for key, value in predictions.items():
            if not isinstance(value, dict) or set(value) != {"text"} or len(words(value["text"])) != 128:
                raise RuntimeError(f"Invalid cached prediction: {key}")
    else:
        atomic_write_json(meta_path, expected_meta)
    failures = []
    for row in tqdm(test_records, desc="Official test generation"):
        if row["id"] in predictions: continue
        completed = False; last_error = None
        for policy_index, policy in enumerate(policies):
            effective_policy = dict(policy)
            effective_policy["seed"] = int(policy["seed"]) + int(hashlib.sha256(row["id"].encode()).hexdigest()[:8], 16)
            started = time.perf_counter()
            try:
                continuation, quality = seq2seq_continuation(
                    model, tokenizer, row["prompt"], device, policy=effective_policy,
                    target_words=128, max_new_tokens=max_new_tokens,
                )
                record = {
                    "id": row["id"], "policy_index": policy_index, "policy_name": policy["name"],
                    "status": "success", "quality": quality, "seconds": time.perf_counter() - started,
                    "created_utc": utc_now(),
                }
                with attempts_path.open("a", encoding="utf-8") as handle:
                    handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                predictions[row["id"]] = {"text": continuation}
                atomic_write_json(partial_path, predictions)
                completed = True; break
            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                with attempts_path.open("a", encoding="utf-8") as handle:
                    handle.write(json.dumps({
                        "id": row["id"], "policy_index": policy_index, "policy_name": policy["name"],
                        "status": "failed", "error": last_error,
                        "seconds": time.perf_counter() - started, "created_utc": utc_now(),
                    }, ensure_ascii=False) + "\n")
        if not completed: failures.append({"id": row["id"], "error": last_error})
    atomic_write_json(DIRS["predictions"] / "generation_failures.json", failures)
    if failures: raise RuntimeError(f"Generation remains incomplete for {len(failures)} IDs; submission packaging is blocked")
    provenance = {
        **expected_meta, "prediction_count": len(predictions),
        "partial_prediction_sha256": sha256_file(partial_path), "attempt_log_sha256": sha256_file(attempts_path),
        "created_utc": utc_now(),
    }
    atomic_write_json(DIRS["predictions"] / "inference_provenance.json", provenance)
    return predictions, failures

def run_local_continuation_evaluation(model, tokenizer, rows, device, *, checkpoint_id):
    if not rows:
        raise ValueError("Local continuation evaluation has no examples")
    policies = list(CFG.decoding.get("policies") or [])
    predictions = []; references = []; diagnostics = []; attempts = []; failures = []
    started_all = time.perf_counter()
    for row in tqdm(rows, desc="Local continuation evaluation"):
        completed = False; last_error = None
        for policy_index, policy in enumerate(policies):
            effective_policy = dict(policy)
            effective_policy["seed"] = int(policy["seed"]) + int(hashlib.sha256(row["id"].encode()).hexdigest()[:8], 16)
            started = time.perf_counter()
            try:
                continuation, quality = seq2seq_continuation(
                    model, tokenizer, row["prompt"], device, policy=effective_policy,
                    target_words=CFG.decoding["target_words"], max_new_tokens=CFG.decoding["max_new_tokens"],
                )
                elapsed = time.perf_counter() - started
                attempts.append({"id": row["id"], "policy": policy["name"], "status": "success", "seconds": elapsed})
                predictions.append(continuation); references.append(row["reference"])
                diagnostics.append({"id": row["id"], "source": row["source"], "seconds": elapsed, **quality})
                completed = True
                break
            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                attempts.append({"id": row["id"], "policy": policy["name"], "status": "failed", "error": last_error, "seconds": time.perf_counter() - started})
        if not completed:
            failures.append({"id": row["id"], "error": last_error})
    record = {
        "checkpoint_id": checkpoint_id,
        "created_utc": utc_now(),
        "protocol": "fixed held-out 512-word prompts and 128-word references; declared ordered decoding policies",
        "examples_requested": len(rows),
        "examples_completed": len(predictions),
        "failures": failures,
        "attempts": attempts,
        "diagnostics": diagnostics,
        "total_seconds": time.perf_counter() - started_all,
        "dependency_free_metrics": compute_local_text_metrics(predictions, references) if predictions else None,
        "official_evaluator_metrics": None,
    }
    if predictions and OFFICIAL_EVALUATOR is not None:
        record["official_evaluator_metrics"] = run_official_evaluator(
            OFFICIAL_EVALUATOR, predictions, references, f"{checkpoint_id}-local"
        )["metrics"]
    if diagnostics:
        record["generation_summary"] = {
            "mean_seconds_per_example": sum(item["seconds"] for item in diagnostics) / len(diagnostics),
            "max_generated_tokens": max(item["generated_tokens"] for item in diagnostics),
            "mean_generated_tokens": sum(item["generated_tokens"] for item in diagnostics) / len(diagnostics),
            "max_token_budget_utilization": max(item["token_budget_utilization"] for item in diagnostics),
            "truncated_prompt_count": sum(bool(item["prompt_truncated"]) for item in diagnostics),
            "incremental_kv_cache_used": all(bool(item["incremental_kv_cache"]) for item in diagnostics),
        }
    output = DIRS["evaluations"] / f"{checkpoint_id}.local.json"
    atomic_write_json(output, record)
    if failures:
        raise RuntimeError(f"Local evaluation failed for {len(failures)} examples; inspect {output}")
    return record

def run_phase1_inference_pipeline(device="cuda:0", run_local_evaluation=False):
    if CFG.execution_mode != "inference_only":
        print("Inference may also be run after training; execution mode is", CFG.execution_mode)
    if MODEL is None or tokenizer is None or RECONSTRUCT_MODEL is None:
        raise RuntimeError("Restore the tokenizer and student model source/config before inference")
    selected = DIRS["checkpoints"] / "selected_model.pt"
    if not selected.exists(): raise FileNotFoundError("selected_model.pt is required")
    selected_model, state = load_selected_model_for_inference(selected, TOKENIZER_META, RECONSTRUCT_MODEL, device)
    checkpoint_id = f"selected-{sha256_file(selected)[:12]}"
    local_evaluation = None
    if run_local_evaluation:
        examples_path = DIRS["data"] / "local_continuation_examples.json"
        if not examples_path.exists():
            raise RuntimeError("Optional local evaluation requires saved development examples")
        local_rows = json.loads(examples_path.read_text(encoding="utf-8"))
        local_evaluation = run_local_continuation_evaluation(
            selected_model, tokenizer, local_rows, device, checkpoint_id=checkpoint_id
        )
    schema_path = DIRS["manifests"] / "resolved_json_schemas.json"
    if RESOLVED_SCHEMA is not None:
        schemas = RESOLVED_SCHEMA
    elif schema_path.exists():
        schemas = json.loads(schema_path.read_text(encoding="utf-8"))
    else:
        raise RuntimeError("Official test schema is unavailable")
    test_records = load_test_records(require_unique_file(DISCOVERED, CFG.test_file), schemas["test"])
    predictions, failures = generate_official_predictions(
        selected_model, tokenizer, test_records, device,
        checkpoint_id=checkpoint_id, tokenizer_meta=TOKENIZER_META,
    )
    return selected_model, local_evaluation, test_records, predictions, failures

def run_inference_benchmark(device="cuda:0", examples=16):
    if MODEL is None or tokenizer is None or RECONSTRUCT_MODEL is None:
        raise RuntimeError("Restore the tokenizer and student model source before benchmarking")
    selected = DIRS["checkpoints"] / "selected_model.pt"
    if not selected.exists():
        raise FileNotFoundError("selected_model.pt is required")
    benchmark_model, _ = load_selected_model_for_inference(selected, TOKENIZER_META, RECONSTRUCT_MODEL, device)
    rows_path = DIRS["data"] / "local_continuation_examples.json"
    if not rows_path.exists():
        raise FileNotFoundError("Local continuation examples are required for the benchmark")
    rows = json.loads(rows_path.read_text(encoding="utf-8"))[:int(examples)]
    checkpoint_id = f"selected-{sha256_file(selected)[:12]}-benchmark-{len(rows)}"
    return run_local_continuation_evaluation(
        benchmark_model, tokenizer, rows, device, checkpoint_id=checkpoint_id
    )

RUN_INFERENCE_BENCHMARK = True
INFERENCE_BENCHMARK_EXAMPLES = 16
INFERENCE_BENCHMARK = (
    run_inference_benchmark("cuda:0", INFERENCE_BENCHMARK_EXAMPLES)
    if RUN_INFERENCE_BENCHMARK else None
)

RUN_INFERENCE = False
RUN_OPTIONAL_LOCAL_EVALUATION = False
if RUN_INFERENCE:
    MODEL, EVAL_RECORD, TEST_RECORDS, PREDICTIONS, FAILURES = run_phase1_inference_pipeline(
        "cuda:0", run_local_evaluation=RUN_OPTIONAL_LOCAL_EVALUATION
    )
else:
    print("Inference is opt-in and does not require the training corpus or local evaluator.")


In [ ]:
def load_selected_model_for_inference(
    checkpoint_path,
    tokenizer_meta,
    reconstruct_model,
    device,
):
    state = torch.load(
        Path(checkpoint_path),
        map_location="cpu",
        weights_only=False,
    )

    if state.get("tokenizer_fingerprint") != tokenizer_meta["fingerprint"]:
        raise RuntimeError("Selected checkpoint tokenizer mismatch")

    if state.get("model_source_sha256") != MODEL_SOURCE_SHA256:
        raise RuntimeError("Selected checkpoint model-source mismatch")

    restored = reconstruct_model(state["model_config"])
    restored.load_state_dict(
        state["model_state"],
        strict=True,
    )

    return restored.to(device).eval(), state


def generate_official_predictions(
    model,
    tokenizer,
    test_records,
    device,
    *,
    checkpoint_id,
    tokenizer_meta,
):
    policies = list(CFG.decoding.get("policies") or [])
    max_new_tokens = CFG.decoding.get("max_new_tokens")

    if not policies or max_new_tokens is None:
        raise RuntimeError(
            "Student must declare ordered decoding policies "
            "and max_new_tokens before inference"
        )

    for index, policy in enumerate(policies):
        if (
            not isinstance(policy, Mapping)
            or not policy.get("name")
            or policy.get("seed") is None
        ):
            raise ValueError(
                f"Decoding policy {index} requires a name "
                "and deterministic seed"
            )

    safe = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        checkpoint_id,
    )

    partial_path = (
        DIRS["predictions"]
        / f"official_predictions.{safe}.partial.json"
    )

    meta_path = partial_path.with_suffix(".meta.json")

    attempts_path = (
        DIRS["predictions"]
        / f"generation_attempts.{safe}.jsonl"
    )

    expected_meta = {
        "checkpoint_id": checkpoint_id,
        "tokenizer_fingerprint": tokenizer_meta["fingerprint"],
        "model_source_sha256": MODEL_SOURCE_SHA256,
        "test_fingerprint": test_records_fingerprint(test_records),
        "decoding": dict(CFG.decoding),
    }

    predictions = {}

    if partial_path.exists():

        if (
            not meta_path.exists()
            or json.loads(
                meta_path.read_text(encoding="utf-8")
            )
            != expected_meta
        ):
            raise RuntimeError(
                "Partial predictions do not authenticate current "
                "checkpoint/tokenizer/model/test/decoding"
            )

        predictions = json.loads(
            partial_path.read_text(encoding="utf-8")
        )

        expected_ids = {
            row["id"]
            for row in test_records
        }

        if set(predictions) - expected_ids:
            raise RuntimeError(
                "Partial prediction file contains foreign IDs"
            )

        for key, value in predictions.items():

            if (
                not isinstance(value, dict)
                or set(value) != {"text"}
                or len(words(value["text"])) != 128
            ):
                raise RuntimeError(
                    f"Invalid cached prediction: {key}"
                )

    else:
        atomic_write_json(
            meta_path,
            expected_meta,
        )

    failures = []

    for row in tqdm(
        test_records,
        desc="Official test generation",
    ):

        if row["id"] in predictions:
            continue

        completed = False
        last_error = None

        for policy_index, policy in enumerate(policies):

            effective_policy = dict(policy)

            effective_policy["seed"] = (
                int(policy["seed"])
                + int(
                    hashlib.sha256(
                        row["id"].encode()
                    ).hexdigest()[:8],
                    16,
                )
            )

            started = time.perf_counter()

            try:
                continuation, quality = seq2seq_continuation(
                    model,
                    tokenizer,
                    row["prompt"],
                    device,
                    policy=effective_policy,
                    target_words=128,
                    max_new_tokens=max_new_tokens,
                )

                record = {
                    "id": row["id"],
                    "policy_index": policy_index,
                    "policy_name": policy["name"],
                    "status": "success",
                    "quality": quality,
                    "seconds": (
                        time.perf_counter() - started
                    ),
                    "created_utc": utc_now(),
                }

                with attempts_path.open(
                    "a",
                    encoding="utf-8",
                ) as handle:
                    handle.write(
                        json.dumps(
                            record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

                predictions[row["id"]] = {
                    "text": continuation
                }

                atomic_write_json(
                    partial_path,
                    predictions,
                )

                completed = True
                break

            except Exception as exc:

                last_error = (
                    f"{type(exc).__name__}: {exc}"
                )

                with attempts_path.open(
                    "a",
                    encoding="utf-8",
                ) as handle:
                    handle.write(
                        json.dumps(
                            {
                                "id": row["id"],
                                "policy_index": policy_index,
                                "policy_name": policy["name"],
                                "status": "failed",
                                "error": last_error,
                                "seconds": (
                                    time.perf_counter()
                                    - started
                                ),
                                "created_utc": utc_now(),
                            },
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

        if not completed:
            failures.append(
                {
                    "id": row["id"],
                    "error": last_error,
                }
            )

    atomic_write_json(
        DIRS["predictions"]
        / "generation_failures.json",
        failures,
    )

    if failures:
        raise RuntimeError(
            f"Generation remains incomplete for "
            f"{len(failures)} IDs; submission packaging is blocked"
        )

    provenance = {
        **expected_meta,
        "prediction_count": len(predictions),
        "partial_prediction_sha256": sha256_file(
            partial_path
        ),
        "attempt_log_sha256": sha256_file(
            attempts_path
        ),
        "created_utc": utc_now(),
    }

    atomic_write_json(
        DIRS["predictions"]
        / "inference_provenance.json",
        provenance,
    )

    return predictions, failures


def run_local_continuation_evaluation(
    model,
    tokenizer,
    rows,
    device,
    *,
    checkpoint_id,
):
    if not rows:
        raise ValueError(
            "Local continuation evaluation has no examples"
        )

    policies = list(
        CFG.decoding.get("policies") or []
    )

    predictions = []
    references = []
    diagnostics = []
    attempts = []
    failures = []

    started_all = time.perf_counter()

    for row in tqdm(
        rows,
        desc="Local continuation evaluation",
    ):

        completed = False
        last_error = None

        for policy_index, policy in enumerate(policies):

            effective_policy = dict(policy)

            effective_policy["seed"] = (
                int(policy["seed"])
                + int(
                    hashlib.sha256(
                        row["id"].encode()
                    ).hexdigest()[:8],
                    16,
                )
            )

            started = time.perf_counter()

            try:
                continuation, quality = seq2seq_continuation(
                    model,
                    tokenizer,
                    row["prompt"],
                    device,
                    policy=effective_policy,
                    target_words=CFG.decoding[
                        "target_words"
                    ],
                    max_new_tokens=CFG.decoding[
                        "max_new_tokens"
                    ],
                )

                elapsed = (
                    time.perf_counter() - started
                )

                attempts.append(
                    {
                        "id": row["id"],
                        "policy": policy["name"],
                        "status": "success",
                        "seconds": elapsed,
                    }
                )

                predictions.append(
                    continuation
                )

                references.append(
                    row["reference"]
                )

                diagnostics.append(
                    {
                        "id": row["id"],
                        "source": row["source"],
                        "seconds": elapsed,
                        **quality,
                    }
                )

                completed = True
                break

            except Exception as exc:

                last_error = (
                    f"{type(exc).__name__}: {exc}"
                )

                attempts.append(
                    {
                        "id": row["id"],
                        "policy": policy["name"],
                        "status": "failed",
                        "error": last_error,
                        "seconds": (
                            time.perf_counter()
                            - started
                        ),
                    }
                )

        if not completed:
            failures.append(
                {
                    "id": row["id"],
                    "error": last_error,
                }
            )

    record = {
        "checkpoint_id": checkpoint_id,
        "created_utc": utc_now(),
        "protocol": (
            "fixed held-out 512-word prompts and "
            "128-word references; declared ordered "
            "decoding policies"
        ),
        "examples_requested": len(rows),
        "examples_completed": len(predictions),
        "failures": failures,
        "attempts": attempts,
        "diagnostics": diagnostics,
        "total_seconds": (
            time.perf_counter() - started_all
        ),
        "dependency_free_metrics": (
            compute_local_text_metrics(
                predictions,
                references,
            )
            if predictions
            else None
        ),
        "official_evaluator_metrics": None,
    }

    if (
        predictions
        and OFFICIAL_EVALUATOR is not None
    ):
        record[
            "official_evaluator_metrics"
        ] = run_official_evaluator(
            OFFICIAL_EVALUATOR,
            predictions,
            references,
            f"{checkpoint_id}-local",
        )["metrics"]

    if diagnostics:
        record["generation_summary"] = {
            "mean_seconds_per_example": (
                sum(
                    item["seconds"]
                    for item in diagnostics
                )
                / len(diagnostics)
            ),
            "max_generated_tokens": max(
                item["generated_tokens"]
                for item in diagnostics
            ),
            "mean_generated_tokens": (
                sum(
                    item["generated_tokens"]
                    for item in diagnostics
                )
                / len(diagnostics)
            ),
            "max_token_budget_utilization": max(
                item[
                    "token_budget_utilization"
                ]
                for item in diagnostics
            ),
            "truncated_prompt_count": sum(
                bool(
                    item["prompt_truncated"]
                )
                for item in diagnostics
            ),
            "incremental_kv_cache_used": all(
                bool(
                    item[
                        "incremental_kv_cache"
                    ]
                )
                for item in diagnostics
            ),
        }

    output = (
        DIRS["evaluations"]
        / f"{checkpoint_id}.local.json"
    )

    atomic_write_json(
        output,
        record,
    )

    if failures:
        raise RuntimeError(
            f"Local evaluation failed for "
            f"{len(failures)} examples; inspect {output}"
        )

    return record


def run_phase1_inference_pipeline(
    device="cuda:0",
    run_local_evaluation=False,
):

    if CFG.execution_mode != "inference_only":
        print(
            "Inference may also be run after training; "
            "execution mode is",
            CFG.execution_mode,
        )

    if (
        MODEL is None
        or tokenizer is None
        or RECONSTRUCT_MODEL is None
    ):
        raise RuntimeError(
            "Restore the tokenizer and student model "
            "source/config before inference"
        )

    selected = (
        DIRS["checkpoints"]
        / "selected_model.pt"
    )

    if not selected.exists():
        raise FileNotFoundError(
            "selected_model.pt is required"
        )

    selected_model, state = (
        load_selected_model_for_inference(
            selected,
            TOKENIZER_META,
            RECONSTRUCT_MODEL,
            device,
        )
    )

    checkpoint_id = (
        f"selected-{sha256_file(selected)[:12]}"
    )

    local_evaluation = None

    if run_local_evaluation:

        examples_path = (
            DIRS["data"]
            / "local_continuation_examples.json"
        )

        if not examples_path.exists():
            raise RuntimeError(
                "Optional local evaluation requires "
                "saved development examples"
            )

        local_rows = json.loads(
            examples_path.read_text(
                encoding="utf-8"
            )
        )

        local_evaluation = (
            run_local_continuation_evaluation(
                selected_model,
                tokenizer,
                local_rows,
                device,
                checkpoint_id=checkpoint_id,
            )
        )

    schema_path = (
        DIRS["manifests"]
        / "resolved_json_schemas.json"
    )

    if RESOLVED_SCHEMA is not None:

        schemas = RESOLVED_SCHEMA

    elif schema_path.exists():

        schemas = json.loads(
            schema_path.read_text(
                encoding="utf-8"
            )
        )

    else:

        raise RuntimeError(
            "Official test schema is unavailable"
        )

    test_records = load_test_records(
        require_unique_file(
            DISCOVERED,
            CFG.test_file,
        ),
        schemas["test"],
    )

    predictions, failures = (
        generate_official_predictions(
            selected_model,
            tokenizer,
            test_records,
            device,
            checkpoint_id=checkpoint_id,
            tokenizer_meta=TOKENIZER_META,
        )
    )

    return (
        selected_model,
        local_evaluation,
        test_records,
        predictions,
        failures,
    )


def run_inference_benchmark(
    device="cuda:0",
    examples=16,
):

    if (
        MODEL is None
        or tokenizer is None
        or RECONSTRUCT_MODEL is None
    ):
        raise RuntimeError(
            "Restore the tokenizer and student "
            "model source before benchmarking"
        )

    selected = (
        DIRS["checkpoints"]
        / "selected_model.pt"
    )

    if not selected.exists():
        raise FileNotFoundError(
            "selected_model.pt is required"
        )

    benchmark_model, _ = (
        load_selected_model_for_inference(
            selected,
            TOKENIZER_META,
            RECONSTRUCT_MODEL,
            device,
        )
    )

    rows_path = (
        DIRS["data"]
        / "local_continuation_examples.json"
    )

    if not rows_path.exists():
        raise FileNotFoundError(
            "Local continuation examples are "
            "required for the benchmark"
        )

    rows = json.loads(
        rows_path.read_text(
            encoding="utf-8"
        )
    )[:int(examples)]

    checkpoint_id = (
        f"selected-{sha256_file(selected)[:12]}"
        f"-benchmark-{len(rows)}"
    )

    return run_local_continuation_evaluation(
        benchmark_model,
        tokenizer,
        rows,
        device,
        checkpoint_id=checkpoint_id,
    )


# ============================================================
# EXECUTION CONTROLS
# ============================================================

# The benchmark is optional and must NOT block official inference.
RUN_INFERENCE_BENCHMARK = False
INFERENCE_BENCHMARK_EXAMPLES = 16

INFERENCE_BENCHMARK = (
    run_inference_benchmark(
        "cuda:0",
        INFERENCE_BENCHMARK_EXAMPLES,
    )
    if RUN_INFERENCE_BENCHMARK
    else None
)


# Run the actual official test-set generation.
RUN_INFERENCE = True

# Keep optional held-out generation evaluation disabled.
# It is not required for creating prediction.json/submission.zip.
RUN_OPTIONAL_LOCAL_EVALUATION = False


if RUN_INFERENCE:

    (
        MODEL,
        EVAL_RECORD,
        TEST_RECORDS,
        PREDICTIONS,
        FAILURES,
    ) = run_phase1_inference_pipeline(
        "cuda:0",
        run_local_evaluation=(
            RUN_OPTIONAL_LOCAL_EVALUATION
        ),
    )

    print()
    print("=" * 60)
    print("OFFICIAL INFERENCE FINISHED")
    print("=" * 60)
    print("Test records :", len(TEST_RECORDS))
    print("Predictions  :", len(PREDICTIONS))
    print("Failures     :", len(FAILURES))

else:

    print(
        "Inference is opt-in and does not require "
        "the training corpus or local evaluator."
    )

In [ ]:
# ============================================================
# FINAL SUBMISSION-02 RECOVERY + GENERATION + PACKAGING CELL
# Run this ONE cell only.
# ============================================================

from pathlib import Path
import copy
import hashlib
import json
import re
import time
import zipfile

import torch
from tqdm.auto import tqdm


print("=" * 80)
print("PHASE 1 SUBMISSION 02 — FINALIZATION")
print("=" * 80)


# ------------------------------------------------------------
# 1. BASIC PATHS / CHECKS
# ------------------------------------------------------------

selected_path = DIRS["checkpoints"] / "selected_model.pt"

if not selected_path.exists():
    raise FileNotFoundError(
        f"Missing selected model: {selected_path}"
    )

print("Selected model:", selected_path)


# ------------------------------------------------------------
# 2. LOAD THE SELECTED TRAINED MODEL
# ------------------------------------------------------------

FINAL_MODEL, FINAL_MODEL_STATE = load_selected_model_for_inference(
    selected_path,
    TOKENIZER_META,
    RECONSTRUCT_MODEL,
    "cuda:0",
)

checkpoint_id = f"selected-{sha256_file(selected_path)[:12]}"

print("Checkpoint ID:", checkpoint_id)


# ------------------------------------------------------------
# 3. LOAD OFFICIAL TEST RECORDS
# ------------------------------------------------------------

schema_path = DIRS["manifests"] / "resolved_json_schemas.json"

if RESOLVED_SCHEMA is not None:
    schemas = RESOLVED_SCHEMA
elif schema_path.exists():
    schemas = json.loads(
        schema_path.read_text(encoding="utf-8")
    )
else:
    raise RuntimeError(
        "Official test schema is unavailable."
    )

TEST_RECORDS = load_test_records(
    require_unique_file(
        DISCOVERED,
        CFG.test_file,
    ),
    schemas["test"],
)

print("Official test records:", len(TEST_RECORDS))

if len(TEST_RECORDS) != 1000:
    print(
        "WARNING: expected 1000 official test records, "
        f"found {len(TEST_RECORDS)}"
    )


# ------------------------------------------------------------
# 4. FIND THE EXISTING PARTIAL PREDICTION FILE
# ------------------------------------------------------------

safe_checkpoint_id = re.sub(
    r"[^A-Za-z0-9_.-]+",
    "_",
    checkpoint_id,
)

partial_path = (
    DIRS["predictions"]
    / f"official_predictions.{safe_checkpoint_id}.partial.json"
)

meta_path = partial_path.with_suffix(".meta.json")

attempts_path = (
    DIRS["predictions"]
    / f"generation_attempts.{safe_checkpoint_id}.jsonl"
)

if partial_path.exists():
    PREDICTIONS = json.loads(
        partial_path.read_text(encoding="utf-8")
    )
else:
    PREDICTIONS = {}

print("Already valid predictions:", len(PREDICTIONS))


# ------------------------------------------------------------
# 5. VERIFY EXISTING PREDICTIONS
# ------------------------------------------------------------

official_ids = {
    str(row["id"])
    for row in TEST_RECORDS
}

bad_cached = []

for pred_id, value in list(PREDICTIONS.items()):

    valid = (
        pred_id in official_ids
        and isinstance(value, dict)
        and set(value.keys()) == {"text"}
        and len(words(value["text"])) == 128
    )

    if not valid:
        bad_cached.append(pred_id)

for pred_id in bad_cached:
    del PREDICTIONS[pred_id]

if bad_cached:
    print(
        "Removed invalid cached predictions:",
        len(bad_cached),
    )

atomic_write_json(
    partial_path,
    PREDICTIONS,
)


# ------------------------------------------------------------
# 6. BUILD RETRY POLICIES
#
# Start with the student's already-declared policies.
# Then retry those same policies with deterministic alternate
# seeds. We are not retraining or modifying model weights.
# ------------------------------------------------------------

base_policies = list(
    CFG.decoding.get("policies") or []
)

if not base_policies:
    raise RuntimeError(
        "CFG.decoding['policies'] is empty."
    )

original_max_new_tokens = int(
    CFG.decoding.get("max_new_tokens") or 0
)

# Give generation more token room, while staying below the
# decoder's configured maximum where possible.
model_decoder_limit = int(
    FINAL_MODEL_STATE.get(
        "model_config",
        {},
    ).get(
        "max_decoder_length",
        FINAL_MODEL_STATE.get(
            "model_config",
            {},
        ).get(
            "max_decoder_positions",
            512,
        ),
    )
)

retry_max_new_tokens = max(
    original_max_new_tokens,
    min(512, model_decoder_limit),
)

print(
    "Generation token budget:",
    original_max_new_tokens,
    "-> retry budget:",
    retry_max_new_tokens,
)


# We preserve each declared policy and only vary deterministic
# seeds for additional attempts.
retry_policies = []

SEED_OFFSETS = [
    0,
    100_003,
    200_003,
    300_007,
    400_009,
    500_009,
    600_011,
    700_001,
]

for base_policy in base_policies:
    for offset in SEED_OFFSETS:

        policy = copy.deepcopy(base_policy)

        policy["seed"] = (
            int(base_policy["seed"])
            + int(offset)
        )

        policy["name"] = (
            f"{base_policy['name']}"
            f"_retry_{offset}"
        )

        retry_policies.append(policy)

print("Retry policies:", len(retry_policies))


# ------------------------------------------------------------
# 7. GENERATE ONLY THE MISSING IDS
# ------------------------------------------------------------

missing_rows = [
    row
    for row in TEST_RECORDS
    if str(row["id"]) not in PREDICTIONS
]

print("Missing predictions before retry:", len(missing_rows))


FINAL_FAILURES = []

for row in tqdm(
    missing_rows,
    desc="Recovering missing official predictions",
):

    row_id = str(row["id"])

    completed = False
    last_error = None

    row_hash = int(
        hashlib.sha256(
            row_id.encode("utf-8")
        ).hexdigest()[:8],
        16,
    )

    for policy_index, policy in enumerate(retry_policies):

        effective_policy = copy.deepcopy(policy)

        effective_policy["seed"] = (
            int(policy["seed"])
            + row_hash
        )

        started = time.perf_counter()

        try:

            continuation, quality = seq2seq_continuation(
                FINAL_MODEL,
                tokenizer,
                row["prompt"],
                "cuda:0",
                policy=effective_policy,
                target_words=128,
                max_new_tokens=retry_max_new_tokens,
            )

            # Hard requirement.
            if len(words(continuation)) != 128:
                raise RuntimeError(
                    "Generated continuation is not exactly "
                    f"128 words: got {len(words(continuation))}"
                )

            PREDICTIONS[row_id] = {
                "text": continuation
            }

            record = {
                "id": row_id,
                "status": "success",
                "policy_index": policy_index,
                "policy_name": policy["name"],
                "quality": quality,
                "seconds": (
                    time.perf_counter()
                    - started
                ),
                "created_utc": utc_now(),
            }

            with attempts_path.open(
                "a",
                encoding="utf-8",
            ) as handle:
                handle.write(
                    json.dumps(
                        record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

            # Save EVERY successful recovered ID.
            atomic_write_json(
                partial_path,
                PREDICTIONS,
            )

            completed = True
            break

        except Exception as exc:

            last_error = (
                f"{type(exc).__name__}: {exc}"
            )

            failure_record = {
                "id": row_id,
                "status": "failed",
                "policy_index": policy_index,
                "policy_name": policy["name"],
                "error": last_error,
                "seconds": (
                    time.perf_counter()
                    - started
                ),
                "created_utc": utc_now(),
            }

            with attempts_path.open(
                "a",
                encoding="utf-8",
            ) as handle:
                handle.write(
                    json.dumps(
                        failure_record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

    if not completed:

        FINAL_FAILURES.append(
            {
                "id": row_id,
                "error": last_error,
            }
        )


atomic_write_json(
    DIRS["predictions"]
    / "generation_failures.json",
    FINAL_FAILURES,
)


# ------------------------------------------------------------
# 8. STRICT FINAL PREDICTION VALIDATION
# ------------------------------------------------------------

print()
print("=" * 80)
print("GENERATION RESULT")
print("=" * 80)

print("Official IDs :", len(official_ids))
print("Predictions  :", len(PREDICTIONS))
print("Failures     :", len(FINAL_FAILURES))


missing_ids = sorted(
    official_ids - set(PREDICTIONS.keys())
)

foreign_ids = sorted(
    set(PREDICTIONS.keys()) - official_ids
)

wrong_lengths = []

for pred_id, value in PREDICTIONS.items():

    if not isinstance(value, dict):
        wrong_lengths.append(pred_id)
        continue

    text = value.get("text", "")

    if len(words(text)) != 128:
        wrong_lengths.append(pred_id)


print("Missing IDs       :", len(missing_ids))
print("Foreign IDs       :", len(foreign_ids))
print("Wrong word lengths:", len(wrong_lengths))


# IMPORTANT:
# Do not create a fake/incomplete Codabench submission.
if (
    FINAL_FAILURES
    or missing_ids
    or foreign_ids
    or wrong_lengths
):

    print()
    print("=" * 80)
    print("SUBMISSION NOT CREATED")
    print("=" * 80)

    print(
        "Some model generations are still invalid. "
        "All successful predictions have been preserved."
    )

    print(
        "Failure file:",
        DIRS["predictions"]
        / "generation_failures.json",
    )

    print(
        "Partial prediction file:",
        partial_path,
    )

    raise RuntimeError(
        "Official prediction set is still incomplete. "
        "The valid generations were saved, but an invalid "
        "submission ZIP was intentionally NOT created."
    )


# ------------------------------------------------------------
# 9. CREATE INFERENCE PROVENANCE
# ------------------------------------------------------------

expected_meta = {
    "checkpoint_id": checkpoint_id,
    "tokenizer_fingerprint": (
        TOKENIZER_META["fingerprint"]
    ),
    "model_source_sha256": (
        MODEL_SOURCE_SHA256
    ),
    "test_fingerprint": (
        test_records_fingerprint(TEST_RECORDS)
    ),
    "decoding": dict(CFG.decoding),
}

provenance = {
    **expected_meta,
    "prediction_count": len(PREDICTIONS),
    "partial_prediction_sha256": (
        sha256_file(partial_path)
    ),
    "attempt_log_sha256": (
        sha256_file(attempts_path)
        if attempts_path.exists()
        else None
    ),
    "recovery_retry_max_new_tokens": (
        retry_max_new_tokens
    ),
    "created_utc": utc_now(),
}

atomic_write_json(
    DIRS["predictions"]
    / "inference_provenance.json",
    provenance,
)


# ------------------------------------------------------------
# 10. WRITE FINAL prediction.json
# ------------------------------------------------------------

submission_dir = DIRS["submission"]

submission_dir.mkdir(
    parents=True,
    exist_ok=True,
)

prediction_path = (
    submission_dir
    / "prediction.json"
)

atomic_write_json(
    prediction_path,
    PREDICTIONS,
)

print()
print(
    "Final prediction.json:",
    prediction_path,
)


# ------------------------------------------------------------
# 11. CREATE submission.zip DIRECTLY
#
# Root of ZIP contains ONLY prediction.json.
# ------------------------------------------------------------

submission_zip = (
    submission_dir
    / "submission.zip"
)

temporary_zip = (
    submission_dir
    / "submission.tmp.zip"
)

temporary_zip.unlink(
    missing_ok=True
)

with zipfile.ZipFile(
    temporary_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    archive.write(
        prediction_path,
        arcname="prediction.json",
    )


# Validate ZIP before promotion.
with zipfile.ZipFile(
    temporary_zip,
    mode="r",
) as archive:

    members = archive.namelist()

    if members != ["prediction.json"]:
        raise RuntimeError(
            "Submission ZIP must contain exactly "
            "root-level prediction.json. "
            f"Found: {members}"
        )

    zipped_predictions = json.loads(
        archive.read(
            "prediction.json"
        ).decode("utf-8")
    )


if set(zipped_predictions.keys()) != official_ids:
    raise RuntimeError(
        "ZIP prediction IDs do not exactly match official IDs."
    )

for pred_id, value in zipped_predictions.items():

    if (
        not isinstance(value, dict)
        or set(value.keys()) != {"text"}
        or len(words(value["text"])) != 128
    ):
        raise RuntimeError(
            f"Invalid ZIP prediction: {pred_id}"
        )


temporary_zip.replace(
    submission_zip
)


# ------------------------------------------------------------
# 12. WRITE SUBMISSION VALIDATION RECORD
# ------------------------------------------------------------

submission_report = {
    "passed": True,
    "prediction_count": len(PREDICTIONS),
    "prediction_sha256": (
        sha256_file(prediction_path)
    ),
    "submission_zip_sha256": (
        sha256_file(submission_zip)
    ),
    "submission_zip_members": [
        "prediction.json"
    ],
    "inference_provenance": provenance,
    "created_utc": utc_now(),
}

atomic_write_json(
    submission_dir
    / "submission_validation.json",
    submission_report,
)


print()
print("=" * 80)
print("SUBMISSION ZIP CREATED")
print("=" * 80)

print(
    "prediction.json:",
    prediction_path,
)

print(
    "submission.zip :",
    submission_zip,
)


# ------------------------------------------------------------
# 13. CREATE COMPLETE RECOVERY ZIP
# ------------------------------------------------------------

print()
print("=" * 80)
print("CREATING FULL RECOVERY ZIP")
print("=" * 80)

RECOVERY_ZIP, RECOVERY_REPORT = (
    create_recovery_zip(ROOT)
)

print(
    "Recovery ZIP:",
    RECOVERY_ZIP,
)


# ------------------------------------------------------------
# 14. RUN FINAL VERIFICATION IF AVAILABLE
# ------------------------------------------------------------

FINAL_REPORT = None

if "final_verification" in globals():

    FINAL_REPORT = final_verification(
        CFG,
        FINAL_MODEL,
        TOKENIZER_META,
        [row["id"] for row in TEST_RECORDS],
    )


# ------------------------------------------------------------
# 15. PRINT ALL IMPORTANT FILES
# ------------------------------------------------------------

print()
print("=" * 80)
print("FINAL ARTIFACTS")
print("=" * 80)

important_paths = [
    DIRS["checkpoints"] / "selected_model.pt",
    DIRS["checkpoints"] / "model_final.pt",
    DIRS["checkpoints"] / "checkpoint.latest.pt",
    DIRS["checkpoints"] / "previous_known_good.pt",
    DIRS["histories"] / "training.jsonl",
    prediction_path,
    submission_zip,
    RECOVERY_ZIP,
    DIRS["recovery"] / "training_recovery.latest.zip",
    DIRS["manifests"] / "final_verification.json",
]

for path in important_paths:

    path = Path(path)

    print(
        "[OK]" if path.exists() else "[MISSING]",
        str(path),
        (
            f"({path.stat().st_size / 1024**2:.1f} MB)"
            if path.exists()
            else ""
        ),
    )


print()
print("=" * 80)
print("DONE")
print("=" * 80)

print(
    "\nCODABENCH FILE TO DOWNLOAD:\n",
    submission_zip,
)

print(
    "\nFULL MODEL/RECOVERY FILE TO DOWNLOAD:\n",
    RECOVERY_ZIP,
)

In [ ]:
# =====================================================================
# SUBMISSION 02 — ONE-CELL FINAL RECOVERY / GENERATION / PACKAGING
#
# Prerequisite:
#   The current Kaggle session is still alive and the notebook setup,
#   tokenizer/model source, DIRS, CFG, helper functions, etc. already exist.
#
# This DOES NOT retrain the model.
# =====================================================================

from pathlib import Path
from collections import Counter
import copy
import hashlib
import json
import os
import re
import tempfile
import time
import zipfile

import torch
from tqdm.auto import tqdm


DEVICE = "cuda:0"

print("=" * 80)
print("SUBMISSION 02 — FINAL RECOVERY")
print("=" * 80)


# =====================================================================
# 1. LOAD SELECTED MODEL
# =====================================================================

selected_path = DIRS["checkpoints"] / "selected_model.pt"

if not selected_path.exists():
    raise FileNotFoundError(
        f"selected_model.pt not found: {selected_path}"
    )

FINAL_MODEL, FINAL_MODEL_STATE = load_selected_model_for_inference(
    selected_path,
    TOKENIZER_META,
    RECONSTRUCT_MODEL,
    DEVICE,
)

MODEL = FINAL_MODEL

checkpoint_id = f"selected-{sha256_file(selected_path)[:12]}"

model_config = FINAL_MODEL_STATE["model_config"]

max_decoder_length = int(
    model_config.get("max_decoder_length", 0)
)

if max_decoder_length < 2:
    raise RuntimeError(
        f"Invalid max_decoder_length: {max_decoder_length}"
    )

# BOS occupies one position.
MAX_GENERATED_TOKENS = max_decoder_length - 1

print("Checkpoint:", checkpoint_id)
print("Decoder context:", max_decoder_length)
print("Usable generated-token budget:", MAX_GENERATED_TOKENS)


# =====================================================================
# 2. LOAD OFFICIAL TEST SET
# =====================================================================

schema_path = DIRS["manifests"] / "resolved_json_schemas.json"

if RESOLVED_SCHEMA is not None:
    schemas = RESOLVED_SCHEMA
elif schema_path.exists():
    schemas = json.loads(
        schema_path.read_text(encoding="utf-8")
    )
else:
    raise RuntimeError(
        "Could not recover official test schema."
    )

TEST_RECORDS = load_test_records(
    require_unique_file(
        DISCOVERED,
        CFG.test_file,
    ),
    schemas["test"],
)

official_ids = [
    str(row["id"])
    for row in TEST_RECORDS
]

official_id_set = set(official_ids)

print("Official test records:", len(TEST_RECORDS))


# =====================================================================
# 3. LOAD THE 542 ALREADY-SUCCESSFUL PREDICTIONS
# =====================================================================

safe_checkpoint_id = re.sub(
    r"[^A-Za-z0-9_.-]+",
    "_",
    checkpoint_id,
)

partial_path = (
    DIRS["predictions"]
    / f"official_predictions.{safe_checkpoint_id}.partial.json"
)

attempts_path = (
    DIRS["predictions"]
    / f"generation_attempts.{safe_checkpoint_id}.jsonl"
)

if partial_path.exists():
    PREDICTIONS = json.loads(
        partial_path.read_text(encoding="utf-8")
    )
else:
    PREDICTIONS = {}


# Keep only genuinely valid cached predictions.
clean_predictions = {}

for pred_id, value in PREDICTIONS.items():

    pred_id = str(pred_id)

    if (
        pred_id in official_id_set
        and isinstance(value, dict)
        and set(value.keys()) == {"text"}
        and len(words(value["text"])) == 128
    ):
        clean_predictions[pred_id] = value

PREDICTIONS = clean_predictions

atomic_write_json(
    partial_path,
    PREDICTIONS,
)

print("Valid cached predictions:", len(PREDICTIONS))
print(
    "Still missing:",
    len(official_id_set - set(PREDICTIONS)),
)


# =====================================================================
# 4. SHOW WHY THE PREVIOUS ATTEMPTS FAILED
# =====================================================================

old_failure_path = (
    DIRS["predictions"]
    / "generation_failures.json"
)

if old_failure_path.exists():

    try:
        old_failures = json.loads(
            old_failure_path.read_text(
                encoding="utf-8"
            )
        )

        failure_counter = Counter(
            row.get("error", "unknown")
            for row in old_failures
        )

        print("\nPrevious failure reasons:")

        for reason, count in failure_counter.most_common(10):
            print(f"  {count:4d}  {reason}")

    except Exception as exc:
        print(
            "Could not summarize previous failure file:",
            exc,
        )


# =====================================================================
# 5. TOKEN-SELECTION HELPER THAT PREVENTS PREMATURE EOS
# =====================================================================

def select_token_without_premature_eos(
    logits,
    generated,
    policy,
    generator,
    eos_id,
    allow_eos,
):
    """
    Uses the notebook's original select_next_token() logic.

    Before 128 words have been produced, EOS is masked.
    This prevents otherwise-valid continuations from terminating early.
    """

    safe_logits = logits.float().clone()

    if (
        eos_id is not None
        and not allow_eos
        and 0 <= int(eos_id) < safe_logits.numel()
    ):
        safe_logits[int(eos_id)] = -torch.inf

    return select_next_token(
        safe_logits,
        generated,
        policy,
        generator,
    )


# =====================================================================
# 6. RECOVERY GENERATOR
# =====================================================================

@torch.inference_mode()
def recovery_continuation(
    model,
    tokenizer,
    prompt,
    device,
    *,
    policy,
    target_words=128,
):
    """
    Same trained encoder-decoder model.

    Main recovery differences:
      1. obey decoder limit: 511 generated tokens maximum;
      2. do not permit EOS before 128 words;
      3. quality statistics remain diagnostics, not hard blockers.
    """

    max_encoder_length = int(
        model_config.get(
            "max_encoder_length",
            model_config.get("context_length", 0),
        )
    )

    max_decoder_length_local = int(
        model_config.get(
            "max_decoder_length",
            0,
        )
    )

    max_new_tokens = (
        max_decoder_length_local - 1
    )

    if max_encoder_length <= 0:
        raise ValueError(
            "Model has no valid max_encoder_length"
        )

    if max_new_tokens <= 0:
        raise ValueError(
            "Model has no decoder generation capacity"
        )


    # ---------------------------------------------------------
    # Encode official prompt
    # ---------------------------------------------------------

    prompt_ids = tokenizer_encode(
        tokenizer,
        prompt,
    )

    if not prompt_ids:
        raise ValueError(
            "Prompt encoded to zero tokens"
        )

    encoder_ids, prompt_report = (
        bounded_encoder_prompt(
            prompt_ids,
            max_encoder_length,
        )
    )


    bos_id = getattr(
        tokenizer,
        "bos_token_id",
        None,
    )

    eos_id = getattr(
        tokenizer,
        "eos_token_id",
        None,
    )

    if bos_id is None:
        raise ValueError(
            "Tokenizer requires bos_token_id"
        )


    encoder = torch.tensor(
        [encoder_ids],
        dtype=torch.long,
        device=device,
    )

    encoder_mask = torch.ones_like(
        encoder,
        dtype=torch.bool,
    )


    # ---------------------------------------------------------
    # Generation state
    # ---------------------------------------------------------

    generated = []

    seed = int(policy["seed"])

    generator = torch.Generator(
        device=device
    ).manual_seed(seed)

    model.eval()


    prepare = getattr(
        MODEL_MODULE,
        "prepare_generation",
        None,
    )

    decode = getattr(
        MODEL_MODULE,
        "decode_step",
        None,
    )

    cache_used = (
        callable(prepare)
        and callable(decode)
    )


    if cache_used:

        generation_state = prepare(
            model,
            encoder,
            encoder_mask,
        )

        next_input = torch.tensor(
            [[int(bos_id)]],
            dtype=torch.long,
            device=device,
        )


    # Decode text every few tokens.
    check_interval = 4


    # ---------------------------------------------------------
    # Generate
    # ---------------------------------------------------------

    for step in range(max_new_tokens):

        if cache_used:

            step_logits, generation_state = decode(
                model,
                next_input,
                generation_state,
            )

            logits = step_logits[0]

        else:

            decoder = torch.tensor(
                [[int(bos_id), *generated]],
                dtype=torch.long,
                device=device,
            )

            all_logits = seq2seq_logits(
                model,
                encoder,
                decoder,
                encoder_mask,
            )

            logits = all_logits[0, -1]


        # Determine whether enough words already exist.
        current_text = ""

        if generated:
            current_text = tokenizer_decode(
                tokenizer,
                generated,
            )

        enough_words = (
            len(words(current_text))
            >= target_words
        )


        # Prevent EOS until target length exists.
        next_id = select_token_without_premature_eos(
            logits,
            generated,
            policy,
            generator,
            eos_id,
            allow_eos=enough_words,
        )


        generated.append(
            int(next_id)
        )


        if cache_used:

            next_input = torch.tensor(
                [[int(next_id)]],
                dtype=torch.long,
                device=device,
            )


        should_check = (
            len(generated) % check_interval == 0
            or step + 1 == max_new_tokens
        )


        if should_check:

            continuation = tokenizer_decode(
                tokenizer,
                generated,
            )

            word_count = len(
                words(continuation)
            )


            if word_count >= target_words:

                result = exact_word_prefix(
                    continuation,
                    target_words,
                )

                report = generation_quality_report(
                    result
                )

                report.update(
                    prompt_report
                )

                report.update(
                    {
                        "generated_tokens": len(generated),
                        "max_new_tokens": max_new_tokens,
                        "token_budget_utilization": (
                            len(generated)
                            / max_new_tokens
                        ),
                        "incremental_kv_cache": cache_used,
                        "recovery_mode": True,
                        "premature_eos_masked": True,
                    }
                )

                return result, report


    raise RuntimeError(
        "Model consumed all "
        f"{max_new_tokens} decoder tokens "
        "without producing 128 whitespace words"
    )


# =====================================================================
# 7. RECOVERY POLICIES
#
# Start with the student's existing declared policies.
# Quality-threshold fields are removed ONLY as rejection gates.
# Sampling parameters themselves are kept.
# =====================================================================

RECOVERY_POLICIES = []

for original_policy in CFG.decoding["policies"]:

    policy = copy.deepcopy(
        original_policy
    )

    # These are optional diagnostics, not submission requirements.
    policy.pop(
        "min_unique_word_ratio",
        None,
    )

    policy.pop(
        "max_word_fraction",
        None,
    )

    policy["name"] = (
        original_policy["name"]
        + "_length_recovery"
    )

    RECOVERY_POLICIES.append(
        policy
    )


# Add deterministic alternate seeds of the SAME policies.
seed_offsets = [
    0,
    100003,
    200003,
    300007,
    400009,
]

expanded_policies = []

for policy in RECOVERY_POLICIES:

    for offset in seed_offsets:

        attempt_policy = copy.deepcopy(
            policy
        )

        attempt_policy["seed"] = (
            int(policy["seed"])
            + offset
        )

        attempt_policy["name"] = (
            f"{policy['name']}_{offset}"
        )

        expanded_policies.append(
            attempt_policy
        )


print(
    "\nRecovery decoding attempts per missing ID:",
    len(expanded_policies),
)


# =====================================================================
# 8. GENERATE ONLY THE MISSING IDS
# =====================================================================

missing_rows = [
    row
    for row in TEST_RECORDS
    if str(row["id"])
    not in PREDICTIONS
]

RECOVERY_FAILURES = []


for row in tqdm(
    missing_rows,
    desc="Generating missing predictions",
):

    row_id = str(row["id"])

    row_seed_hash = int(
        hashlib.sha256(
            row_id.encode("utf-8")
        ).hexdigest()[:8],
        16,
    )

    completed = False
    last_error = None


    for policy_index, base_policy in enumerate(
        expanded_policies
    ):

        policy = copy.deepcopy(
            base_policy
        )

        policy["seed"] = (
            int(policy["seed"])
            + row_seed_hash
        )

        started = time.perf_counter()


        try:

            continuation, quality = (
                recovery_continuation(
                    FINAL_MODEL,
                    tokenizer,
                    row["prompt"],
                    DEVICE,
                    policy=policy,
                    target_words=128,
                )
            )


            if len(words(continuation)) != 128:
                raise RuntimeError(
                    "Recovery output was not "
                    "exactly 128 words"
                )


            PREDICTIONS[row_id] = {
                "text": continuation
            }


            # Persist after EVERY successful ID.
            atomic_write_json(
                partial_path,
                PREDICTIONS,
            )


            with attempts_path.open(
                "a",
                encoding="utf-8",
            ) as handle:

                handle.write(
                    json.dumps(
                        {
                            "id": row_id,
                            "status": "success",
                            "policy_index": policy_index,
                            "policy_name": policy["name"],
                            "quality": quality,
                            "seconds": (
                                time.perf_counter()
                                - started
                            ),
                            "recovery": True,
                            "created_utc": utc_now(),
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )


            completed = True
            break


        except Exception as exc:

            last_error = (
                f"{type(exc).__name__}: {exc}"
            )


            with attempts_path.open(
                "a",
                encoding="utf-8",
            ) as handle:

                handle.write(
                    json.dumps(
                        {
                            "id": row_id,
                            "status": "failed",
                            "policy_index": policy_index,
                            "policy_name": policy["name"],
                            "error": last_error,
                            "seconds": (
                                time.perf_counter()
                                - started
                            ),
                            "recovery": True,
                            "created_utc": utc_now(),
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )


    if not completed:

        RECOVERY_FAILURES.append(
            {
                "id": row_id,
                "error": last_error,
            }
        )


atomic_write_json(
    DIRS["predictions"]
    / "generation_failures.json",
    RECOVERY_FAILURES,
)


# =====================================================================
# 9. FINAL STRICT CHECK
# =====================================================================

missing_ids = (
    official_id_set
    - set(PREDICTIONS)
)

foreign_ids = (
    set(PREDICTIONS)
    - official_id_set
)

wrong_length_ids = [
    key
    for key, value in PREDICTIONS.items()
    if (
        not isinstance(value, dict)
        or set(value) != {"text"}
        or len(words(value["text"])) != 128
    )
]


print()
print("=" * 80)
print("FINAL GENERATION STATUS")
print("=" * 80)

print("Predictions :", len(PREDICTIONS))
print("Missing     :", len(missing_ids))
print("Wrong length:", len(wrong_length_ids))
print("Foreign IDs :", len(foreign_ids))


if (
    missing_ids
    or foreign_ids
    or wrong_length_ids
):

    print("\nFirst recovery failures:")

    for failure in RECOVERY_FAILURES[:10]:
        print(failure)

    raise RuntimeError(
        f"Still incomplete: "
        f"{len(missing_ids)} IDs missing. "
        "Successful predictions remain safely saved."
    )


# =====================================================================
# 10. WRITE AUTHENTICATED INFERENCE PROVENANCE
# =====================================================================

provenance = {
    "checkpoint_id": checkpoint_id,
    "tokenizer_fingerprint": (
        TOKENIZER_META["fingerprint"]
    ),
    "model_source_sha256": (
        MODEL_SOURCE_SHA256
    ),
    "test_fingerprint": (
        test_records_fingerprint(
            TEST_RECORDS
        )
    ),
    "decoding": {
        **dict(CFG.decoding),
        "recovery": {
            "max_new_tokens": (
                MAX_GENERATED_TOKENS
            ),
            "premature_eos_masked": True,
            "quality_thresholds_used_as_diagnostics_only": True,
        },
    },
    "prediction_count": len(PREDICTIONS),
    "partial_prediction_sha256": (
        sha256_file(partial_path)
    ),
    "attempt_log_sha256": (
        sha256_file(attempts_path)
        if attempts_path.exists()
        else None
    ),
    "created_utc": utc_now(),
}


atomic_write_json(
    DIRS["predictions"]
    / "inference_provenance.json",
    provenance,
)


# =====================================================================
# 11. CREATE VALIDATED CODABENCH ZIP
# =====================================================================

SUBMISSION_ZIP, SUBMISSION_REPORT = (
    create_submission_zip(
        PREDICTIONS,
        official_ids,
        DIRS["submission"],
    )
)


print()
print("=" * 80)
print("CODABENCH SUBMISSION CREATED")
print("=" * 80)

print("Submission ZIP:")
print(SUBMISSION_ZIP)


# =====================================================================
# 12. CREATE COMPLETE RECOVERY ZIP
#
# Contains model/checkpoints/tokenizer/history/manifests/predictions/etc.
# =====================================================================

RECOVERY_ZIP, RECOVERY_REPORT = (
    create_recovery_zip(ROOT)
)


print()
print("=" * 80)
print("COMPLETE RECOVERY ZIP CREATED")
print("=" * 80)

print(RECOVERY_ZIP)


# =====================================================================
# 13. FINAL VERIFICATION
# =====================================================================

FINAL_REPORT = final_verification(
    CFG,
    FINAL_MODEL,
    TOKENIZER_META,
    official_ids,
)


# =====================================================================
# 14. FINAL DOWNLOAD SUMMARY
# =====================================================================

print()
print("=" * 80)
print("EVERYTHING FINISHED")
print("=" * 80)


paths_to_check = {
    "Codabench submission":
        DIRS["submission"] / "submission.zip",

    "Prediction JSON":
        DIRS["submission"] / "prediction.json",

    "Complete recovery ZIP":
        DIRS["recovery"]
        / "phase1_submission02_recovery.zip",

    "Training recovery ZIP":
        DIRS["recovery"]
        / "training_recovery.latest.zip",

    "Selected model":
        DIRS["checkpoints"]
        / "selected_model.pt",

    "Final model":
        DIRS["checkpoints"]
        / "model_final.pt",

    "Latest checkpoint":
        DIRS["checkpoints"]
        / "checkpoint.latest.pt",

    "Training history":
        DIRS["histories"]
        / "training.jsonl",

    "Final verification":
        DIRS["manifests"]
        / "final_verification.json",
}


for label, path in paths_to_check.items():

    path = Path(path)

    if path.exists():

        print(
            f"[OK] {label}\n"
            f"     {path}\n"
            f"     {path.stat().st_size / (1024**2):.2f} MB"
        )

    else:

        print(
            f"[MISSING] {label}\n"
            f"          {path}"
        )


print()
print("UPLOAD THIS FILE TO CODABENCH:")
print(
    DIRS["submission"]
    / "submission.zip"
)

print()
print("DOWNLOAD THIS FILE TO KEEP THE MODEL FOR LATER PHASES:")
print(
    DIRS["recovery"]
    / "phase1_submission02_recovery.zip"
)

In [ ]:
# =====================================================================
# FINAL PACKAGING ONLY — DO NOT REGENERATE
# =====================================================================

from pathlib import Path
import json
import os
import shutil
import tempfile
import zipfile
import hashlib


print("=" * 80)
print("FINAL PACKAGING — USING EXISTING 1000 PREDICTIONS")
print("=" * 80)


# ---------------------------------------------------------------------
# 1. VERIFY WE REALLY HAVE ALL 1000 VALID PREDICTIONS
# ---------------------------------------------------------------------

if "PREDICTIONS" not in globals():
    raise RuntimeError("PREDICTIONS is not present in memory.")

if "TEST_RECORDS" not in globals():
    raise RuntimeError("TEST_RECORDS is not present in memory.")


official_ids = [str(row["id"]) for row in TEST_RECORDS]
official_id_set = set(official_ids)

prediction_ids = set(str(x) for x in PREDICTIONS.keys())

missing_ids = official_id_set - prediction_ids
foreign_ids = prediction_ids - official_id_set

wrong_length_ids = []

for pred_id, value in PREDICTIONS.items():

    if (
        not isinstance(value, dict)
        or set(value.keys()) != {"text"}
        or len(words(value["text"])) != 128
    ):
        wrong_length_ids.append(str(pred_id))


print("Official IDs :", len(official_ids))
print("Predictions  :", len(PREDICTIONS))
print("Missing      :", len(missing_ids))
print("Foreign      :", len(foreign_ids))
print("Wrong length :", len(wrong_length_ids))


if (
    len(PREDICTIONS) != len(official_ids)
    or missing_ids
    or foreign_ids
    or wrong_length_ids
):
    raise RuntimeError(
        "Prediction validation failed. "
        "Do not package an incomplete submission."
    )


# ---------------------------------------------------------------------
# 2. CREATE SUBMISSION DIRECTORY
# ---------------------------------------------------------------------

submission_dir = DIRS["submission"]

submission_dir.mkdir(
    parents=True,
    exist_ok=True,
)

prediction_path = (
    submission_dir
    / "prediction.json"
)

submission_zip = (
    submission_dir
    / "submission.zip"
)

submission_validation_path = (
    submission_dir
    / "submission_validation.json"
)


# ---------------------------------------------------------------------
# 3. WRITE FINAL prediction.json
# ---------------------------------------------------------------------

# Keep official-ID ordering.
ordered_predictions = {
    pred_id: PREDICTIONS[pred_id]
    for pred_id in official_ids
}

atomic_write_json(
    prediction_path,
    ordered_predictions,
)

print("\nCreated:")
print(prediction_path)


# ---------------------------------------------------------------------
# 4. CREATE CODABENCH submission.zip
#
# ZIP STRUCTURE:
#
# submission.zip
# └── prediction.json
#
# Nothing else is placed inside.
# ---------------------------------------------------------------------

temp_zip = (
    submission_dir
    / "submission.tmp.zip"
)

temp_zip.unlink(
    missing_ok=True
)

with zipfile.ZipFile(
    temp_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    archive.write(
        prediction_path,
        arcname="prediction.json",
    )


# ---------------------------------------------------------------------
# 5. VERIFY THE ZIP BEFORE PROMOTING IT
# ---------------------------------------------------------------------

with zipfile.ZipFile(
    temp_zip,
    mode="r",
) as archive:

    members = archive.namelist()

    if members != ["prediction.json"]:
        raise RuntimeError(
            f"Bad ZIP structure: {members}"
        )

    zipped_predictions = json.loads(
        archive.read(
            "prediction.json"
        ).decode("utf-8")
    )


if set(zipped_predictions.keys()) != official_id_set:
    raise RuntimeError(
        "ZIP IDs do not match official test IDs."
    )


for pred_id, value in zipped_predictions.items():

    if (
        not isinstance(value, dict)
        or set(value.keys()) != {"text"}
        or len(words(value["text"])) != 128
    ):
        raise RuntimeError(
            f"Invalid prediction in ZIP: {pred_id}"
        )


os.replace(
    temp_zip,
    submission_zip,
)


# ---------------------------------------------------------------------
# 6. BUILD / RECOVER INFERENCE PROVENANCE
# ---------------------------------------------------------------------

selected_path = (
    DIRS["checkpoints"]
    / "selected_model.pt"
)

if not selected_path.exists():
    raise FileNotFoundError(
        selected_path
    )


checkpoint_id = (
    f"selected-{sha256_file(selected_path)[:12]}"
)


inference_provenance_path = (
    DIRS["predictions"]
    / "inference_provenance.json"
)


if inference_provenance_path.exists():

    provenance = json.loads(
        inference_provenance_path.read_text(
            encoding="utf-8"
        )
    )

else:

    provenance = {
        "checkpoint_id": checkpoint_id,
        "tokenizer_fingerprint": (
            TOKENIZER_META["fingerprint"]
        ),
        "model_source_sha256": (
            MODEL_SOURCE_SHA256
        ),
        "test_fingerprint": (
            test_records_fingerprint(
                TEST_RECORDS
            )
        ),
        "prediction_count": len(
            ordered_predictions
        ),
        "recovery_generation": True,
        "created_utc": utc_now(),
    }

    atomic_write_json(
        inference_provenance_path,
        provenance,
    )


# Ensure final provenance has correct count.
provenance["prediction_count"] = len(
    ordered_predictions
)

atomic_write_json(
    inference_provenance_path,
    provenance,
)


# ---------------------------------------------------------------------
# 7. CREATE SUBMISSION VALIDATION RECORD
# ---------------------------------------------------------------------

submission_report = {
    "passed": True,
    "prediction_count": len(
        ordered_predictions
    ),
    "all_official_ids_present": True,
    "all_outputs_exactly_128_words": True,
    "prediction_sha256": (
        sha256_file(prediction_path)
    ),
    "submission_zip_sha256": (
        sha256_file(submission_zip)
    ),
    "submission_zip_members": [
        "prediction.json"
    ],
    "inference_provenance": provenance,
    "created_utc": utc_now(),
}


atomic_write_json(
    submission_validation_path,
    submission_report,
)


print("\n" + "=" * 80)
print("CODABENCH PACKAGE CREATED")
print("=" * 80)

print("prediction.json:")
print(prediction_path)

print("\nsubmission.zip:")
print(submission_zip)


# ---------------------------------------------------------------------
# 8. CREATE / REFRESH COMPLETE RECOVERY ZIP
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("CREATING COMPLETE RECOVERY ZIP")
print("=" * 80)


if "create_recovery_zip" in globals():

    RECOVERY_ZIP, RECOVERY_REPORT = (
        create_recovery_zip(ROOT)
    )

else:

    # Fallback if helper disappeared from the live kernel.
    recovery_dir = (
        ROOT / "recovery"
    )

    recovery_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    RECOVERY_ZIP = (
        recovery_dir
        / "phase1_submission02_recovery.zip"
    )

    temp_recovery = (
        recovery_dir
        / "phase1_submission02_recovery.tmp.zip"
    )

    temp_recovery.unlink(
        missing_ok=True
    )

    exclusions = {
        RECOVERY_ZIP.resolve(),
        temp_recovery.resolve(),
        (
            recovery_dir
            / "phase1_submission02_recovery.previous.zip"
        ).resolve(),
    }

    with zipfile.ZipFile(
        temp_recovery,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        allowZip64=True,
    ) as archive:

        for path in sorted(
            p
            for p in ROOT.rglob("*")
            if p.is_file()
        ):

            resolved = path.resolve()

            if resolved in exclusions:
                continue

            # Avoid recursively putting recovery archives
            # inside the final recovery archive.
            if (
                path.parent == recovery_dir
                and path.suffix == ".zip"
            ):
                continue

            # Same large transient data exclusion used
            # by the notebook's own recovery routine.
            bounded_store = (
                ROOT
                / "data"
                / "bounded_document_store"
            )

            if (
                bounded_store.exists()
                and (
                    resolved == bounded_store.resolve()
                    or bounded_store.resolve()
                    in resolved.parents
                )
            ):
                continue

            archive.write(
                path,
                arcname=path.relative_to(
                    ROOT
                ).as_posix(),
            )


    # Verify fallback archive.
    with zipfile.ZipFile(
        temp_recovery,
        "r",
    ) as archive:

        bad = archive.testzip()

        if bad is not None:
            raise RuntimeError(
                f"Recovery ZIP corrupted at: {bad}"
            )


    os.replace(
        temp_recovery,
        RECOVERY_ZIP,
    )

    RECOVERY_REPORT = {
        "status": "PASS",
        "path": str(RECOVERY_ZIP),
        "created_utc": utc_now(),
    }


print("Recovery ZIP:")
print(RECOVERY_ZIP)


# ---------------------------------------------------------------------
# 9. RUN FINAL VERIFICATION IF AVAILABLE
# ---------------------------------------------------------------------

FINAL_REPORT = None

if "final_verification" in globals():

    try:

        model_for_verification = (
            FINAL_MODEL
            if "FINAL_MODEL" in globals()
            else MODEL
        )

        FINAL_REPORT = final_verification(
            CFG,
            model_for_verification,
            TOKENIZER_META,
            official_ids,
        )

    except Exception as exc:

        # Packaging is already valid; record the verification
        # exception rather than destroying finished artifacts.
        print(
            "\nFinal-verification helper raised:",
            type(exc).__name__,
            str(exc),
        )


# ---------------------------------------------------------------------
# 10. VERIFY IMPORTANT MODEL FILES
# ---------------------------------------------------------------------

important_files = {
    "Codabench submission":
        submission_zip,

    "Prediction JSON":
        prediction_path,

    "Submission validation":
        submission_validation_path,

    "Complete recovery ZIP":
        Path(RECOVERY_ZIP),

    "Selected model":
        DIRS["checkpoints"]
        / "selected_model.pt",

    "Final model":
        DIRS["checkpoints"]
        / "model_final.pt",

    "Latest checkpoint":
        DIRS["checkpoints"]
        / "checkpoint.latest.pt",

    "Previous known good":
        DIRS["checkpoints"]
        / "previous_known_good.pt",

    "Training history":
        DIRS["histories"]
        / "training.jsonl",

    "Tokenizer metadata":
        DIRS["tokenizer"]
        / "tokenizer_metadata.json",
}


print("\n" + "=" * 80)
print("FINAL FILE CHECK")
print("=" * 80)


for label, path in important_files.items():

    path = Path(path)

    if path.exists():

        size_mb = (
            path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"[OK] {label}"
        )

        print(
            f"     {path}"
        )

        print(
            f"     {size_mb:.2f} MB"
        )

    else:

        print(
            f"[MISSING] {label}"
        )

        print(
            f"          {path}"
        )


# ---------------------------------------------------------------------
# 11. FINAL SUCCESS MESSAGE
# ---------------------------------------------------------------------

print()
print("=" * 80)
print("PACKAGING COMPLETE")
print("=" * 80)

print(
    "\nUPLOAD THIS TO CODABENCH:\n"
    f"{submission_zip}"
)

print(
    "\nDOWNLOAD THIS TO KEEP THE COMPLETE MODEL "
    "FOR LATER PHASES:\n"
    f"{RECOVERY_ZIP}"
)

print(
    "\nPredictions packaged:",
    len(ordered_predictions),
)

print(
    "All outputs exactly 128 words:",
    True,
)

## 13. Codabench validation and packaging

In [ ]:
def validate_predictions(predictions: Any, official_ids: Sequence[str], target_words: int = 128) -> dict[str, Any]:
    errors = []
    if not isinstance(predictions, dict):
        return {"passed": False, "errors": ["prediction root must be an object"]}
    expected, actual = set(map(str, official_ids)), set(map(str, predictions))
    if missing := sorted(expected - actual):
        errors.append(f"missing IDs: {missing[:20]}")
    if unknown := sorted(actual - expected):
        errors.append(f"unknown IDs: {unknown[:20]}")
    for identifier in sorted(expected & actual):
        entry = predictions[identifier]
        if not isinstance(entry, dict) or set(entry) != {"text"}:
            errors.append(f"{identifier}: entry must contain only a text field")
            continue
        text = entry["text"]
        if not isinstance(text, str) or not text.strip():
            errors.append(f"{identifier}: empty/non-string text")
        elif len(words(text)) != target_words:
            errors.append(f"{identifier}: {len(words(text))} words, expected {target_words}")
    return {"passed": not errors, "errors": errors, "expected_ids": len(expected), "actual_ids": len(actual)}

def create_submission_zip(predictions: Mapping[str, Any], official_ids: Sequence[str], output_dir: Path) -> tuple[Path, dict[str, Any]]:
    report = validate_predictions(predictions, official_ids, CFG.decoding["target_words"])
    if not report["passed"]:
        raise ValueError("Submission validation failed:\n" + "\n".join(report["errors"]))
    provenance_path=DIRS['predictions']/'inference_provenance.json'
    if not provenance_path.exists(): raise RuntimeError('Missing inference provenance')
    provenance=json.loads(provenance_path.read_text(encoding='utf-8'))
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    prediction_path = output_dir / "prediction.json"
    atomic_write_json(prediction_path, predictions)
    zip_path = output_dir / "submission.zip"
    with tempfile.NamedTemporaryFile(dir=output_dir, suffix=".zip", delete=False) as handle:
        temporary = Path(handle.name)
    try:
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:
            archive.write(prediction_path, "prediction.json")
        zip_report = validate_zip(temporary)
        if zip_report["members"] != ["prediction.json"]:
            raise ValueError(f"Invalid ZIP root structure: {zip_report['members']}")
        os.replace(temporary, zip_path)
    finally:
        temporary.unlink(missing_ok=True)
    report['prediction_sha256']=sha256_file(prediction_path)
    report['quality_diagnostics']={key:generation_quality_report(value['text']) for key,value in predictions.items()}
    report['inference_provenance']=provenance
    report["zip"] = validate_zip(zip_path)
    atomic_write_json(output_dir / "submission_validation.json", report)
    return zip_path, report

if 'PREDICTIONS' in globals() and 'TEST_RECORDS' in globals():
    SUBMISSION_ZIP,SUBMISSION_REPORT=create_submission_zip(PREDICTIONS,[row['id'] for row in TEST_RECORDS],DIRS['submission']);print('Created validated Codabench package:',SUBMISSION_ZIP)
else: SUBMISSION_ZIP=SUBMISSION_REPORT=None;print('Submission packaging deferred until authenticated official predictions exist.')


## 14. Recovery archive and integrity manifest

Periodic and final archives share the same verified format and restore helper. Tensor checkpoints are stored without ineffective recompression, and integrity verification/extraction is streamed in bounded memory. The complete archive deliberately excludes the periodic ZIP to avoid duplicating large checkpoints.


In [ ]:
def build_integrity_manifest(root: Path, excluded: set[Path] | None = None) -> dict[str, Any]:
    root = Path(root).resolve()
    excluded = {Path(path).resolve() for path in (excluded or set())}

    files = []

    for path in sorted(item for item in root.rglob("*") if item.is_file()):
        resolved = path.resolve()

        if (
            resolved in excluded
            or any(
                item.is_dir() and (item == resolved or item in resolved.parents)
                for item in excluded
            )
            or path.is_symlink()
        ):
            continue

        files.append(
            {
                "path": path.relative_to(root).as_posix(),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

    return {
        "format": "cs781_recovery_integrity_v1",
        "created_utc": utc_now(),
        "files": files,
    }


def create_recovery_zip(root: Path) -> tuple[Path, dict[str, Any]]:
    root = Path(root).resolve()

    output_dir = root / "recovery"
    output_dir.mkdir(parents=True, exist_ok=True)

    output = output_dir / "phase1_submission02_recovery.zip"
    previous = output_dir / "phase1_submission02_recovery.previous.zip"

    manifest_path = root / RECOVERY_MANIFEST_MEMBER

    excluded = {
        output,
        previous,
        manifest_path,
        output_dir / "recovery_validation.json",
        output_dir / "training_recovery.latest.zip",
        output_dir / "training_recovery.latest.json",
        root / "data" / "bounded_document_store",
    }

    notebook_backup = (
        Path(CFG.completed_notebook_path).expanduser()
        if CFG.completed_notebook_path
        else None
    )

    if notebook_backup is not None:
        if not notebook_backup.is_file():
            raise FileNotFoundError(notebook_backup)

        shutil.copy2(
            notebook_backup,
            root / notebook_backup.name,
        )

    manifest = build_integrity_manifest(root, excluded)

    files = [
        root / row["path"]
        for row in manifest["files"]
    ]

    if output.exists():
        verify_recovery_archive(output)

        with tempfile.NamedTemporaryFile(
            dir=output_dir,
            suffix=".zip",
            delete=False,
        ) as handle:
            previous_temporary = Path(handle.name)

        try:
            shutil.copyfile(
                output,
                previous_temporary,
            )

            os.replace(
                previous_temporary,
                previous,
            )

        finally:
            previous_temporary.unlink(missing_ok=True)

    report = write_verified_recovery_archive(
        root,
        files,
        output,
        manifest=manifest,
    )

    atomic_write_json(
        manifest_path,
        manifest,
    )

    atomic_write_json(
        output_dir / "recovery_validation.json",
        report,
    )

    return output, report


RECOVERY_ZIP, RECOVERY_REPORT = create_recovery_zip(ROOT)

In [ ]:
print("Recovery ZIP:", RECOVERY_ZIP)
print("Report:", RECOVERY_REPORT)

## 15. Final PASS/FAIL verification

In [ ]:
def final_verification(cfg, model=None, tokenizer_meta=None, official_ids=None):
    checks = {
        "student_owned_architecture_attested": bool(cfg.student_owned_architecture_attestation),
        "student_configuration_confirmed": bool(cfg.student_configuration_confirmed),
        "trained_from_scratch_attested": bool(cfg.trained_from_scratch_attestation),
    }

    model_validation = DIRS["manifests"] / "model_validation.json"
    report = (
        json.loads(model_validation.read_text(encoding="utf-8"))
        if model_validation.exists()
        else {}
    )

    checks["parameter_count_50m_to_300m"] = (
        50_000_000 <= int(report.get("total", -1)) <= 300_000_000
    )

    checks["model_source_frozen"] = bool(
        MODEL_SOURCE_PATH
        and Path(MODEL_SOURCE_PATH).exists()
        and report.get("model_source_sha256") == sha256_file(MODEL_SOURCE_PATH)
    )

    smoke = DIRS["manifests"] / "gpu_training_smoke_test.json"

    checks["gpu_training_smoke_passed"] = (
        smoke.exists()
        and json.loads(smoke.read_text()).get("status") == "PASS"
    )

    tokenizer_path = DIRS["tokenizer"] / "tokenizer_metadata.json"

    checks["tokenizer_registered"] = tokenizer_path.exists()

    checks["tokenizer_integrity"] = bool(
        tokenizer_path.exists()
        and json.loads(tokenizer_path.read_text()).get("fingerprint")
        == directory_fingerprint(DIRS["tokenizer"])
    )

    checks["split_manifest_present"] = (
        DIRS["manifests"] / "document_split.json"
    ).exists()

    checks["validation_protocol_present"] = (
        DIRS["manifests"] / "validation_sequence_manifest.json"
    ).exists()

    checks["training_history_present"] = (
        DIRS["histories"] / "training.jsonl"
    ).exists()

    selected = DIRS["checkpoints"] / "selected_model.pt"
    final = DIRS["checkpoints"] / "model_final.pt"

    checks["selected_model_present"] = selected.exists()
    checks["final_model_present"] = final.exists()

    submission = DIRS["submission"] / "submission.zip"

    checks["submission_zip_valid"] = False
    checks["artifact_provenance_fresh"] = False

    if submission.exists() and official_ids is not None:
        try:
            with zipfile.ZipFile(submission) as archive:
                if archive.namelist() == ["prediction.json"]:
                    predictions = json.loads(
                        archive.read("prediction.json")
                    )

                    checks["submission_zip_valid"] = validate_predictions(
                        predictions,
                        official_ids,
                        128,
                    )["passed"]

            validation = json.loads(
                (
                    DIRS["submission"]
                    / "submission_validation.json"
                ).read_text()
            )

            provenance = validation.get("inference_provenance", {})

            checks["artifact_provenance_fresh"] = bool(
                selected.exists()
                and provenance.get("checkpoint_id")
                == f"selected-{sha256_file(selected)[:12]}"
                and provenance.get("model_source_sha256")
                == MODEL_SOURCE_SHA256
                and validation.get("prediction_sha256")
                == sha256_file(
                    DIRS["submission"] / "prediction.json"
                )
            )

        except Exception:
            pass

    periodic_recovery = (
        DIRS["recovery"] / "training_recovery.latest.zip"
    )

    checks["periodic_recovery_zip_valid"] = False

    if periodic_recovery.exists():
        try:
            verify_recovery_archive(periodic_recovery)
            checks["periodic_recovery_zip_valid"] = True
        except Exception:
            pass

    recovery = (
        DIRS["recovery"]
        / "phase1_submission02_recovery.zip"
    )

    checks["recovery_zip_valid"] = False
    checks["recovery_is_fresh"] = False

    if recovery.exists():
        try:
            verify_recovery_archive(recovery)

            checks["recovery_zip_valid"] = True

            inputs = [
                path
                for path in (selected, submission)
                if path.exists()
            ]

            checks["recovery_is_fresh"] = bool(
                inputs
                and recovery.stat().st_mtime
                >= max(path.stat().st_mtime for path in inputs)
            )

        except Exception:
            pass

    artifact_status = (
        "PASS"
        if all(checks.values())
        else "FAIL"
    )

    codabench_record = (
        DIRS["manifests"] / "codabench_acceptance.json"
    )

    accepted = False

    if codabench_record.exists():
        record = json.loads(
            codabench_record.read_text(encoding="utf-8")
        )

        accepted = bool(
            record.get("accepted")
            and record.get("submission_id")
        )

    outcome = {
        "artifact_status": artifact_status,
        "codabench_accepted": accepted,
        "completion_status": (
            "COMPLETE"
            if artifact_status == "PASS" and accepted
            else "INCOMPLETE"
        ),
        "checks": checks,
        "optional_local_evaluation_present": any(
            DIRS["evaluations"].glob("*.json")
        ),
        "created_utc": utc_now(),
    }

    atomic_write_json(
        DIRS["manifests"] / "final_verification.json",
        outcome,
    )

    print(json.dumps(outcome, indent=2))

    return outcome


# After Codabench accepts the upload, record truthful evidence, for example:
#
# atomic_write_json(
#     DIRS["manifests"] / "codabench_acceptance.json",
#     {
#         "accepted": True,
#         "submission_id": "REAL_ID",
#         "submitted_utc": utc_now(),
#         "result": {
#             "bertscore_f1": "REAL_VALUE",
#             "rouge_l_f1": "REAL_VALUE",
#             "bleu_4": "REAL_VALUE",
#         },
#     },
# )


FINAL_REPORT = final_verification(
    CFG,
    MODEL,
    TOKENIZER_META,
    [row["id"] for row in TEST_RECORDS],
)

## Kaggle execution checklist for Submission 02

1. Keep Submission 01 backed up. Submission 02 uses only `.../submission_02` paths.
2. Attach the official Phase 1 Kaggle dataset under `/kaggle/input`. The schema inspection must show
   each training file separately; training arrays without ID fields receive deterministic source+row IDs.
3. Set `CFG.corpus_document_cap_per_source`, review the bounded-cache estimate, and enable
   `RUN_CORPUS_PREPROCESSING` once. The full official corpus is never copied into `/kaggle/working`.
4. Point `CFG.tokenizer_import_path` to the frozen Submission 01/course tokenizer when performing a
   controlled architecture comparison. Never change its token-to-ID mapping after training begins.
5. Write the encoder-decoder module yourself from the course material, save it as a Python file, set
   `CFG.student_model_source_path`, and fill the exact numerical model configuration. The validation
   report must show 50M-300M parameters, causal decoder behavior, encoder conditioning, finite gradients,
   and exact reconstruction.
6. Fill every `None` in `TRAINING_CONFIG`, write the hypothesis/rationale, and record the matched
   Submission 01 control. These are student-owned decisions, not auto-tuned values.
7. Run the representative GPU smoke test. Review actual precision, peak VRAM, loss, and gradient norm.
   If two GPUs are present, this notebook still uses only the explicitly selected GPU.
8. Choose exactly one mode: `fresh_train` refuses an existing checkpoint; `resume_train` requires one
   and prints the restored optimizer step. Enable full training only after both explicit gates are true.
9. Verify and download `training_recovery.latest.zip` milestones outside Kaggle. The same restore helper
   accepts periodic and complete archives; a file left only in `/kaggle/working` is not durable.
10. For inference-only, restore tokenizer, model source/config, and `selected_model.pt`. Do not load the
    training corpus. Run the 16-example inference benchmark and inspect timing, truncation, and token-budget use.
11. Declare ordered decoding/retry policies, then generate all official IDs. Validate exactly 128 continuation words and create the root-level
    `prediction.json` inside `submission.zip`, and rebuild the final recovery archive.
12. Upload only `submission.zip` to Codabench. Record the real acceptance ID/results afterward; do not
    fabricate metrics. Submission 02 is complete only when artifact verification passes and Codabench
    acceptance is truthfully recorded.
